<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:30px 34px 26px;margin:8px 0 14px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; EGYPT 2021 · HOUSEHOLD INCOME ANALYSIS</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:38px;line-height:1.2;font-weight:700;color:#f3d77a;">
Socioeconomic Factors Associated with Household Income per Capita in Egypt</h1><div style="height:1px;background-color:#8a6a1c;margin:14px 0 10px;"></div><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:13.5px;line-height:1.55;color:#a8a086;font-style:italic;">Loads, cleans, and merges the Egypt 2021 Individual and Household datasets.</div></div>

<a id="sec1"></a>
<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 1</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">1. Setup &amp; Configuration</h1><div style="height:1px;background-color:#8a6a1c;margin:14px 0 10px;"></div><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:13.5px;line-height:1.55;color:#a8a086;font-style:italic;">Imports, data locations, reusable helper functions and the plotting style used throughout the notebook.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">1.1 Importing Libraries</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Imports every library used in the notebook and silences warnings.</div></div>

In [ ]:
import warnings
import re
import json
import html
import sqlite3
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2
from scipy import stats as st
from IPython.display import display, HTML
import plotly.io as pio
import statsmodels.formula.api as smf
from scipy.stats import mannwhitneyu, spearmanr

warnings.filterwarnings("ignore")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">1.2 Data Location &amp; Global Settings</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Finds the two Excel files automatically using their names and fixes the random seed.</div></div>

In [ ]:
# Set a fixed random seed for repeatable results. Find the two input Excel files in common
# folders, require them to be together, and save their full paths.
RANDOM_STATE = 42

INDIVIDUAL_NAME = "Egypt21_Individual_Translated.xlsx"
HOUSEHOLD_NAME = "Egypt21_Household_Translated.xlsx"

# Search automatically without using a device-specific path.
search_roots = [
    Path.cwd(),
    Path.home() / "Downloads",
    Path.home() / "Desktop",
    Path.home() / "Documents",
]

DATA_DIR = None

for root in dict.fromkeys(search_roots):
    if not root.exists():
        continue

    for individual_file in root.rglob(INDIVIDUAL_NAME):
        folder = individual_file.parent

        if (folder / HOUSEHOLD_NAME).is_file():
            DATA_DIR = folder
            break

    if DATA_DIR is not None:
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Keep both Excel files in the same folder and run the notebook again."
    )

INDIVIDUAL_FILE = DATA_DIR / INDIVIDUAL_NAME
HOUSEHOLD_FILE = DATA_DIR / HOUSEHOLD_NAME

print(f"Data folder: {DATA_DIR.resolve()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">1.3 Cleaning Helper Functions</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Small reusable functions that keep the cleaning code short and consistent.</div></div>

In [ ]:
# Define reusable cleaning functions for reading spreadsheets, standardizing text and Yes/No
# fields, and reporting values. Keeping these helpers in one place makes later cleaning steps
# consistent.
def load_excel(path):
    """Read an Excel file, treating only empty cells as missing values."""
    return pd.read_excel(path, keep_default_na=False, na_values=[""])


def strip_text(series):
    """Convert to the pandas string type and trim surrounding spaces."""
    return series.astype("string").str.strip()


def clean_text(series):
    """Trim spaces and turn empty strings into missing values."""
    return strip_text(series).replace("", pd.NA)


def normalized(series):
    """Lower-cased clean text, used for case-insensitive comparisons."""
    return clean_text(series).str.lower()


def is_(series, value):
    """Boolean mask that is True where the series equals `value` (missing -> False)."""
    return series.eq(value).fillna(False).astype(bool)


def standardize_yes_no(series):
    """Map yes/no/true/false variants to 'Yes' and 'No'."""
    return normalized(series).replace(
        {"yes": "Yes", "no": "No", "true": "Yes", "false": "No"}
    )


def report(data, col):
    """Print the value counts, data type and remaining missing values of a column."""
    print(data[col].value_counts().to_string())
    print(f"Data type: {data[col].dtype}")
    print(f"Remaining missing: {data[col].isna().sum()}")

def weighted_quantile(values, weights, quantile):
    values = pd.to_numeric(values, errors="coerce")
    weights = pd.to_numeric(weights, errors="coerce")

    valid = values.notna() & weights.notna() & (weights > 0)
    values = values[valid]
    weights = weights[valid]

    if len(values) == 0:
        return np.nan

    order = np.argsort(values)
    values = values.iloc[order]
    weights = weights.iloc[order]

    cumulative_weights = weights.cumsum()
    cutoff = quantile * weights.sum()

    return values.iloc[
        np.searchsorted(cumulative_weights.to_numpy(), cutoff)
    ]


def fill_employment_blanks(values, age, labor_status, main_activity):
    """Label blank employment values using age, labor-force status and main activity.

    Any blank that is still unexplained (or already 'Not stated') becomes 'Unemployed'.
    """
    values = values.copy()
    blank = values.isna()

    values[blank & age.lt(6)] = "Not applicable - under 6"
    values[blank & is_(labor_status, "inactive")] = "Not applicable - not in labor force"
    values[blank & is_(main_activity, "unemployed")] = "Unemployed"

    values = values.fillna("Not stated")
    values[is_(values.str.strip().str.lower(), "not stated")] = "Unemployed"
    return values

def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce").replace(0, np.nan)
    return numerator / denominator


def add_feature(frame, name, values, registry=None):
    frame[name] = values
    if registry is not None and name not in registry:
        registry.append(name)


def weighted_mean(values, weights):
    values = pd.to_numeric(values, errors="coerce")
    weights = pd.to_numeric(weights, errors="coerce")
    valid = values.notna() & weights.notna() & weights.gt(0)
    if not valid.any():
        return np.nan
    return np.average(values[valid], weights=weights[valid])


def weighted_median(values, weights):
    values = pd.to_numeric(values, errors="coerce")
    weights = pd.to_numeric(weights, errors="coerce")
    valid = values.notna() & weights.notna() & weights.gt(0)
    if not valid.any():
        return np.nan
    values = values[valid].to_numpy(dtype=float)
    weights = weights[valid].to_numpy(dtype=float)
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    return values[np.searchsorted(np.cumsum(weights), weights.sum() / 2)]


def weighted_percentage(values, weights):
    values = pd.to_numeric(values, errors="coerce")
    weights = pd.to_numeric(weights, errors="coerce")
    valid = values.notna() & weights.notna() & weights.gt(0)
    if not valid.any():
        return np.nan
    return np.average(values[valid], weights=weights[valid]) * 100


def weighted_percentage_table(data, column):
    temp = data[[column, "household_weight"]].copy()
    temp[column] = temp[column].astype("string").fillna("Missing")
    temp["household_weight"] = pd.to_numeric(
        temp["household_weight"], errors="coerce"
    )
    temp = temp[temp["household_weight"].gt(0)].dropna(
        subset=["household_weight"]
    )
    result = (
        temp.groupby(column, dropna=False)["household_weight"]
        .sum()
        .sort_values(ascending=False)
        .rename("Weighted households")
        .to_frame()
    )
    result["Weighted percentage"] = (
        result["Weighted households"]
        / result["Weighted households"].sum()
        * 100
    ).round(2)
    return result


def normalize_column(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def get_matching_column(frame, valid_names):
    for column in frame.columns:
        if normalize_column(column) in valid_names:
            return column
    return None


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">1.4 Dashboard Theme</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Applies the black-and-gold colours of the dashboard to every chart and table in the notebook.</div></div>

In [ ]:
# Applies the black-and-gold look of the Egypt 2021 dashboard to the whole notebook: one shared colour
# palette, matching Matplotlib and Plotly defaults, and a dark gold style for every data table.
# Only appearance is changed. No data, calculation or condition is touched.


THEME = dict(
    bg="#000000", panel="#0b0905", line="#2b2410",
    gold="#d4af37", gold_light="#f3d77a", gold_pale="#fff3b0", gold_dark="#8a6a1c",
    text="#f5efe0", dim="#a8a086", negative="#e5645f", amber="#d97736",
)
GOLD_SEQUENCE = ["#f3d77a", "#d4af37", "#b8902a", "#8a6a1c", "#c9c2ab", "#5f5a47"]

# Matplotlib defaults
plt.rcParams.update({
    "figure.facecolor": THEME["panel"], "axes.facecolor": THEME["panel"], "savefig.facecolor": THEME["panel"],
    "axes.edgecolor": THEME["line"], "axes.labelcolor": THEME["text"], "axes.titlecolor": THEME["gold_light"],
    "text.color": THEME["text"], "xtick.color": THEME["dim"], "ytick.color": THEME["dim"],
    "grid.color": THEME["line"], "axes.prop_cycle": plt.cycler(color=GOLD_SEQUENCE),
    "font.family": "DejaVu Sans",
})

# Plotly default template
pio.templates["egypt_gold"] = go.layout.Template(layout=dict(
    paper_bgcolor=THEME["panel"], plot_bgcolor=THEME["panel"],
    font=dict(color=THEME["text"]), colorway=GOLD_SEQUENCE,
    title=dict(font=dict(color=THEME["gold_light"])),
    xaxis=dict(gridcolor=THEME["line"], linecolor=THEME["line"], zerolinecolor=THEME["line"]),
    yaxis=dict(gridcolor=THEME["line"], linecolor=THEME["line"], zerolinecolor=THEME["line"]),
    hoverlabel=dict(bgcolor="#14100a", bordercolor=THEME["gold"], font=dict(color=THEME["gold_light"])),
))
pio.templates.default = "egypt_gold"

# Table style (DataFrames and styled tables)
TABLE_CSS = """
<style>
table.dataframe, table[id^="T_"] {
  border-collapse: collapse; border: 1px solid #2b2410; background-color: #0b0905; color: #f5efe0;
  font-family: Manrope, 'Segoe UI', Helvetica, Arial, sans-serif; font-size: 12.5px;
}
table.dataframe thead th, table[id^="T_"] thead th {
  background-color: #17130a; color: #f3d77a; font-weight: 700; padding: 7px 10px;
  border-bottom: 2px solid #8a6a1c; border-top: 0; border-color: #2b2410;
}
table.dataframe thead th { background-color: #17130a !important; color: #f3d77a !important; border-bottom: 2px solid #8a6a1c !important; }
table.dataframe tbody th, table[id^="T_"] tbody th { color: #f3d77a; font-weight: 600; background-color: #0b0905; padding: 6px 10px; border-color: #2b2410; }
table.dataframe tbody td, table[id^="T_"] tbody td { color: #f5efe0; background-color: #0b0905; padding: 6px 10px; border-color: #2b2410; }
table.dataframe tbody th { background-color: #0b0905 !important; color: #f3d77a !important; }
table.dataframe tbody td { background-color: #0b0905 !important; color: #f5efe0 !important; }
table.dataframe tbody tr:nth-child(even) th, table.dataframe tbody tr:nth-child(even) td { background-color: #100d07 !important; }
table.dataframe tbody tr:hover th, table.dataframe tbody tr:hover td { background-color: #1a1606 !important; }
table[id^="T_"] tbody tr { background-color: #0b0905 !important; }
table[id^="T_"] tbody tr:nth-child(even) { background-color: #100d07 !important; }
table[id^="T_"] caption { background-color: #0b0905; color: #f3d77a; font-weight: 700; font-size: 14px; text-align: left; padding: 10px 14px; border: 1px solid #2b2410; border-bottom: 0; }
</style>
"""

SWATCHES = [("Black", THEME["bg"]), ("Panel", THEME["panel"]), ("Line", THEME["line"]),
            ("Dark gold", THEME["gold_dark"]), ("Gold", THEME["gold"]), ("Light gold", THEME["gold_light"]),
            ("Pale gold", THEME["gold_pale"]), ("Text", THEME["text"])]
chips = "".join(
    f'<div style="flex:1;min-width:92px;text-align:center;font:600 11px Manrope,Arial,sans-serif;color:#a8a086;">'
    f'<div style="height:34px;border-radius:6px;border:1px solid #4a3a10;background-color:{hex_};margin-bottom:6px;"></div>'
    f'{name}<br><span style="color:#f3d77a;font-weight:700;">{hex_}</span></div>'
    for name, hex_ in SWATCHES
)
display(HTML(TABLE_CSS + f'<div style="background-color:#000;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:8px;padding:14px 16px;display:flex;gap:10px;flex-wrap:wrap;">{chips}</div>'))


<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 2</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">2. Describe the Data</h1></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">2.1 Individual Dataset — Raw Data Inspection</h2></div>

<a id="sec2"></a>
<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 2</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">2. Individual Dataset — Preprocessing</h1></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">2.1.1 Loading the Dataset</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Loads the individual-level survey data into <code style="background-color:#17130a;color:#f3d77a;border:1px solid #2b2410;border-radius:4px;padding:1px 6px;font-family:SFMono-Regular,Consolas,'Liberation Mono',monospace;font-size:.92em;">df</code>.</div></div>

In [ ]:
# Read the individual-level Excel file into a DataFrame. Each row represents a record that will be
# checked and cleaned.
df = load_excel(INDIVIDUAL_FILE)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">2.1.2 Initial Data Inspection</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">A first look at the raw individual data.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">2.1.2.1 Preview the First Rows</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Shows the first five records.</div></div>

In [ ]:
# Display the first few individual records to confirm the file loaded correctly and inspect column
# contents.
df.head()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">2.1.2.2 Columns and Data Types</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Lists every column with its data type and non-null count.</div></div>

In [ ]:
# Show each column’s data type and number of non-missing entries. This helps find columns needing
# conversion or cleaning.
df.info()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">2.1.2.3 Missing Values</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Counts missing values in each column.</div></div>

In [ ]:
# Count missing values in every original individual-data column. Use this as the starting point
# before any missing-value rules are applied.
df.isnull().sum()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">2.1.2.4 Duplicate Rows</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Counts fully duplicated rows.</div></div>

In [ ]:
# Count fully duplicated rows in the original individual dataset to check for repeated records.
df.duplicated().sum()


<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 3</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">3. Data Cleaning</h1></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">3.1 Individual Data Cleaning</h2></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.1.1 Creating the Cleaning Copy</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Copies the raw data so the original stays untouched.</div></div>

In [ ]:
# Create a separate working copy of the individual data. This keeps the original DataFrame
# available for comparisons.
df_cleaned = df.copy()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.1.2 Removing Redundant Columns</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Drops columns that add no information.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:4px solid #8a6a1c;border-radius:6px;padding:9px 16px 8px;margin:16px 0 4px;"><h4 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:15.5px;line-height:1.25;font-weight:700;color:#d4af37;">3.1.2.1 Detailed and Unrecoded Columns</h4><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Removes the detailed and unrecoded versions of columns.</div></div>

In [ ]:
# Identify extra detailed or unrecoded columns by their names. Remove these columns from the
# cleaned dataset to avoid keeping redundant versions.
redundant_columns = [
    col for col in df_cleaned.columns
    if "detailed" in col.lower() or "unrecoded" in col.lower()
]
df_cleaned.drop(columns=redundant_columns, inplace=True)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:4px solid #8a6a1c;border-radius:6px;padding:9px 16px 8px;margin:16px 0 4px;"><h4 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:15.5px;line-height:1.25;font-weight:700;color:#d4af37;">3.1.2.2 Constant Columns</h4><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Removes Country and Survey Year when they hold a single value.</div></div>

In [ ]:
# Check whether the listed country and survey-year columns have only one value. Remove them if
# they give no variation across records.
constant_columns = [
    col for col in ["Country", "Survey Year"]
    if df_cleaned[col].nunique(dropna=False) == 1
]
df_cleaned.drop(columns=constant_columns, inplace=True)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.1.3 General Data Type Formatting</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Converts text (object) columns to the pandas string type.</div></div>

In [ ]:
# Convert text-like individual columns to pandas string type. This makes later string handling and
# missing-value treatment more consistent.
string_columns = df_cleaned.select_dtypes(include=["object"]).columns
df_cleaned[string_columns] = df_cleaned[string_columns].astype("string")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.1.4 Display Formatting</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Defines a preview helper that shows an age of 0 as 'Under 1 year' (display only).</div></div>

In [ ]:
# Create a simple preview helper that displays a few records vertically. Rename the age label in
# the preview only; do not change stored data.
def display_data(data, rows=5):
    """Transposed preview with a friendlier Age label; stored values are unchanged."""
    preview = data.head(rows).T.copy()

    if "Age" in preview.index:
        preview.loc["Age"] = preview.loc["Age"].apply(
            lambda value: (
                "Under 1 year" if pd.notna(value) and value == 0
                else f"{value:.0f}" if pd.notna(value)
                else ""
            )
        )

    display(preview)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.1.5 Numeric Age Reference</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Creates a numeric Age series that the cleaning rules below rely on.</div></div>

In [ ]:
# Convert age values into numbers, treating unrecognized entries as missing. Later rules use age
# to decide when a question does not apply.
age = pd.to_numeric(df["Age"], errors="coerce")


<a id="sec3"></a>
<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 3</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">3. Individual Dataset — Cleaning</h1></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">3.2 Demographic Characteristics</h2></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.2.1 Marital Status</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Labels missing marital status as 'Minor' for people under 18.</div></div>

In [ ]:
# Clean marital status and label missing entries for people under 18 as minors. Report the
# resulting category counts for review.
col = "Marital status [Standardized version]"

marital = clean_text(df[col])
marital[marital.isna() & age.lt(18)] = "Minor"
df_cleaned[col] = marital

print("Marital status")
print(df_cleaned[col].value_counts(dropna=False))
print("Remaining missing values:", df_cleaned[col].isna().sum())


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">3.3 Education</h2></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.3.1 Literacy Status</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Marks missing literacy status as not applicable for children under 10.</div></div>

In [ ]:
# Mark missing literacy entries below age 10 as not applicable. Leave other values unchanged for
# later review.
col = "Literacy status"
df_cleaned.loc[age.lt(10) & df_cleaned[col].isna(), col] = "Not applicable - under 10"


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.3.2 Ever Attended School</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Standardizes to Yes/No, marks children under 4 as not applicable and removes the redundant status column.</div></div>

In [ ]:
# Standardize school-attendance history into consistent Yes/No labels. Apply age-based labels
# where the question does not apply and inspect the result.
col = "Ever attended school"

attended = standardize_yes_no(df[col])
under_4 = attended.isna() & age.lt(4)
attended[under_4] = "Not applicable"

if attended.isna().any():
    display(df.loc[attended.isna(), ["Age", col]])
    raise ValueError("Unexpected missing values remain.")

df_cleaned[col] = attended
df_cleaned.drop(columns=["Ever attended school status"], errors="ignore", inplace=True)

print("Not applicable", under_4.sum())
print("Remaining missing values:", df_cleaned[col].isna().sum())

df_cleaned[col].value_counts(dropna=False)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.3.3 Educational Level</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Marks missing educational level as not applicable for children under 10.</div></div>

In [ ]:
# Clean the reported education level and identify children below age 10 for whom it is not
# applicable. Review the frequency table afterward.
col = "Educational level [Standardized version]"

education_level = clean_text(df[col])
education_level[education_level.isna() & age.lt(10)] = "Not applicable - under 10"
df_cleaned[col] = education_level

print("Remaining missing values:", df_cleaned[col].isna().sum())
print("\nEducational level")

df_cleaned[col].value_counts(dropna=False)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.3.4 Number of Effective Years of Schooling</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing years of schooling with age-based rules and education-level averages.</div></div>

In [ ]:
# Convert years of schooling to numbers and compare them with education level and age. Use the
# coded rules to handle missing or inconsistent schooling information.
years_col = "Number of effective years of schooling"
education_col = "Educational level [Standardized version]"

years = pd.to_numeric(df[years_col], errors="coerce")
education = clean_text(df[education_col])
attendance = normalized(df["School attendance status"])
ever_attended = normalized(df["Ever attended school"])
current_level = normalized(df["Level of schooling presently attending"])

# Average years of primary-school attendees (age 10+), rounded to a fixed value
primary_reference = (
    age.ge(10) & is_(attendance, "yes") & is_(current_level, "primary") & years.notna()
)
primary_mean = years[primary_reference].mean()
primary_fixed = int(round(primary_mean))

# Average years per education level (age 10+), rounded
education_reference = age.ge(10) & years.notna() & education.notna()
education_means = (
    years[education_reference]
    .groupby(education[education_reference])
    .mean()
    .round()
)

# Children under 4 -> 0 years
under_4_mask = years.isna() & age.lt(4)
years[under_4_mask] = 0

# Children aged 4-9
age_4_to_9_missing = years.isna() & age.ge(4) & age.lt(10)
attending_yes = age_4_to_9_missing & is_(attendance, "yes")
never_attended = age_4_to_9_missing & is_(attendance, "no") & is_(ever_attended, "no")
previously_attended = age_4_to_9_missing & is_(attendance, "no") & is_(ever_attended, "yes")

years[attending_yes | previously_attended] = primary_fixed
years[never_attended] = 0

# Age 10+ -> average of their education level
adult_missing = years.isna() & age.ge(10) & education.notna()
years[adult_missing] = education[adult_missing].map(education_means).astype("float64")

if (years.dropna() % 1 != 0).any():
    raise ValueError("Non-integer values remain; review before converting to Int64.")

df_cleaned[years_col] = years.astype("Int64")

print(f"Primary average: {primary_mean:.2f} -> fixed value: {primary_fixed}")
print(f"Under age 4 set to 0: {under_4_mask.sum()}")
print(f"Age 4-9, attendance Yes assigned {primary_fixed}: {attending_yes.sum()}")
print(f"Age 4-9, never attended assigned 0: {never_attended.sum()}")
print(f"Age 4-9, previously attended assigned {primary_fixed}: {previously_attended.sum()}")
print(f"Age 10+, filled using education-level averages: {adult_missing.sum()}")
print(f"Remaining missing values: {df_cleaned[years_col].isna().sum()}")
print(f"Data type: {df_cleaned[years_col].dtype}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.3.5 School Attendance Status</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Standardizes to Yes/No and labels children under 4 as too young for school.</div></div>

In [ ]:
# Normalize current school attendance and use age-based rules for very young children. Show the
# resulting categories to check the changes.
col = "School attendance status"

attendance = standardize_yes_no(df[col])
attendance[attendance.isna() & age.lt(4)] = "Too young for school"
df_cleaned[col] = attendance

print(f"{col} | dtype: {df_cleaned[col].dtype}")
print(f"Remaining missing values: {df_cleaned[col].isna().sum()}")
print("\nValue counts:")
print(df_cleaned[col].value_counts(dropna=False).to_string())


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.3.6 Level of Schooling Presently Attending</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing levels from age, attendance and the agreed schooling-year ranges.</div></div>

In [ ]:
# Clean the level of school currently attended. Use school-attendance answers and years of
# schooling to classify missing or non-applicable cases.
col = "Level of schooling presently attending"

level = clean_text(df[col])
attendance = normalized(df["School attendance status"])
years = pd.to_numeric(df["Number of effective years of schooling"], errors="coerce")

# Children under 4
under_4 = level.isna() & age.lt(4)
level[under_4] = "Too young for school"

# People who are not attending
not_attending = level.isna() & is_(attendance, "no")
level[not_attending] = "Not currently attending"

# Current attendees: infer the level from years of schooling
level_from_years = pd.Series(pd.NA, index=level.index, dtype="string")
level_from_years[years.between(0, 2)] = "Kindergarten"
level_from_years[years.between(3, 8)] = "Primary"
level_from_years[years.between(9, 11)] = "Preparatory"
level_from_years[years.between(12, 14)] = "Secondary"
level_from_years[years.ge(15)] = "University"

classified = level.isna() & is_(attendance, "yes") & level_from_years.notna()
level[classified] = level_from_years[classified]

# Anything left is unexplained
not_stated = level.isna()
level[not_stated] = "Not stated"

df_cleaned[col] = level

print(f"Under 4: {under_4.sum()}")
print(f"Not currently attending: {not_attending.sum()}")
print(f"Classified using schooling-year ranges: {classified.sum()}")
print(f"Not stated: {not_stated.sum()}")
print(f"Remaining missing: {df_cleaned[col].isna().sum()}")
print(f"Data type: {df_cleaned[col].dtype}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.3.7 Type of School Attending</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Labels missing school type as 'Not attending school' or 'Not stated'.</div></div>

In [ ]:
# Clean the type of school and mark people who are not attending as not attending school. Report
# the distribution and remaining missing values.
col = "Type of school attending"

school_type = clean_text(df[col])
not_attending = school_type.isna() & is_(normalized(df["School attendance status"]), "no")
school_type[not_attending] = "Not attending school"

not_stated = school_type.isna()
school_type[not_stated] = "Not stated"

df_cleaned[col] = school_type

print(f"Not attending school: {not_attending.sum()}")
print(f"Not stated: {not_stated.sum()}")
print(f"Remaining missing: {df_cleaned[col].isna().sum()}")
print(f"Data type: {df_cleaned[col].dtype}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">3.4 Health and Social Protection</h2></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.4.1 Disability Status</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Standardizes disability status to Yes/No.</div></div>

In [ ]:
# Convert disability answers to a standard Yes/No form. This avoids separate categories caused by
# inconsistent formatting.
df_cleaned["Disability status"] = standardize_yes_no(df_cleaned["Disability status"])


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.4.2 Type of Disability</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing disability types with 'Not stated'.</div></div>

In [ ]:
# Clean the type-of-disability field and label missing reports as not stated. Print counts to make
# the category distribution easy to check.
col = "Type of disability"

df_cleaned[col] = clean_text(df[col]).fillna("Not stated")

print("Type of disability")
for category, count in df_cleaned[col].value_counts().items():
    print(f"{category}: {count}")

print(f"\nDisability status - No: {normalized(df['Disability status']).eq('no').sum()}")
print(f"Remaining missing values: {df_cleaned[col].isna().sum()}")
print(f"Data type: {df_cleaned[col].dtype}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.4.3 Chronic Disease</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Converts Yes/No into a nullable Boolean column.</div></div>

In [ ]:
# Convert chronic-disease Yes/No answers into nullable Boolean values. Unknown values remain
# missing instead of being forced to False.
df_cleaned["Chronic Disease"] = pd.array(
    strip_text(df["Chronic Disease"]).map({"Yes": True, "No": False}),
    dtype="boolean",
)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.4.4 Type of Health Insurance</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Combines the insurance columns into one 'Type of insurance' column placed after Chronic Disease.</div></div>

In [ ]:
# Combine the separate insurance-type indicators into the cleaned health-insurance representation.
# Review the categories after applying the rules.
insurance_types = {
    "Kind of Health Insurance- Governmental": "Governmental",
    "Kind of Health Insurance- Private": "Private",
    "Kind of Health Insurance- Other": "Other",
}

insurance = pd.Series("No health insurance", index=df_cleaned.index)
insurance.loc[df_cleaned["Health Insurance"].eq("Yes")] = "Unknown insurance type"

for column, insurance_type in insurance_types.items():
    insurance.loc[df_cleaned[column].eq("Yes")] = insurance_type

df_cleaned.drop(columns=["Health Insurance", *insurance_types], inplace=True)
df_cleaned.insert(
    df_cleaned.columns.get_loc("Chronic Disease") + 1,
    "Type of insurance",
    insurance.astype("string"),
)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.4.5 Social Security</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Converts Social Security to a nullable Boolean (Yes = True).</div></div>

In [ ]:
# Convert social-security responses into a Boolean column. This makes the field easier to use in
# later calculations.
df_cleaned["Social Security"] = pd.array(
    strip_text(df["Social Security"]).eq("Yes"),
    dtype="boolean",
)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">3.5 Labor Force and Employment</h2></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.5.1 Labor Force Status</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing labor-force status for children under 6.</div></div>

In [ ]:
# Fill blank labor-force status with the specified under-age label and print a summary of
# resulting categories.
col = "Labor force status in the current period"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not applicable - under 6")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.5.2 Main Activity Status</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Marks missing main activity as not applicable for children under 6.</div></div>

In [ ]:
# Clean main activity in the current period and distinguish records that do not apply because of
# age. Review the resulting counts.
col = "Main activity status in the current period [Standardized version]"

activity = clean_text(df[col])
activity[activity.isna() & age.lt(6)] = "Not applicable - under 6"
df_cleaned[col] = activity

category_order = [
    "Employed", "Homemaker (Housewife)", "Others", "Pensioners/retired/disabled",
    "Student", "Unemployed", "Not applicable - under 6", "Not stated",
]
counts = df_cleaned[col].value_counts(dropna=False)

print(f"{col} | dtype: {df_cleaned[col].dtype}")
for category in category_order:
    if category in counts.index:
        print(f"{category}: {counts[category]}")
print(f"Remaining missing: {df_cleaned[col].isna().sum()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.5.3 Status of Employment</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Labels blanks by age, labor-force status and main activity; unexplained blanks become Unemployed.</div></div>

In [ ]:
# Use current labor-force status and main activity to clean employment status. Different labels
# indicate when employment questions do not apply.
col = "Status of Employment"

labor_status = normalized(df["Labor force status in the current period"])
main_activity = normalized(
    df["Main activity status in the current period [Standardized version]"]
)

df_cleaned[col] = fill_employment_blanks(
    clean_text(df[col]), age, labor_status, main_activity
)

category_order = [
    "Employee", "Employer", "Own-account, self-employed",
    "Contributing (unpaid) family worker", "Unemployed",
    "Not applicable - not in labor force", "Not applicable - under 6",
]
counts = df_cleaned[col].value_counts(dropna=False)

print(f"{col} | dtype: {df_cleaned[col].dtype}")
for category in category_order:
    if category in counts.index:
        print(f"{category}: {counts[category]}")
print(f"Remaining missing: {df_cleaned[col].isna().sum()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.5.4 Occupation Classification for the Main Job</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Applies the same blank-labeling rules as employment status.</div></div>

In [ ]:
# Clean main-job occupation and apply the employment-specific missing-value helper. This separates
# people without a current job from unreported occupations.
col = "Occupation classification  for the main job [Standardized version]"

df_cleaned[col] = fill_employment_blanks(
    clean_text(df[col]), age, labor_status, main_activity
)

print(f"{col} | dtype: {df_cleaned[col].dtype}")
print(f"Unemployed: {df_cleaned[col].eq('Unemployed').sum()}")
print(f"Not applicable - under 6: {df_cleaned[col].eq('Not applicable - under 6').sum()}")
print(f"Not applicable - not in labor force: {df_cleaned[col].eq('Not applicable - not in labor force').sum()}")
print(f"Remaining missing: {df_cleaned[col].isna().sum()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.5.5 Industry Classification for the Main Job</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing industry with 'Not applicable - no current job'.</div></div>

In [ ]:
# Fill missing main-job industry entries with the no-current-job label. Show the breakdown and
# number of not-stated responses.
col = "Industry classification  for the main job [Standardized version]"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not applicable - no current job")

print(df_cleaned[col].value_counts().to_string())
print(f"Not stated: {df_cleaned[col].eq('Not stated').sum()}")
print(f"Data type: {df_cleaned[col].dtype}")
print(f"Remaining missing: {df_cleaned[col].isna().sum()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.5.6 Sector of Employment</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Merges 'Not stated' and 'Not applicable - unemployed' into 'Not applicable - no current employment'.</div></div>

In [ ]:
# Standardize sector of employment and replace selected missing or not-applicable categories
# according to the cleaning rules.
col = "Sector of employment [Standardized version]"

df_cleaned[col] = (
    clean_text(df_cleaned[col])
    .replace(["Not stated", "Not applicable - unemployed"], pd.NA)
    .fillna("Not applicable - no current employment")
)
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">3.6 Father Characteristics</h2></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.6.1 Father Present at Home</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the indicator for whether the father lives in the home. Keep unreported information in a
# clearly named category.
col = "Father present at home"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.6.2 ID of the Father</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Converts the father's ID to a nullable integer.</div></div>

In [ ]:
# Convert father IDs to nullable integers so missing identifiers are allowed. Report how many IDs
# are available.
col = "ID of the father"

df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors="coerce").astype("Int64")

print(f"Available IDs: {df_cleaned[col].notna().sum()}")
print(f"NA values: {df_cleaned[col].isna().sum()}")
print(f"Data type: {df_cleaned[col].dtype}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.6.3 Age of the Father</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Converts the father's age to a nullable integer.</div></div>

In [ ]:
# Convert the reported father age to nullable integers and summarize missing versus available
# ages.
col = "Age of the father"

df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors="coerce").astype("Int64")

print(f"Available values: {df_cleaned[col].notna().sum()}")
print(f"NA values: {df_cleaned[col].isna().sum()}")
print(f"Data type: {df_cleaned[col].dtype}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.6.4 Marital Status of the Father</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the father’s marital status and use a consistent label for missing answers. Display the
# category counts.
col = "Marital status of the father [Standardized version]"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.6.5 Main Activity of the Father</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the father’s main activity field and label unreported values. Display the resulting
# distribution.
col = "Main activity of the father [Standardized version]"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.6.6 Occupation Classification for the Father's Main Job</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Treats 'Not applicable - no main job' and blanks as 'Not stated'.</div></div>

In [ ]:
# Standardize the father’s occupation category and reconcile the stated non-applicable or unknown
# labels.
col = "Occupation classification  for the main job of the father [Standardized version]"

df_cleaned[col] = (
    clean_text(df_cleaned[col])
    .replace("Not applicable - no main job", "Not stated")
    .fillna("Not stated")
)
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.6.7 Industry Classification for the Father's Main Job</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the father’s industry classification and label missing answers as not stated. Display a
# summary.
col = "Industry classification for the main job of the father [Standardized version]"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.6.8 Sector of Employment of the Father</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the father’s employment-sector category and give missing answers a consistent label.
col = "Sector of employment of the father [Standardized version]"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">3.7 Mother Characteristics</h2></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.7.1 Mother Present at Home</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the indicator for whether the mother is present at home. Missing reports are shown as not
# stated.
col = "Mother present at home"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.7.2 ID of the Mother</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Converts the mother's ID to a nullable integer.</div></div>

In [ ]:
# Convert the mother’s ID to nullable integers. Check the number of available values and the final
# data type.
col = "ID of the mother"

df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors="coerce").astype("Int64")

print(f"Available IDs: {df_cleaned[col].notna().sum()}")
print(f"Data type: {df_cleaned[col].dtype}")
print(f"NA values: {df_cleaned[col].isna().sum()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.7.3 Age of the Mother</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Converts the mother's age to a nullable integer.</div></div>

In [ ]:
# Convert the mother’s age to nullable integers. Print counts and basic range checks for the
# available ages.
col = "Age of the mother"

df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors="coerce").astype("Int64")

print(f"Available ages: {df_cleaned[col].notna().sum()}")
print(f"Minimum age: {df_cleaned[col].min()}")
print(f"Maximum age: {df_cleaned[col].max()}")
print(f"Data type: {df_cleaned[col].dtype}")
print(f"NA values: {df_cleaned[col].isna().sum()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.7.4 Marital Status of the Mother</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the mother’s marital status category and use not stated for missing responses.
col = "Marital status of the mother [Standardized version]"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.7.5 Educational Level of the Mother</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the mother’s education-level category and make missing responses easy to identify.
col = "Educational level of the mother [Standardized version]"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.7.6 Main Activity of the Mother</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the mother’s main activity classification and report the resulting response counts.
col = "Main activity of the mother [Standardized version]"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.7.7 Employment Status of the Mother</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the mother’s employment-status category and label unanswered records consistently.
col = "Employment status of the mother [Standardized version]"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.7.8 Occupation Classification for the Mother's Main Job</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the mother’s occupation classification and identify unreported information.
col = "Occupation classification  for the main job of the mother [Standardized version]"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.7.9 Industry Classification for the Mother's Main Job</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the mother’s industry classification and identify unreported information.
col = "Industry classification for the main job of the mother [Standardized version]"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.7.10 Sector of Employment of the Mother</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing values with 'Not stated'.</div></div>

In [ ]:
# Clean the mother’s employment-sector category and identify unreported information.
col = "Sector of employment of the mother [Standardized version]"

df_cleaned[col] = clean_text(df_cleaned[col]).fillna("Not stated")
report(df_cleaned, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">3.8 Final Cleaning Check</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Confirms the structure and the remaining gaps of the cleaned individual data.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.8.1 Columns and Data Types</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Reviews the cleaned columns and their data types.</div></div>

In [ ]:
# Review the final structure and data types of the cleaned individual dataset.
df_cleaned.info()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">3.8.2 Remaining Missing Values</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Counts any missing values left after cleaning.</div></div>

In [ ]:
# Count remaining missing values after individual-level cleaning to see which fields still need
# attention.
df_cleaned.isnull().sum()


<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 4</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">4. Household Dataset — Raw Data Inspection and Preparation</h1></div>

<a id="sec4"></a>
<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 4</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">4. Household Dataset — Preprocessing</h1></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">4.1 Loading the Dataset</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Loads the household-level survey data into <code style="background-color:#17130a;color:#f3d77a;border:1px solid #2b2410;border-radius:4px;padding:1px 6px;font-family:SFMono-Regular,Consolas,'Liberation Mono',monospace;font-size:.92em;">hh_raw</code>.</div></div>

In [ ]:
# Load the household-level Excel file into a separate DataFrame for cleaning and analysis.
hh_raw = load_excel(HOUSEHOLD_FILE)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">4.2 Initial Data Inspection</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">A first look at the raw household data.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">4.2.1 Preview the First Rows</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Shows the first five records.</div></div>

In [ ]:
# Preview household records to inspect the fields and confirm that the data loaded.
hh_raw.head()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">4.2.2 Columns and Data Types</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Lists every column with its data type and non-null count.</div></div>

In [ ]:
# Inspect household column types and completeness before cleaning.
hh_raw.info()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">4.2.3 Missing Values</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Displays the missing-value count of every column.</div></div>

In [ ]:
# Show a complete column-by-column household missing-value table, without truncating rows.
with pd.option_context("display.max_rows", None):
    display(hh_raw.isna().sum().to_frame("Missing values"))


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">4.2.4 Duplicate Rows</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Counts fully duplicated rows.</div></div>

In [ ]:
# Check how many household rows are exact duplicates.
hh_raw.duplicated().sum()


<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 5</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">5. Household Data Cleaning</h1></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">5.1 Creating the Cleaning Copy</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Copies the raw data so the original stays untouched.</div></div>

In [ ]:
# Create a working copy of the household dataset so the raw data remains available.
hh_clean = hh_raw.copy()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">5.2 Removing Redundant Columns</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Drops constant, spouse and detailed-expenditure columns.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">5.2.1 General and Spouse Columns</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Lists constant, non-standardized and spouse columns to remove.</div></div>

In [ ]:
# Define name patterns for spouse-related columns and identify general household fields that
# should be excluded.
spouse_prefixes = (
    "agesp_", "sexsp_", "litsp_", "edusp_",
    "massp_", "emssp_", "occsp_", "indsp_", "typewksp_",
)

general_columns_to_drop = [
    col for col in hh_clean.columns
    if col in ["Country", "Survey Year"]
    or "detailed" in col.lower()
    or "unrecoded" in col.lower()
    or col.startswith(spouse_prefixes)
]


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">5.2.2 Detailed Expenditure Columns</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Lists the detailed food and non-food expenditure columns to remove (totals are kept).</div></div>

In [ ]:
# List detailed food-expenditure items that are not needed once the chosen summary variables are
# kept.
# Food: keep the totals, drop the detailed components.
detailed_food_columns_to_drop = [
    "Rice expenditures", "Grains and flour expenditures", "Bread and cakes",
    "Pasta expenditures", "Cereal preparation", "Own-Produced Bread and Cereals",
    "Sheep and goats Meat expenditures", "Cow meat expenditures",
    "Other meat expenditures", "Chicken meat expenditure",
    "Luncheon and Tinned meat", "Own-Produced Meat and Poultry",
    "Fish expenditures", "Seafood expenditures", "Own-Produced Fish and Seafood",
    "Milk expenditures", "Yogurt expenditures", "Cheese expenditures",
    "Eggs expenditures", "Own-Produced Dairy Products",
    "Olive oil expend.", "Other oils expend.", "Butter and margarine",
    "Fat expenses", "Own-Produced Oil & Fats", "Own-Produced Fruits",
    "Own-Produced Vegetables", "Own-Produced Sugar and Jam",
    "Own-Produced Spices", "Own-Produced Beverages",
    "Other non-specified own-produced food", "Tobacco expenditures",
    "Purchased Bread and Cereals", "Purchased Meat and Poultry",
    "Purchased Fish and Seafood", "Purchased Dairy Products",
    "Purchased Oil & Fats", "Fruits expenses", "Vegetables expenses",
    "Sugar and jam expenditure", "Spices expenditures", "Beverages expenditure",
    "Other non-specified food expenses",
]

# Non-food: keep the main COICOP totals, drop the detailed components.
detailed_non_food_columns_to_drop = [
    "Clothes for men, women and children", "Clothing material", "Tailoring charges",
    "Footwear expenditures", "School Clothes, Uniforms and Footwear",
    "Actual rentals \nfor housing", "Imputed rentals for housing",
    "Maintenance and repair for the Dwelling",
    "Water supply and miscellaneous services relating to the dwelling",
    "Electricity, gas and other fuels", "Housing furnishings and furniture",
    "Housing furnishings and furniture repair", "Household textiles",
    "Major Household appliances", "Small Household appliances",
    "Repair of appliances", "Glassware, tableware and household utensils",
    "Major other housing equipment and operation",
    "Small other housing equipment and operation",
    "Goods and services for routine household maintenance",
    "Pharmaceutical and other medical products",
    "Therapeutic appliances and equipment", "Outpatient and Hospital services",
    "Expenses on medical services Abroad", "Purchase of transportation vehicles",
    "Operation of personal transport equipment", "Transportation services",
    "School Transportation services", "Postal telephone and telefax services",
    "Telephone and telefax equipment", "Durables for recreation and culture",
    "Recreation and cultural equipments", "Recreation and culture services",
    "School recreation and culture services",
    "Expenses on pre-primary and primary education",
    "Expenses lower and upper-secondary education", "Expenses tertiary education",
    "Expenses on education not defined by level", "Expenses on Education Abroad",
    "Catering services", "School restaurants expenditures",
    "Accommodation and hotel expenses", "Personal care and personal effects",
    "Social protection", "Insurance and financial services",
    "Penalties and other fees", "Private transfers given",
]


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">5.2.3 Applying Column Removal</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Drops all listed columns plus completely empty ones in a single step.</div></div>

In [ ]:
# Find fully empty household columns and combine all removal lists without duplicates. Drop the
# selected fields from the working dataset.
empty_columns = [col for col in hh_clean.columns if hh_clean[col].isna().all()]

columns_to_drop = list(dict.fromkeys(
    general_columns_to_drop
    + detailed_food_columns_to_drop
    + detailed_non_food_columns_to_drop
    + empty_columns
))

hh_clean.drop(
    columns=[col for col in columns_to_drop if col in hh_clean.columns],
    inplace=True,
)

print(f"Dropped columns: {len(columns_to_drop)}")
print(f"Remaining columns: {hh_clean.shape[1]}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">5.3 General Data Type Formatting</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Converts text columns to the pandas string type and trims spaces.</div></div>

In [ ]:
# Trim leading and trailing spaces in all household text columns so equivalent responses match.
string_columns = hh_clean.select_dtypes(include=["object", "string"]).columns
hh_clean[string_columns] = hh_clean[string_columns].apply(strip_text)


<a id="sec5"></a>
<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 5</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">5. Household Dataset — Cleaning</h1></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">5.4 Housing Characteristics</h2></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">5.4.1 Number of Rooms</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Translates the Arabic room labels to English.</div></div>

In [ ]:
# Translate room-count labels into English categories while preserving the original values.

room_translation = {
    "\u063a\u0631\u0641\u0629 \u0648\u0627\u062d\u062f\u0629": "One room", "\u063a\u0631\u0641\u062a\u064a\u0646": "Two rooms", "\u062b\u0644\u0627\u062b \u063a\u0631\u0641": "Three rooms",
    "\u0627\u0631\u0628\u0639 \u063a\u0631\u0641": "Four rooms", "\u062e\u0645\u0633 \u063a\u0631\u0641": "Five rooms", "\u0633\u062a \u063a\u0631\u0641": "Six rooms",
    "\u0633\u0628\u0639 \u063a\u0631\u0641": "Seven rooms", "\u062b\u0645\u0627\u0646 \u063a\u0631\u0641": "Eight rooms", "\u062a\u0633\u0639 \u063a\u0631\u0641": "Nine rooms",
    "\u0639\u0634\u0631 \u063a\u0631\u0641": "Ten rooms",
}

col = "Number of rooms"
hh_clean[col] = hh_clean[col].replace(room_translation).astype("string")

report(hh_clean, col)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">5.5 Household Assets</h2></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">5.5.1 Durable Ownership</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Converts the ownership labels of all asset columns to Boolean values.</div></div>

In [ ]:
# Locate household asset-indicator columns and standardize their values for comparison.
asset_columns = [
    col for col in hh_clean.columns
    if col.startswith("Has ") or col == "watermotor"
]
ownership = {"Own one or many": True, "None": False}

for col in asset_columns:
    hh_clean[col] = hh_clean[col].map(ownership).astype("boolean")

print(f"Converted columns: {len(asset_columns)}")
print(f"Remaining missing: {hh_clean[asset_columns].isna().sum().sum()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">5.6 Expenditure and Income</h2></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">5.6.1 Total Gifts and In-Kind Food</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills missing gifts with 0 after verifying they equal total food minus purchased food.</div></div>

In [ ]:
# Check food-gift amounts against purchased and total food values. Apply the rules for cleaning
# the total gifts and in-kind food field.
col = "Total Gifts and in-kind food"

gifts = pd.to_numeric(hh_clean[col], errors="coerce")
purchased_food = pd.to_numeric(hh_clean["Total purchased food"], errors="coerce").fillna(0)
total_food = pd.to_numeric(hh_clean["Food and non-alcoholic beverages"], errors="coerce")

missing = gifts.isna()
expected_gifts = total_food - purchased_food

if not np.isclose(expected_gifts[missing], 0, atol=0.01).all():
    raise ValueError("Unverified missing gift values were found.")

hh_clean.loc[missing, col] = 0

print(f"Filled with 0: {missing.sum()}")
print(f"Remaining missing: {hh_clean[col].isna().sum()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">5.6.2 Expenditure Values</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills blank expenditure components with 0 (they then add up to total expenditure).</div></div>

In [ ]:
# Identify major household spending categories and convert them into consistent numeric
# expenditure values.
expenditure_columns = [
    "Total purchased food", "Alcohol and tobacco",
    "Clothing and footwear", "Health", "Transport",
    "Communication", "Recreation and culture", "Education",
    "Restaurants and hotels", "Miscellaneous goods and services",
    "Total expenses on durables", "Lumpy expenses",
    "Expenditures on taxes and levies",
]

filled_values = int(hh_clean[expenditure_columns].isna().sum().sum())
hh_clean[expenditure_columns] = hh_clean[expenditure_columns].fillna(0)

print(f"Filled with 0: {filled_values}")
print(f"Remaining missing: {hh_clean[expenditure_columns].isna().sum().sum()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">5.6.3 Income Values</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Fills blank income components with 0 (they then add up to total disposable income).</div></div>

In [ ]:
# Identify household income sources and prepare their values for total-income calculations.
income_columns = [
    "Net wages and salaries", "Self-employment income",
    "Income less expenses from rentals", "Property income",
    "Current transfers received",
]

filled_values = int(hh_clean[income_columns].isna().sum().sum())
hh_clean[income_columns] = hh_clean[income_columns].fillna(0)

print(f"Filled with 0: {filled_values}")
print(f"Remaining missing: {hh_clean[income_columns].isna().sum().sum()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">5.7 Household Head Characteristics</h2></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">5.7.1 Marital Status of the Head</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Sets 'Never married' for single heads under 18 with no children; other blanks become 'Not stated'.</div></div>

In [ ]:
# Clean the household head’s marital-status field and handle missing responses for heads below age
# 18.
col = "Marital status of the head [Standardized version]"

single_minor = (
    hh_clean[col].isna()
    & hh_clean["Age of the head"].lt(18)
    & hh_clean["Partnership and parenthood status of the head"].eq("Single Head, with no children")
)

hh_clean.loc[single_minor, col] = "Never married"
hh_clean[col] = hh_clean[col].fillna("Not stated").astype("string")

print(hh_clean[col].value_counts().to_string())
print(f"Remaining missing: {hh_clean[col].isna().sum()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">5.7.2 Employment Information of the Head</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Marks missing job fields as 'Not applicable - no current job' for heads without a job; other blanks become 'Not stated'.</div></div>

In [ ]:
# Clean employment-related fields for the household head with consistent handling of missing or
# not-applicable entries.
employment_columns = [
    "Status of Employment of the head [Standardized version]",
    "Occupation classification for the main job of the head [Standardized version]",
    "Industry classification for the main job of the head [Standardized version]",
    "Sector of employment of the head [Standardized version]",
]
no_job_activities = [
    "Pensioners/retired/disabled", "Homemaker (Housewife)",
    "Unemployed", "Student", "Others",
]

activity = hh_clean["Main activity status of the head [Standardized version]"]

for col in employment_columns:
    no_job = hh_clean[col].isna() & activity.isin(no_job_activities)
    hh_clean.loc[no_job, col] = "Not applicable - no current job"
    hh_clean[col] = hh_clean[col].fillna("Not stated").astype("string")

print(f"Cleaned columns: {len(employment_columns)}")
print(f"Remaining missing: {hh_clean[employment_columns].isna().sum().sum()}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">5.8 Final Cleaning Check</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Confirms the structure and the remaining gaps of the cleaned household data.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">5.8.1 Columns and Data Types</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Reviews the cleaned columns and their data types.</div></div>

In [ ]:
# Inspect the final types and non-missing counts in the cleaned household dataset.
hh_clean.info()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">5.8.2 Remaining Missing Values</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Displays every remaining missing-value count.</div></div>

In [ ]:
# Display remaining missing values in every household column after cleaning.
with pd.option_context("display.max_rows", None):
    display(hh_clean.isna().sum().to_frame("Missing values"))


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">5.8.3 Changing Columns Names</h2></div>

In [ ]:
# Define readable, consistent column names for individual and household information. Apply naming
# rules so shared variables can be matched across datasets.

BASE = {
    "unique unit identifier": "household_id", "household weight": "household_weight",
    "urban/rural residence": "urban_rural", "person identifier": "person_id", "id": "person_id",
    "relationship to head": "relation_to_head", "marital status": "marital_status",
    "literacy status": "literacy_status", "educational level": "education_level",
    "ever attended school": "ever_attended_school",
    "number of effective years of schooling": "years_of_schooling",
    "school attendance status": "school_attendance_status",
    "level of schooling presently attending": "current_school_level",
    "type of school attending": "current_school_type", "disability status": "disability_status",
    "type of disability": "disability_type", "chronic disease": "chronic_disease",
    "type of insurance": "insurance_type", "social security": "has_social_security",
    "labor force status in the current period": "labor_force_status",
    "main activity status in the current period": "main_activity_status",
    "main activity status": "main_activity_status", "main activity": "main_activity_status",
    "status of employment": "employment_status", "employment status": "employment_status",
    "occupation classification for the main job": "job_occupation",
    "industry classification for the main job": "job_industry",
    "sector of employment": "job_sector",
    "father present at home": "father_present_at_home", "mother present at home": "mother_present_at_home",
    "age": "age", "gender": "gender",
    "number of males": "num_males", "number of females": "num_females",
    "number of persons": "household_size", "number of persons under age 14": "num_under_14",
    "number of persons aged 65 or more": "num_aged_65_plus",
    "household composition": "household_composition",
    "head living in couple": "head_living_in_couple", "number of earners": "num_earners",
    "source of income": "income_source",
    "partnership and parenthood status": "partnership_parenthood_status",
    "type of dwelling": "dwelling_type", "tenure of dwelling": "dwelling_tenure",
    "number of rooms": "num_rooms", "electrified": "has_electricity",
    "source of energy for lighting": "lighting_energy_source",
    "source of energy for cooking": "cooking_energy_source", "water facilities": "water_facility",
    "toilet facility": "toilet_facility", "toilet facility shared": "toilet_shared",
    "garbage disposal method": "garbage_disposal_method",
    "watermotor": "has_water_motor",
    "total purchased food": "exp_purchased_food", "total gifts and in-kind food": "exp_gifts_inkind_food",
    "food and non-alcoholic beverages": "exp_food_beverages", "alcohol and tobacco": "exp_alcohol_tobacco",
    "clothing and footwear": "exp_clothing_footwear", "housing and utilities": "exp_housing_utilities",
    "furniture & housing equipment": "exp_furniture_equipment", "health": "exp_health",
    "transport": "exp_transport", "communication": "exp_communication",
    "recreation and culture": "exp_recreation_culture", "education": "exp_education",
    "restaurants and hotels": "exp_restaurants_hotels",
    "miscellaneous goods and services": "exp_misc_goods_services",
    "total food expenditures": "exp_total_food", "total expenses on durables": "exp_durables",
    "actual and imputed rentals for housing": "exp_housing_rent", "total expenditures": "exp_total",
    "total expenditures on non-food and non-durables": "exp_total_nonfood_nondurables",
    "lumpy expenses": "exp_lumpy", "expenditures on taxes and levies": "exp_taxes_levies",
    "net wages and salaries": "income_wages", "self-employment income": "income_self_employment",
    "income less expenses from rentals": "income_rentals", "property income": "income_property",
    "current transfers received": "income_transfers", "total disposable income": "income_total_disposable",
}
HAS = [("computer", "computer"), ("internet", "internet"), ("car", "vehicle"),
       ("bicycle", "bicycle_motorcycle"), ("tv", "tv"), ("satellite", "satellite_dish"),
       ("dvd", "audio_player"), ("camera", "camera"), ("telephone", "phone"),
       ("refrigerator", "refrigerator"), ("cooker", "cooker_stove"), ("microwave", "microwave"),
       ("food processor", "food_processor"), ("washing", "washing_machine"),
       ("dishwasher", "dishwasher"), ("air conditioner", "air_conditioner"), ("fan", "fan"),
       ("water heater", "water_heater"), ("heater", "heater"), ("sewing", "sewing_machine"),
       ("vacuum", "vacuum_cleaner"), ("iron", "iron"), ("water motor", "water_motor"),
       ("other", "other_durables")]

def simple(col):
    s = re.sub(r"\s+", " ", col.replace("[Standardized version]", "")).strip().lower()
    if s.startswith("has "):
        return "has_" + next(v for k, v in HAS if k in s)
    m = re.match(r"(.+) of the (father|mother|head)$", s)
    base, role = (m.group(1), m.group(2)) if m else (s, None)
    name = BASE.get(base) or re.sub(r"\W+", "_", base).strip("_")
    return f"{role}_{name}" if role and not name.startswith(role) else name

for label, data in [("df_cleaned", df_cleaned), ("hh_clean", hh_clean)]:
    new_cols = [simple(c) for c in data.columns]
    assert len(set(new_cols)) == len(new_cols), "Duplicate column names!"
    print(f"\n{label}: {len(new_cols)} columns\n")
    for old, new in zip(data.columns, new_cols):
        print(f"{old}  ->  {new}")
    data.columns = new_cols


<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 6</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">6. Merge and Validate the Cleaned Datasets</h1></div>

<a id="sec6"></a>
<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 6</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">6. Merging Individual and Household Data</h1><div style="height:1px;background-color:#8a6a1c;margin:14px 0 10px;"></div><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:13.5px;line-height:1.55;color:#a8a086;font-style:italic;">Joins every person to the cleaned record of the household they live in, so the rest of the analysis runs on one table.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">6.1 Checking the Join Key</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Confirms the household key is complete, has the same type on both sides and is unique, every person has a household, the shared columns agree, and household sizes reconcile.</div></div>

In [ ]:
# Check the household ID and other shared fields in each cleaned dataset before combining them.
# Verify that keys and columns are available.
key = "household_id"
shared_columns = ["region", "area", "urban_rural", "household_weight"]

# 1) The key must exist on both sides, be complete and have a compatible type.
for name, data in [("individual", df_cleaned), ("household", hh_clean)]:
    if key not in data.columns:
        raise KeyError(f"'{key}' is missing from the {name} data.")
    if data[key].isna().any():
        raise ValueError(f"The {name} data has missing values in '{key}'.")

both_integer = all(pd.api.types.is_integer_dtype(d[key]) for d in (df_cleaned, hh_clean))
if not both_integer and df_cleaned[key].dtype != hh_clean[key].dtype:
    raise TypeError(f"Key types differ: {df_cleaned[key].dtype} vs {hh_clean[key].dtype}")

# 2) One row per household on the "one" side of the join.
print(f"Households : {len(hh_clean):,} rows | unique keys: {hh_clean[key].nunique():,}")
if not hh_clean[key].is_unique:
    raise ValueError("The household key is not unique: the merge would duplicate individuals.")

# 3) Every person has a household (households without people are only reported).
print(f"Individuals: {len(df_cleaned):,} rows | households referenced: {df_cleaned[key].nunique():,}")
orphans = (~df_cleaned[key].isin(hh_clean[key])).sum()
print(f"Individuals without a matching household: {orphans}")
print(f"Households without any individual       : {(~hh_clean[key].isin(df_cleaned[key])).sum()}")
if orphans:
    raise ValueError("Some individuals have no household record.")

# 4) The shared columns must hold the same values on both sides before one copy is dropped.
check = df_cleaned[[key] + shared_columns].merge(
    hh_clean[[key] + shared_columns], on=key, suffixes=("_ind", "_hh"), validate="many_to_one"
)
for col in shared_columns:
    print(f"{col}: mismatches = {(check[col + '_ind'] != check[col + '_hh']).sum()}")

# 5) The household size reported in the household file should equal the people found in the individual file.
size_check = (
    hh_clean[[key, "household_size"]]
    .merge(df_cleaned.groupby(key).size().rename("Persons in individual file").reset_index(),
           on=key, how="left", validate="one_to_one")
)
size_gap = size_check["household_size"] != size_check["Persons in individual file"]
print(f"Households whose size differs between the two files: {size_gap.sum():,}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">6.2 Merging the Datasets</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Left-joins the household columns onto the individuals (many persons to one household).</div></div>

In [ ]:
# Merge person records with their household information using household ID. Keep all individual
# rows by using a left join.
df_merged = df_cleaned.merge(
    hh_clean.drop(columns=shared_columns),
    on=key,
    how="left",
    validate="many_to_one",
)

if len(df_merged) != len(df_cleaned):
    raise ValueError("The merge changed the number of individuals.")
if not np.array_equal(df_merged[key].to_numpy(), df_cleaned[key].to_numpy()):
    raise ValueError("The merge changed the order or the keys of the individuals.")

print(f"Merged shape: {df_merged.shape}")
print(f"Individual columns: {df_cleaned.shape[1]}")
print(f"Household columns added: {df_merged.shape[1] - df_cleaned.shape[1]}")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">6.3 Merged Data Check</h2><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Shows the data types of the merged table and the household columns that still contain gaps.</div></div>

In [ ]:
# Check data types and missing household columns after the merge to catch unmatched or incomplete
# records.
print(df_merged.dtypes.astype(str).value_counts().to_string())

household_added = [col for col in hh_clean.columns if col not in shared_columns + [key]]
missing_after_merge = df_merged[household_added].isna().sum()
print("\nHousehold columns with missing values:", int((missing_after_merge > 0).sum()))


<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 7</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">7. Data Modeling</h1><div style="height:1px;background-color:#8a6a1c;margin:14px 0 10px;"></div><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:13.5px;line-height:1.55;color:#a8a086;font-style:italic;">Builds, validates, and finalizes the SQLite analytical data model used for structured querying and downstream analysis.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">7.1 Prepare Household-Level Analytical Data</h2></div>

In [ ]:
# Prepare household-level analytical data
# income groups, asset groups, deprivation groups, and household segments.

# ============================================================
# 1. Create one record per household
# ============================================================

hh = (
    df_merged
    .drop_duplicates("household_id")
    .set_index("household_id")
    .copy()
)

by_hh = df_merged.groupby("household_id")

# ============================================================
# 2. Convert required numeric columns
# ============================================================

hh["income_total_disposable"] = pd.to_numeric(
    hh["income_total_disposable"],
    errors="coerce"
)

hh["household_size"] = pd.to_numeric(
    hh["household_size"],
    errors="coerce"
)

# ============================================================
# 3. Calculate household income per capita
# ============================================================

valid_household_size = hh["household_size"].gt(0)

hh["income_per_capita"] = (
    hh["income_total_disposable"]
    .div(hh["household_size"].where(valid_household_size))
)

print(
    "Valid income per capita values:",
    hh["income_per_capita"].notna().sum()
)

print(
    "Invalid household-size values:",
    (~valid_household_size).sum()
)

# ============================================================
# 4. Household-level health and social-protection indicators
# ============================================================

hh["any_disabled"] = by_hh["disability_status"].apply(
    lambda s: s.eq("disabled").any()
)

hh["any_chronic"] = by_hh["chronic_disease"].apply(
    lambda s: s.fillna(False).astype(bool).any()
)

hh["any_covered"] = by_hh["has_social_security"].apply(
    lambda s: s.fillna(False).astype(bool).any()
)

# ============================================================
# 5. Household dependency indicators
# ============================================================

size = hh["household_size"]

dependents = (
    hh["num_under_14"] +
    hh["num_aged_65_plus"]
)

working_age = (
    size - dependents
).replace(0, np.nan)

# ============================================================
# 6. Income quintiles
# ============================================================

LABELS = [
    "Bottom 20%",
    "Lower-middle 20%",
    "Middle 20%",
    "Upper-middle 20%",
    "Top 20%"
]

quintile = lambda s: pd.qcut(
    s.rank(method="first"),
    5,
    labels=LABELS
)

hh["income_group"] = quintile(
    hh["income_per_capita"]
)

# ============================================================
# 7. Asset ownership score and asset quintiles
# ============================================================

assets = [
    c for c in hh.columns
    if c.startswith("has_")
    and c not in [
        "has_social_security",
        "has_electricity"
    ]
]

score = (
    hh[assets]
    .fillna(False)
    .astype(int)
    .sum(axis=1)
)

hh["asset_group"] = quintile(score)

# ============================================================
# 8. Housing deprivation
# ============================================================

words = [
    "One", "Two", "Three", "Four", "Five",
    "Six", "Seven", "Eight", "Nine", "Ten"
]

rooms = (
    hh["num_rooms"]
    .astype("string")
    .str.split()
    .str[0]
    .map(dict(zip(words, range(1, 11))))
)

rooms = pd.to_numeric(
    rooms,
    errors="coerce"
)

deprivations = (
    (size / rooms > 3).astype(int)
    + hh["water_facility"].ne("Piped supply").astype(int)
    + (
        hh["toilet_facility"].ne("Yes and connected to sewage")
        | hh["toilet_shared"].eq("Shared toilet")
    ).astype(int)
    + hh["has_electricity"].ne("Electrified").astype(int)
    + (
        ~hh["cooking_energy_source"].isin(
            ["Gas", "Electricity"]
        )
    ).astype(int)
)

hh["housing_deprivation"] = pd.cut(
    deprivations,
    bins=[-1, 0, 1, 5],
    labels=[
        "Not deprived",
        "Moderately deprived",
        "Severely deprived"
    ]
)

# ============================================================
# 9. Household segmentation variables
# ============================================================

hh["size_group"] = pd.cut(
    size,
    bins=[0, 3, 5, np.inf],
    labels=[
        "Small (1-3)",
        "Medium (4-5)",
        "Large (6+)"
    ]
)

dependency_ratio = (
    dependents / working_age
)

hh["dependency_group"] = np.where(
    dependency_ratio < 1,
    "Low dependency (<1)",
    "High dependency (>=1)"
)

hh["earners_group"] = pd.cut(
    hh["num_earners"],
    bins=[-1, 0, 1, np.inf],
    labels=[
        "No earner",
        "One earner",
        "Two or more earners"
    ]
)

hh["children_group"] = np.where(
    hh["num_under_14"] > 0,
    "With children",
    "No children"
)

hh["elderly_group"] = np.where(
    hh["num_aged_65_plus"] > 0,
    "With elderly (65+)",
    "No elderly"
)

hh["health_group"] = np.where(
    hh["any_disabled"] | hh["any_chronic"],
    "Has disabled/chronic member",
    "None"
)

hh["social_protection_group"] = np.where(
    hh["any_covered"],
    "Covered",
    "Not covered"
)

# ============================================================
# 10. Geographic classification
# ============================================================

hh["macro_region"] = (
    hh["area"]
    .astype("string")
    .str.replace(
        r"^(Urban|Rural) (?!Governorates)",
        "",
        regex=True
    )
)

# ============================================================
# 11. Gender of household head
# ============================================================

hh["head_gender_group"] = hh["head_gender"].map(
    {
        "Male": "Male-headed",
        "Female": "Female-headed"
    }
)

# ============================================================
# 12. Add household-level groups back to merged data
# ============================================================

group_cols = [
    "income_group",
    "asset_group",
    "housing_deprivation",
    "size_group",
    "dependency_group",
    "earners_group",
    "children_group",
    "elderly_group",
    "health_group",
    "social_protection_group",
    "macro_region",
    "head_gender_group",
]

for column in group_cols:
    df_merged[column] = df_merged[
        "household_id"
    ].map(hh[column])

    print(
        f"\n{column} (% of households)"
    )

    print(
        (
            hh[column]
            .value_counts(normalize=True)
            .sort_index()
            * 100
        )
        .round(1)
        .to_string()
    )

# ============================================================
# 13. Preview the final household analytical table
# ============================================================

display(
    hh[
        [
            "income_total_disposable",
            "household_size",
            "income_per_capita",
            "income_group",
            "num_earners",
            "size_group",
            "dependency_group",
            "housing_deprivation"
        ]
    ].head()
)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">7.1.1 Feature Engineering</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Creates new household-level variables (economic, income structure, spending structure, demographics, human capital, housing and assets) from the cleaned data, checks their quality and flags which ones are safe to use as predictors.</div></div>

In [ ]:
# Calculate simple money features for each household: income per person, spending per person,
# saving, and whether the household is below the poverty line.

FE = []   # list that remembers the names of the new columns

# Short names for the columns we use (the fe_ prefix avoids clashing with other cells)
fe_size = hh["household_size"].astype(float)
fe_children = hh["num_under_14"].astype(float)
fe_elderly = hh["num_aged_65_plus"].astype(float)
fe_income = hh["income_total_disposable"].astype(float)
fe_spend = hh["exp_total"].astype(float)

# 1) Money per person (so a family of 2 and a family of 8 can be compared)
add_feature(hh, "income_per_capita", safe_divide(fe_income, fe_size), FE)
add_feature(hh, "expenditure_per_capita", safe_divide(fe_spend, fe_size), FE)
add_feature(hh, "log_income_per_capita", np.log(hh["income_per_capita"].where(hh["income_per_capita"] > 0)), FE)     # log makes the very rich households less extreme

# 2) Saving: positive = saves money, negative = spends more than it earns
add_feature(hh, "saving_rate", safe_divide(fe_income - fe_spend, fe_income), FE)
add_feature(hh, "deficit_household", (fe_spend > fe_income).astype(int), FE)      # 1 = spends more than it earns

# 3) Poverty line = 60% of the typical (median) income per person
fe_median = weighted_quantile(hh["income_per_capita"], hh["household_weight"], 0.5)
add_feature(hh, "relative_income", hh["income_per_capita"] / fe_median, FE)       # 1.0 = exactly the typical household
add_feature(hh, "relative_poverty", (hh["relative_income"] < 0.6).astype(int), FE)  # 1 = below the poverty line

print(f"Typical income per person (median): {fe_median:,.0f} EGP")
print(f"Poverty line (60% of the median)  : {0.6 * fe_median:,.0f} EGP")
print(f"Households below the poverty line : {hh['relative_poverty'].mean() * 100:.1f}%")
print(f"Households spending more than they earn: {hh['deficit_household'].mean() * 100:.1f}%")


In [ ]:
# ============================================================
# Household Demographic and Human-Capital Features
# ============================================================

FE = globals().get("FE", [])

fe_dependents = fe_children + fe_elderly
fe_workers = (fe_size - fe_dependents).replace(0, np.nan)

add_feature(
    hh,
    "child_ratio",
    safe_divide(fe_children, fe_size),
    FE
)

add_feature(
    hh,
    "elderly_ratio",
    safe_divide(fe_elderly, fe_size),
    FE
)

add_feature(
    hh,
    "earner_ratio",
    safe_divide(hh["num_earners"], fe_size),
    FE
)

add_feature(
    hh,
    "dependency_ratio",
    safe_divide(fe_dependents, fe_workers),
    FE
)

add_feature(
    hh,
    "head_age_group",
    pd.cut(
        hh["head_age"],
        bins=[0, 29, 44, 59, 200],
        labels=["Under 30", "30-44", "45-59", "60+"]
    ),
    FE
)

add_feature(
    hh,
    "head_is_female",
    hh["head_gender"].eq("Female").astype(int),
    FE
)

fe_people = df_merged[
    [
        "household_id",
        "age",
        "years_of_schooling",
        "literacy_status",
        "main_activity_status",
        "insurance_type",
    ]
].copy()

fe_people["age"] = pd.to_numeric(
    fe_people["age"],
    errors="coerce"
)

fe_people["years_of_schooling"] = pd.to_numeric(
    fe_people["years_of_schooling"],
    errors="coerce"
)

fe_people["literate"] = (
    fe_people["literacy_status"]
    .astype("string")
    .eq("Can read AND write")
)

fe_people["employed"] = (
    fe_people["main_activity_status"]
    .astype("string")
    .eq("Employed")
)

fe_people["insured"] = (
    fe_people["insurance_type"]
    .astype("string")
    .ne("No health insurance")
)

fe_adults_only = fe_people[
    fe_people["age"].ge(18)
].copy()

add_feature(
    hh,
    "avg_adult_schooling",
    fe_adults_only
    .groupby("household_id")["years_of_schooling"]
    .mean(),
    FE
)

add_feature(
    hh,
    "adult_literacy_rate",
    fe_adults_only
    .groupby("household_id")["literate"]
    .mean(),
    FE
)

add_feature(
    hh,
    "adult_employment_rate",
    fe_adults_only
    .groupby("household_id")["employed"]
    .mean(),
    FE
)

add_feature(
    hh,
    "health_insured_share",
    fe_people
    .groupby("household_id")["insured"]
    .mean(),
    FE
)

print("Household analytical features created:")

display(
    hh[
        [
            "child_ratio",
            "elderly_ratio",
            "earner_ratio",
            "dependency_ratio",
            "avg_adult_schooling",
            "adult_literacy_rate",
            "adult_employment_rate",
            "health_insured_share",
        ]
    ]
    .describe()
    .T
    .round(2)
)

<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">7.2 Organize Analytical Variable Groups</h2></div>

In [ ]:
# Organize the merged variables into topic groups such as identity, demographics, and education.
# These groups help make later analysis easier to navigate.
GROUPS = {
    "Identification & location": (["household_id", "household_weight", "region", "area", "urban_rural", "person_id"], None),
    "Demographics": (["age", "gender", "marital_status", "relation_to_head"], None),
    "Education": (["literacy_status", "ever_attended_school", "education_level", "years_of_schooling", "school_attendance_status", "current_school_level", "current_school_type"], None),
    "Health & insurance": (["disability_status", "disability_type", "chronic_disease", "insurance_type", "has_social_security"], None),
    "Employment (person)": (["labor_force_status", "main_activity_status", "employment_status", "job_occupation", "job_industry", "job_sector"], None),
    "Father": ([], "father_"),
    "Mother": ([], "mother_"),
    "Household composition": (["num_males", "num_females", "household_size", "num_under_14", "num_aged_65_plus", "household_composition", "num_earners", "income_source"], None),
    "Household head": ([], "head_"),
    "Housing & utilities": (["dwelling_type", "dwelling_tenure", "num_rooms", "has_electricity", "lighting_energy_source", "cooking_energy_source", "water_facility", "toilet_facility", "toilet_shared", "garbage_disposal_method"], None),
    "Household assets": ([], "has_"),
    "Expenditure": ([], "exp_"),
    "Income": ([], "income_"),
    "Household assets": ([], "has_"),
"Expenditure": ([], "exp_"),
"Income": ([], "income_"),
"Feature engineering": ([
    "asset_group", "housing_deprivation", "size_group",
    "dependency_group", "earners_group", "children_group",
    "elderly_group", "health_group", "social_protection_group",
    "macro_region"
], None),
}

group_of = {
    c: g for g, (cols, prefix) in GROUPS.items()
    for c in df_merged.columns
    if c in cols or (prefix and c.startswith(prefix))
}

unassigned = [c for c in df_merged.columns if c not in group_of]
if unassigned:
    raise ValueError(f"Unassigned columns: {unassigned}")

column_groups = {
    g: [c for c in df_merged.columns if group_of[c] == g]
    for g in GROUPS
}

df_merged = df_merged[[c for cols in column_groups.values() for c in cols]]

for g, cols in column_groups.items():
    print(f"\n{g} ({len(cols)})")
    for c in cols:
        print("   ", c)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">7.3 Model Readiness Check</h2></div>

In [ ]:
# Check the shape of household and merged DataFrames before exporting them to a SQLite database.

print("df_merged shape:", df_merged.shape)
print("hh shape:", hh.shape)

print("\nPerson table checks:")
print("household_id column:", "household_id" in df_merged.columns)
print("person_id column:", "person_id" in df_merged.columns)
print("Unique person_id:", df_merged["person_id"].nunique())
print("Unique household-person pairs:",
      df_merged[["household_id", "person_id"]].drop_duplicates().shape[0])

print("\nHousehold table checks:")
print("Unique household IDs:", hh.index.nunique())
print("Household index name:", hh.index.name)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">7.4 Build the SQLite Data Model</h2></div>

In [ ]:
# Create and populate the SQLite data model using the prepared tables. Define supporting tables
# and metadata needed by the analysis.
# ============================================================
# HIECS 2021 — Household Socioeconomic Data Model + ERD
# Uses existing df_merged and hh; no cleaning or row deletion
# ============================================================


# -----------------------------
# 1. Validate existing data
# -----------------------------
assert "df_merged" in globals(), "df_merged is not defined. Run its preparation cells first."
assert "hh" in globals(), "hh is not defined. Run its preparation cells first."
assert df_merged.index.is_unique or "household_id" in df_merged.columns

persons_df = df_merged.copy()
households_df = hh.copy()

# household_id is the index in hh, so preserve it as a column.
if "household_id" not in households_df.columns:
    households_df = households_df.reset_index()

if "household_id" not in persons_df.columns:
    raise ValueError("df_merged must contain household_id.")

if "person_id" not in persons_df.columns:
    raise ValueError("df_merged must contain person_id.")

# Keep source DataFrames unchanged.
print("Source data:")
print("Persons:", persons_df.shape)
print("Households:", households_df.shape)
print("Unique households:", persons_df["household_id"].nunique())
print(
    "Unique household-person pairs:",
    persons_df[["household_id", "person_id"]].drop_duplicates().shape[0]
)

# -----------------------------
# 2. Define normalized tables
# -----------------------------
# Only use columns that exist in the source DataFrames.
def available(df, columns):
    return [c for c in columns if c in df.columns]


household_cols = available(households_df, [
    "household_id", "household_weight", "location_id",
    "region", "area", "urban_rural",
    "household_size", "num_males", "num_females",
    "num_under_14", "num_aged_65_plus", "num_earners",
    "head_age", "head_gender", "head_marital_status",
    "head_literacy_status", "head_education_level",
    "head_main_activity_status", "head_employment_status",
    "head_job_occupation", "head_job_industry", "head_job_sector",
    "dwelling_type", "dwelling_tenure", "num_rooms",
    "lighting_energy_source", "cooking_energy_source",
    "water_facility", "toilet_facility", "toilet_shared",
    "garbage_disposal_method", "has_social_security",
    "has_electricity"
])

economics_cols = available(households_df, [
    "household_id", "income_total_disposable", "income_wages",
    "income_self_employment", "income_rentals", "income_property",
    "income_transfers", "income_source", "exp_total",
    "exp_total_food", "exp_total_nonfood_nondurables",
    "exp_durables", "exp_housing_rent", "exp_taxes_levies"
])

feature_cols = available(households_df, [
    "household_id", "income_per_capita", "income_group",
    "asset_group", "housing_deprivation", "size_group",
    "dependency_group", "earners_group", "children_group",
    "elderly_group", "health_group", "social_protection_group",
    "macro_region", "any_disabled", "any_chronic", "any_covered"
])

# Household-level indicator table, only if at least one indicator exists.
indicator_cols = available(households_df, [
    "household_id", "any_disabled", "any_chronic", "any_covered",
    "has_social_security", "housing_deprivation"
])

person_cols = available(persons_df, [
    "household_id", "person_id", "age", "gender",
    "marital_status", "relation_to_head", "literacy_status",
    "ever_attended_school", "education_level", "years_of_schooling",
    "school_attendance_status", "current_school_level",
    "current_school_type", "disability_status", "disability_type",
    "chronic_disease", "insurance_type", "labor_force_status",
    "main_activity_status", "employment_status", "job_occupation",
    "job_industry", "job_sector"
])

parent_cols = available(persons_df, [
    "household_id", "person_id", "father_person_id", "mother_person_id"
])

# Location dimension only when location attributes exist.
location_cols = available(households_df, [
    "region", "area", "urban_rural"
])

# -----------------------------
# 3. Create a new SQLite database
# -----------------------------
output_dir = Path.cwd()
base_name = "egypt_household_2021_model"

# Keep existing databases and automatically choose the next available filename.
db_path = output_dir / f"{base_name}.db"
version = 2

while db_path.exists():
    db_path = output_dir / f"{base_name}_v{version}.db"
    version += 1

conn = sqlite3.connect(str(db_path))
conn.execute("PRAGMA foreign_keys = ON;")

def write_table(df, table_name, columns, index=False):
    """Write selected existing columns without modifying source DataFrames."""
    if not columns:
        print(f"Skipped {table_name}: no matching source columns.")
        return

    out = df[columns].copy()
    out.to_sql(table_name, conn, if_exists="fail", index=index)
    print(f"Created {table_name}: {out.shape[0]:,} rows, {out.shape[1]} columns")


try:
    # Core household table
    write_table(households_df, "HOUSEHOLDS", household_cols)

    # Person table; use a composite primary key in the SQL schema below.
    write_table(persons_df, "PERSONS", person_cols)

    # Economic facts and engineered features
    write_table(households_df, "HOUSEHOLD_ECONOMICS", economics_cols)
    write_table(households_df, "HOUSEHOLD_FEATURES", feature_cols)
    write_table(households_df, "ECONOMIC_INDICATORS", indicator_cols)

    # Parent links, when present
    write_table(persons_df, "PARENT_LINKS", parent_cols)

    # Location dimension: unique combinations, with generated surrogate key.
    if location_cols:
        locations = (
            households_df[location_cols]
            .drop_duplicates()
            .reset_index(drop=True)
        )
        locations.insert(0, "location_id", range(1, len(locations) + 1))
        locations.to_sql("DIM_LOCATION", conn, if_exists="fail", index=False)
        print(f"Created DIM_LOCATION: {len(locations):,} rows")

    # -----------------------------
    # 4. Add PKs / FKs and indexes
    # -----------------------------
    # SQLite cannot add PK/FK constraints to tables after to_sql.
    # Build relational views and indexes for the generated tables.
    conn.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS ux_households_id
        ON HOUSEHOLDS(household_id)
    """)

    conn.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS ux_person_household_person
        ON PERSONS(household_id, person_id)
    """)

    for table in [
        "HOUSEHOLD_ECONOMICS",
        "HOUSEHOLD_FEATURES",
        "ECONOMIC_INDICATORS"
    ]:
        exists = conn.execute(
            "SELECT 1 FROM sqlite_master WHERE type='table' AND name=?",
            (table,)
        ).fetchone()

        if exists:
            conn.execute(
                f'CREATE UNIQUE INDEX IF NOT EXISTS "ux_{table.lower()}_household" '
                f'ON "{table}"(household_id)'
            )

    conn.execute("""
        CREATE INDEX IF NOT EXISTS ix_persons_household
        ON PERSONS(household_id)
    """)

    conn.commit()

except Exception:
    conn.rollback()
    conn.close()

    if db_path.exists():
        db_path.unlink()

    raise

# -----------------------------
# 5. Validation
# -----------------------------
print("\nDatabase created:", db_path)
print("Database size:", f"{db_path.stat().st_size:,} bytes")

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name
""", conn)

display(tables)

print("\nRow counts:")
for table in tables["name"]:
    count = conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
    print(f"{table}: {count:,}")

conn.close()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">7.5 Validate the SQLite Model</h2></div>

In [ ]:
# Open the SQLite model and run validation queries. Check that stored tables and records match the
# expected data.
# ============================================================
# STEP 2 — Validate keys and prepare relationship mapping
# Read-only validation; no database changes
# ============================================================


db_path = "egypt_household_2021_model.db"

with sqlite3.connect(db_path) as conn:
    checks = {
        "HOUSEHOLDS household_id unique":
            "SELECT COUNT(*) - COUNT(DISTINCT household_id) FROM HOUSEHOLDS",

        "PERSONS household-person pair duplicates":
            """
            SELECT COUNT(*) FROM (
                SELECT household_id, person_id
                FROM PERSONS
                GROUP BY household_id, person_id
                HAVING COUNT(*) > 1
            )
            """,

        "PERSONS without matching household":
            """
            SELECT COUNT(*)
            FROM PERSONS p
            LEFT JOIN HOUSEHOLDS h
              ON p.household_id = h.household_id
            WHERE h.household_id IS NULL
            """,

        "ECONOMICS without matching household":
            """
            SELECT COUNT(*)
            FROM HOUSEHOLD_ECONOMICS e
            LEFT JOIN HOUSEHOLDS h
              ON e.household_id = h.household_id
            WHERE h.household_id IS NULL
            """,

        "FEATURES without matching household":
            """
            SELECT COUNT(*)
            FROM HOUSEHOLD_FEATURES f
            LEFT JOIN HOUSEHOLDS h
              ON f.household_id = h.household_id
            WHERE h.household_id IS NULL
            """
    }

    print("RELATIONAL KEY VALIDATION")
    print("=" * 45)

    for label, query in checks.items():
        result = conn.execute(query).fetchone()[0]
        print(f"{label}: {result:,}")

    print("\nLocation columns in HOUSEHOLDS:")
    print(pd.read_sql_query(
        "SELECT household_id, region, area, urban_rural "
        "FROM HOUSEHOLDS LIMIT 5",
        conn
    ).to_string(index=False))

    print("\nLocation dimension sample:")
    print(pd.read_sql_query(
        "SELECT * FROM DIM_LOCATION LIMIT 5",
        conn
    ).to_string(index=False))


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">7.6 Finalize the Portable Database</h2></div>

In [ ]:
# Prepare a final SQLite database file from the model and verify the paths used for export.

source_db = Path.cwd() / "egypt_household_2021_model.db"
final_db = Path.cwd() / "egypt_household_2021_final.db"

# Remove only the failed output database, never the source.
if final_db.exists():
    final_db.unlink()

src = sqlite3.connect(f"file:{source_db.resolve()}?mode=ro", uri=True)
dst = sqlite3.connect(final_db)
dst.execute("PRAGMA foreign_keys = ON")

try:
    src.backup(dst)

    # Rebuild DIM_LOCATION with a real primary key.
    loc_rows = dst.execute("""
        SELECT location_id, region, area, urban_rural
        FROM DIM_LOCATION
        ORDER BY location_id
    """).fetchall()

    dst.execute("DROP TABLE DIM_LOCATION")

    dst.execute("""
        CREATE TABLE DIM_LOCATION (
            location_id INTEGER PRIMARY KEY,
            region TEXT,
            area TEXT,
            urban_rural TEXT,
            UNIQUE(region, area, urban_rural)
        )
    """)

    dst.executemany("""
        INSERT INTO DIM_LOCATION
        (location_id, region, area, urban_rural)
        VALUES (?, ?, ?, ?)
    """, loc_rows)

    dst.commit()

    print("DIM_LOCATION rebuilt successfully.")
    print("Rows:", dst.execute(
        "SELECT COUNT(*) FROM DIM_LOCATION"
    ).fetchone()[0])
    print("Primary key:", dst.execute(
        "PRAGMA table_info(DIM_LOCATION)"
    ).fetchall())

except Exception:
    dst.rollback()
    dst.close()
    src.close()
    raise

dst.close()
src.close()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">7.7 Validate the Final Database</h2></div>

In [ ]:
# Read the final database and examine its tables, structure, or stored records for consistency.

final_db = Path.cwd() / "egypt_household_2021_final.db"

conn = sqlite3.connect(final_db)
conn.execute("PRAGMA foreign_keys = ON")

try:
    conn.execute("DROP TABLE IF EXISTS HOUSEHOLDS_NEW")

    conn.execute("""
        CREATE TABLE HOUSEHOLDS_NEW (
            household_id INTEGER PRIMARY KEY,
            household_weight REAL,
            location_id INTEGER NOT NULL,
            household_size INTEGER,
            num_males INTEGER,
            num_females INTEGER,
            num_under_14 INTEGER,
            num_aged_65_plus INTEGER,
            num_earners INTEGER,
            head_age INTEGER,
            head_gender TEXT,
            head_education_level TEXT,
            head_employment_status TEXT,
            dwelling_type TEXT,
            dwelling_tenure TEXT,
            num_rooms REAL,
            has_social_security TEXT,
            has_electricity TEXT,
            FOREIGN KEY (location_id)
                REFERENCES DIM_LOCATION(location_id)
        )
    """)

    conn.execute("""
        INSERT INTO HOUSEHOLDS_NEW (
            household_id, household_weight, location_id,
            household_size, num_males, num_females,
            num_under_14, num_aged_65_plus, num_earners,
            head_age, head_gender, head_education_level,
            head_employment_status, dwelling_type, dwelling_tenure,
            num_rooms, has_social_security, has_electricity
        )
        SELECT
            h.household_id, h.household_weight, d.location_id,
            h.household_size, h.num_males, h.num_females,
            h.num_under_14, h.num_aged_65_plus, h.num_earners,
            h.head_age, h.head_gender, h.head_education_level,
            h.head_employment_status, h.dwelling_type, h.dwelling_tenure,
            h.num_rooms, h.has_social_security, h.has_electricity
        FROM HOUSEHOLDS h
        JOIN DIM_LOCATION d
          ON h.region = d.region
         AND h.area = d.area
         AND h.urban_rural = d.urban_rural
    """)

    original_count = conn.execute(
        "SELECT COUNT(*) FROM HOUSEHOLDS"
    ).fetchone()[0]

    mapped_count = conn.execute(
        "SELECT COUNT(*) FROM HOUSEHOLDS_NEW"
    ).fetchone()[0]

    print("Original households:", original_count)
    print("Mapped households:", mapped_count)
    print("Unmapped households:", original_count - mapped_count)

    if original_count != mapped_count:
        raise ValueError("Some households could not be mapped to a location.")

    conn.commit()
    print("\nHousehold-location relationship created successfully.")

except Exception:
    conn.rollback()
    raise

finally:
    conn.close()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">7.8 Inspect the Final Model Structure</h2></div>

In [ ]:
# Run additional SQL checks against the final database to confirm its contents and relationships.

db_path = Path.cwd() / "egypt_household_2021_final.db"
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON")

try:
    # Replace HOUSEHOLDS with the mapped version.
    conn.execute("DROP TABLE HOUSEHOLDS")
    conn.execute("""
        ALTER TABLE HOUSEHOLDS_NEW RENAME TO HOUSEHOLDS
    """)

    # Create PERSONS with a composite primary key and household FK.
    person_cols = [
        row[1] for row in conn.execute("PRAGMA table_info(PERSONS_OLD)")
    ] if conn.execute("""
        SELECT 1 FROM sqlite_master
        WHERE type='table' AND name='PERSONS_OLD'
    """).fetchone() else [
        row[1] for row in conn.execute("PRAGMA table_info(PERSONS)")
    ]

    if conn.execute("""
        SELECT 1 FROM sqlite_master
        WHERE type='table' AND name='PERSONS'
    """).fetchone():
        conn.execute("ALTER TABLE PERSONS RENAME TO PERSONS_STAGE")

        person_cols = [
            row[1] for row in conn.execute("PRAGMA table_info(PERSONS_STAGE)")
        ]

        col_defs = []
        for col in person_cols:
            if col in ("household_id", "person_id"):
                col_defs.append(f'"{col}" INTEGER NOT NULL')
            else:
                col_defs.append(f'"{col}"')

        conn.execute(f"""
            CREATE TABLE PERSONS (
                {", ".join(col_defs)},
                PRIMARY KEY (household_id, person_id),
                FOREIGN KEY (household_id)
                    REFERENCES HOUSEHOLDS(household_id)
            )
        """)

        cols_sql = ", ".join(f'"{c}"' for c in person_cols)
        conn.execute(f"""
            INSERT INTO PERSONS ({cols_sql})
            SELECT {cols_sql} FROM PERSONS_STAGE
        """)
        conn.execute("DROP TABLE PERSONS_STAGE")

    # Rebuild household child tables with actual FK constraints.
    for table in [
        "HOUSEHOLD_ECONOMICS",
        "HOUSEHOLD_FEATURES",
        "ECONOMIC_INDICATORS"
    ]:
        exists = conn.execute(
            "SELECT 1 FROM sqlite_master WHERE type='table' AND name=?",
            (table,)
        ).fetchone()

        if not exists:
            continue

        stage = f"{table}_STAGE"
        conn.execute(f'ALTER TABLE "{table}" RENAME TO "{stage}"')

        cols = [
            row[1] for row in conn.execute(f'PRAGMA table_info("{stage}")')
        ]
        definitions = []
        for col in cols:
            if col == "household_id":
                definitions.append('"household_id" INTEGER PRIMARY KEY')
            else:
                definitions.append(f'"{col}"')

        conn.execute(f"""
            CREATE TABLE "{table}" (
                {", ".join(definitions)},
                FOREIGN KEY (household_id)
                    REFERENCES HOUSEHOLDS(household_id)
            )
        """)

        cols_sql = ", ".join(f'"{c}"' for c in cols)
        conn.execute(f"""
            INSERT INTO "{table}" ({cols_sql})
            SELECT {cols_sql} FROM "{stage}"
        """)
        conn.execute(f'DROP TABLE "{stage}"')

    # Rebuild parent links referencing PERSONS.
    if conn.execute("""
        SELECT 1 FROM sqlite_master
        WHERE type='table' AND name='PARENT_LINKS'
    """).fetchone():
        conn.execute("ALTER TABLE PARENT_LINKS RENAME TO PARENT_LINKS_STAGE")
        conn.execute("""
            CREATE TABLE PARENT_LINKS (
                household_id INTEGER NOT NULL,
                person_id INTEGER NOT NULL,
                father_person_id INTEGER,
                mother_person_id INTEGER,
                PRIMARY KEY (household_id, person_id),
                FOREIGN KEY (household_id, person_id)
                    REFERENCES PERSONS(household_id, person_id)
            )
        """)
        conn.execute("""
            INSERT INTO PARENT_LINKS
            SELECT household_id, person_id, father_person_id, mother_person_id
            FROM PARENT_LINKS_STAGE
        """)
        conn.execute("DROP TABLE PARENT_LINKS_STAGE")

    # Add an index for common household-to-person joins.
    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_persons_household
        ON PERSONS(household_id)
    """)

    conn.commit()

    print("Final schema built.")
    print("Foreign key violations:",
          conn.execute("PRAGMA foreign_key_check").fetchall())

    print("\nTables and row counts:")
    tables = conn.execute("""
        SELECT name FROM sqlite_master
        WHERE type='table' AND name NOT LIKE 'sqlite_%'
        ORDER BY name
    """).fetchall()

    for (table,) in tables:
        n = conn.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
        print(f"{table}: {n:,}")

except Exception:
    conn.rollback()
    raise

finally:
    conn.close()


In [ ]:
# Visualize the final SQLite data model and table relationships.
# The diagram reads the schema directly from the exported database.

# Use a clear, readable font
plt.rcParams["font.family"] = "DejaVu Sans"

# Read the existing database
db_path = Path.cwd() / "egypt_household_2021_final.db"
conn = sqlite3.connect(f"file:{db_path.resolve()}?mode=ro", uri=True)

tables = [
    r[0] for r in conn.execute("""
        SELECT name FROM sqlite_master
        WHERE type='table' AND name NOT LIKE 'sqlite_%'
        ORDER BY name
    """)
]

schema = {}
relationships = []

for table in tables:
    cols = conn.execute(f'PRAGMA table_info("{table}")').fetchall()
    fks = conn.execute(f'PRAGMA foreign_key_list("{table}")').fetchall()

    schema[table] = [
        (
            c[1],
            c[2] or "TEXT",
            "PK" if c[5] else
            "FK" if c[1] in {fk[3] for fk in fks} else ""
        )
        for c in cols
    ]

    for fk in fks:
        relationships.append((fk[2], table, fk[4], fk[3]))

conn.close()

# Luxury Gold & Bronze Color Palette
colors = {
    "DIM_LOCATION": "#F5EFE0",
    "HOUSEHOLDS": "#E8C35A",
    "PERSONS": "#D4AF37",
    "HOUSEHOLD_ECONOMICS": "#B8902A",
    "HOUSEHOLD_FEATURES": "#A07F22",
    "ECONOMIC_INDICATORS": "#C9A24A",
    "PARENT_LINKS": "#8A6A1C",
}

# Card dimensions
box_w = 0.23
header_h = 0.045
row_step = 0.018
top_pad = 0.016
bottom_pad = 0.016


def card_height(table):
    cols = schema[table]
    shown_count = min(10, len(cols))
    has_more = len(cols) > shown_count

    return (
        header_h
        + top_pad
        + shown_count * row_step
        + bottom_pad
        + (0.018 if has_more else 0)
    )


box_heights = {
    table: card_height(table)
    for table in schema
}

# Main table positions
positions = {
    "DIM_LOCATION": (0.035, 0.82),
    "HOUSEHOLDS": (0.36, 0.82),
    "PERSONS": (0.36, 0.27),
}

# Place the right-side tables with vertical gaps
right_tables = [
    "HOUSEHOLD_ECONOMICS",
    "HOUSEHOLD_FEATURES",
    "ECONOMIC_INDICATORS",
    "PARENT_LINKS",
]

right_x = 0.76
right_top = 1.20
table_gap = 0.075

for table in right_tables:
    if table not in schema:
        continue

    h = box_heights[table]
    center_y = right_top - h / 2
    positions[table] = (right_x, center_y)
    right_top = center_y - h / 2 - table_gap

# Figure and background
fig, ax = plt.subplots(figsize=(16, 10.5))
fig.patch.set_facecolor("#0b0905")
ax.set_facecolor("#0b0905")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.35)
ax.axis("off")

# Draw relationship lines behind the tables
for parent, child, parent_col, child_col in relationships:
    if parent not in positions or child not in positions:
        continue

    x1, y1 = positions[parent]
    x2, y2 = positions[child]
    h1 = box_heights[parent]
    h2 = box_heights[child]

    dx = x2 - x1
    dy = y2 - y1

    if abs(dx) >= abs(dy):
        if dx >= 0:
            start = (x1 + box_w, y1)
            end = (x2, y2)
        else:
            start = (x1, y1)
            end = (x2 + box_w, y2)
    else:
        if dy < 0:
            start = (x1 + box_w / 2, y1 - h1 / 2)
            end = (x2 + box_w / 2, y2 + h2 / 2)
        else:
            start = (x1 + box_w / 2, y1 + h1 / 2)
            end = (x2 + box_w / 2, y2 - h2 / 2)

    ax.add_patch(FancyArrowPatch(
        start,
        end,
        arrowstyle="-",
        connectionstyle="arc3,rad=0.0",
        linewidth=1.25,
        color="#8A826A",
        alpha=0.6,
        zorder=1
    ))

# Draw the table cards
for table, (x, y) in positions.items():
    if table not in schema:
        continue

    cols = schema[table]
    shown = cols[:10]
    has_more = len(cols) > len(shown)
    actual_h = box_heights[table]
    y0 = y - actual_h / 2
    accent = colors.get(table, "#D4AF37")

    card = FancyBboxPatch(
        (x, y0),
        box_w,
        actual_h,
        boxstyle="round,pad=0.006,rounding_size=0.014",
        linewidth=1.3,
        edgecolor=accent,
        facecolor="#13100A",
        zorder=2
    )

    card.set_path_effects([
        pe.Stroke(linewidth=10, foreground=accent, alpha=0.12),
        pe.Stroke(linewidth=4, foreground=accent, alpha=0.35),
        pe.Normal()
    ])

    ax.add_patch(card)

    # Colored table header
    header = FancyBboxPatch(
        (x, y0 + actual_h - header_h),
        box_w,
        header_h,
        boxstyle="round,pad=0.006,rounding_size=0.014",
        linewidth=0,
        facecolor=accent,
        zorder=3
    )

    header.set_path_effects([
        pe.Stroke(linewidth=5, foreground=accent, alpha=0.20),
        pe.Normal()
    ])

    ax.add_patch(header)

    ax.text(
        x + 0.014,
        y0 + actual_h - header_h / 2,
        table,
        color="#13100A",
        fontsize=9.0,
        fontweight="bold",
        va="center",
        zorder=4
    )

    # Field names and PK/FK badges
    for i, (name, dtype, key) in enumerate(shown):
        yy = y0 + actual_h - header_h - top_pad - (i + 0.5) * row_step

        ax.text(
            x + 0.015,
            yy,
            name,
            color="#F5EFE0",
            fontsize=8.0,
            va="center",
            zorder=4
        )

        if key:
            key_color = "#FFF3B0" if key == "PK" else "#E8C35A"

            ax.text(
                x + box_w - 0.014,
                yy,
                key,
                color=key_color,
                fontsize=7.0,
                fontweight="bold",
                ha="right",
                va="center",
                bbox=dict(
                    boxstyle="round,pad=0.18",
                    facecolor="#17130A",
                    edgecolor="none"
                ),
                zorder=4
            )

    if has_more:
        ax.text(
            x + 0.015,
            y0 + 0.011,
            f"+ {len(cols) - len(shown)} more columns",
            color="#A8A086",
            fontsize=7.0,
            style="italic",
            va="bottom",
            zorder=4
        )

# Title and subtitle
ax.text(
    0.5,
    1.31,
    "HIECS 2021",
    ha="center",
    va="center",
    fontsize=18,
    fontweight="bold",
    color="#F5EFE0"
)

ax.text(
    0.5,
    1.275,
    "Household Socioeconomic Data Model",
    ha="center",
    va="center",
    fontsize=9.5,
    color="#C7AE82"
)

# Legend
ax.text(
    0.5,
    0.04,
    "PK   Primary Key     •     FK   Foreign Key     •     Showing up to 10 fields per table",
    ha="center",
    fontsize=8.5,
    color="#C7AE82"
)

plt.subplots_adjust(left=0.02, right=0.98, top=0.92, bottom=0.065)
plt.show()


<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 8</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">8. Statistics</h1><div style="height:1px;background-color:#8a6a1c;margin:14px 0 10px;"></div><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:13.5px;line-height:1.55;color:#a8a086;font-style:italic;">Organizes household-level analytical variables, then presents descriptive and inferential statistical analysis in a logical order.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">8.1 Descriptive Statistics</h2></div>

In [ ]:
# ============================================================
# Main Descriptive Statistics Table
# ============================================================

# Work on a copy so the original household table is not changed
desc_data = hh.copy()

# Required variables
desc_columns = [
    "income_per_capita",
    "income_total_disposable",
    "household_size",
    "num_earners",
    "num_under_14",
    "num_aged_65_plus",
    "exp_total",
    "exp_total_food",
]

# Check that required columns exist
missing_columns = [
    col for col in desc_columns
    if col not in desc_data.columns
]

if missing_columns:
    raise KeyError(
        f"Missing columns for descriptive statistics: {missing_columns}"
    )

# Convert numeric variables safely
for col in desc_columns:
    desc_data[col] = pd.to_numeric(
        desc_data[col],
        errors="coerce"
    )

# Convert number of rooms to numeric
# The cleaned data may contain values such as "One room", "Two rooms", etc.
room_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
    "Six": 6,
    "Seven": 7,
    "Eight": 8,
    "Nine": 9,
    "Ten": 10,
}

rooms_numeric = pd.to_numeric(
    desc_data["num_rooms"],
    errors="coerce"
)

rooms_from_text = (
    desc_data["num_rooms"]
    .astype("string")
    .str.split()
    .str[0]
    .map(room_map)
)

desc_data["num_rooms"] = rooms_numeric.fillna(
    rooms_from_text
)

# Add number of rooms to the descriptive variables
desc_columns_with_rooms = desc_columns + ["num_rooms"]

# Create descriptive statistics table
descriptive_stats = (
    desc_data[desc_columns_with_rooms]
    .describe()
    .T
    .rename(
        columns={
            "count": "Count",
            "mean": "Mean",
            "std": "Standard deviation",
            "min": "Minimum",
            "25%": "Q1",
            "50%": "Median",
            "75%": "Q3",
            "max": "Maximum",
        }
    )
)

# Arrange columns in the requested order
descriptive_stats = descriptive_stats[
    [
        "Count",
        "Mean",
        "Median",
        "Standard deviation",
        "Minimum",
        "Maximum",
        "Q1",
        "Q3",
    ]
].round(2)

display(descriptive_stats)


In [ ]:
# ============================================================
# Weighted Descriptive Statistics
# ============================================================

weighted_data = hh.copy()

# Check required columns
if "household_weight" not in weighted_data.columns:
    raise KeyError(
        "Column 'household_weight' was not found in hh."
    )

if "income_per_capita" not in weighted_data.columns:
    raise KeyError(
        "Column 'income_per_capita' was not found in hh. "
        "Run the income per capita cell first."
    )

# Convert weights to numeric
weighted_data["household_weight"] = pd.to_numeric(
    weighted_data["household_weight"],
    errors="coerce"
)

# Numeric variables
weighted_numeric_columns = [
    "income_per_capita",
    "income_total_disposable",
    "household_size",
    "num_earners",
    "num_under_14",
    "num_aged_65_plus",
    "exp_total",
    "exp_total_food",
]

# Convert numeric variables safely
for col in weighted_numeric_columns:
    weighted_data[col] = pd.to_numeric(
        weighted_data[col],
        errors="coerce"
    )

# Convert rooms to numeric
room_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
    "Six": 6,
    "Seven": 7,
    "Eight": 8,
    "Nine": 9,
    "Ten": 10,
}

rooms_numeric = pd.to_numeric(
    weighted_data["num_rooms"],
    errors="coerce"
)

rooms_from_text = (
    weighted_data["num_rooms"]
    .astype("string")
    .str.split()
    .str[0]
    .map(room_map)
)

weighted_data["num_rooms_numeric"] = rooms_numeric.fillna(
    rooms_from_text
)

weighted_numeric_columns.append(
    "num_rooms_numeric"
)


# Weighted mean and weighted median table
# ------------------------------------------------------------

weighted_rows = []

for col in weighted_numeric_columns:
    weighted_rows.append(
        {
            "Variable": col,
            "Valid observations": weighted_data[col].notna().sum(),
            "Weighted mean": weighted_mean(
                weighted_data[col],
                weighted_data["household_weight"]
            ),
            "Weighted median": weighted_median(
                weighted_data[col],
                weighted_data["household_weight"]
            ),
        }
    )

weighted_descriptive_stats = (
    pd.DataFrame(weighted_rows)
    .set_index("Variable")
    .round(2)
)

display(weighted_descriptive_stats)


# Weighted percentages for categorical variables
# ------------------------------------------------------------

categorical_columns = [
    "income_group",
    "urban_rural",
    "region",
    "head_education_level",
    "housing_deprivation",
    "asset_group",
    "size_group",
    "earners_group",
]

for col in categorical_columns:
    if col in weighted_data.columns:
        print(f"\nWeighted percentages: {col}")
        display(
            weighted_percentage_table(
                weighted_data,
                col
            )
        )


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">8.1.1 Correlation Matrix Heatmap</h3></div>


In [ ]:
# ============================================================
# 8.1.1 Correlation Matrix of Analytical Household Features
# ============================================================

THEME = {
    "panel": "#0b0905",
    "line": "#2b2410",
    "gold": "#d4af37",
    "gold_light": "#f3d77a",
    "text": "#f5efe0",
    "dim": "#a8a086",
}

analysis_columns = [
    "income_per_capita",
    "expenditure_per_capita",
    "saving_rate",
    "household_size",
    "num_earners",
    "child_ratio",
    "elderly_ratio",
    "dependency_ratio",
    "avg_adult_schooling",
    "adult_literacy_rate",
    "adult_employment_rate",
    "health_insured_share",
]

available_columns = [
    column
    for column in analysis_columns
    if column in hh.columns
]

missing_columns = [
    column
    for column in analysis_columns
    if column not in hh.columns
]

if missing_columns:
    print("Unavailable columns were skipped:")
    print(missing_columns)

if len(available_columns) < 2:
    raise ValueError(
        "Not enough analytical variables are available."
    )

correlation_data = hh[available_columns].copy()

for column in correlation_data.columns:
    correlation_data[column] = pd.to_numeric(
        correlation_data[column],
        errors="coerce"
    )

correlation_matrix = correlation_data.corr(
    method="spearman",
    min_periods=30
)

display_names = {
    "income_per_capita": "Income per capita",
    "expenditure_per_capita": "Expenditure per capita",
    "saving_rate": "Saving rate",
    "household_size": "Household size",
    "num_earners": "Number of earners",
    "child_ratio": "Child ratio",
    "elderly_ratio": "Elderly ratio",
    "dependency_ratio": "Dependency ratio",
    "avg_adult_schooling": "Adult schooling",
    "adult_literacy_rate": "Adult literacy",
    "adult_employment_rate": "Adult employment",
    "health_insured_share": "Health-insured share",
}

correlation_matrix = correlation_matrix.rename(
    index=display_names,
    columns=display_names
)

gold_correlation_cmap = LinearSegmentedColormap.from_list(
    "egypt_gold_correlation",
    [
        "#4A3A10",
        "#8A6A1C",
        "#F5EFE0",
        "#D4AF37",
        "#F3D77A",
    ]
)

plt.close("all")

fig, ax = plt.subplots(
    figsize=(13, 11),
    facecolor=THEME["panel"]
)

ax.set_facecolor(THEME["panel"])

heatmap = sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap=gold_correlation_cmap,
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.8,
    linecolor=THEME["panel"],
    square=True,
   annot_kws={
    "color": "black",
    "size": 9,
    "fontweight": "bold",
},
    cbar_kws={
        "label": "Spearman correlation",
        "shrink": 0.85,
        "pad": 0.03,
    },
    ax=ax,
)

ax.set_title(
    "Correlation Matrix of Analytical Household Features",
    fontsize=18,
    fontweight="bold",
    color=THEME["gold_light"],
    pad=20,
)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
    fontsize=10,
    color=THEME["dim"],
)

ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0,
    fontsize=10,
    color=THEME["dim"],
)

ax.tick_params(
    colors=THEME["dim"],
    length=0
)

colorbar = heatmap.collections[0].colorbar

colorbar.set_label(
    "Spearman correlation",
    color=THEME["gold_light"],
    fontsize=11,
    labelpad=12
)

colorbar.ax.tick_params(
    colors=THEME["dim"],
    length=0
)

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">8.1.2 Household Disadvantage Heatmap</h3></div>

In [ ]:
W = "household_weight"

# ==============================================================================
# 1. Build disadvantage flags
# ==============================================================================

hm_flags = pd.DataFrame(index=hh.index)

hm_flags["Bottom 20% income"] = (
    hh["income_group"].eq("Bottom 20%")
)

hm_flags["Bottom 20% assets"] = (
    hh["asset_group"].eq("Bottom 20%")
)

hm_flags["Severe housing deprivation"] = (
    hh["housing_deprivation"].eq("Severely deprived")
)

hm_flags["No social protection"] = (
    hh["social_protection_group"].eq("Not covered")
)

hm_flags["Disabled / chronic member"] = (
    hh["health_group"].eq("Has disabled/chronic member")
)

hm_flags = hm_flags.astype(float)


# ==============================================================================
# 2. Indicator that defines a group -> mask its own cells
# ==============================================================================

SELF = {
    "income_group": "Bottom 20% income",
    "asset_group": "Bottom 20% assets",
    "housing_deprivation": "Severe housing deprivation",
    "social_protection_group": "No social protection",
    "health_group": "Disabled / chronic member",
}


# ==============================================================================
# 3. Helper functions
# ==============================================================================

def wpct(sel, col):
    """
    Weighted percentage of households with flag `col`
    inside the selected rows.
    """

    w = hh.loc[sel, W]

    if w.sum() == 0:
        return np.nan

    return (
        np.average(
            hm_flags.loc[sel, col],
            weights=w
        )
        * 100
    )


def pretty(c):
    return (
        c.replace("_group", "")
        .replace("_", " ")
        .title()
    )


# ==============================================================================
# 4. Build heatmap table
# ==============================================================================

hm_dims = [
    c for c in group_cols
    if c in hh.columns
]

hm_inds = list(hm_flags.columns)

hm_rows = []
hm_labels = []

for dim in hm_dims:

    # Add section header row
    hm_rows.append(
        [np.nan] * len(hm_inds)
    )

    hm_labels.append(
        pretty(dim).upper()
    )

    # Get categories
    if isinstance(
        hh[dim].dtype,
        pd.CategoricalDtype
    ):
        categories = hh[dim].cat.categories

    else:
        categories = hh[dim].dropna().unique()

    # Add each category
    for cat in categories:

        m = hh[dim].eq(cat)

        if not m.any():
            continue

        share = (
            hh.loc[m, W].sum()
            / hh[W].sum()
            * 100
        )

        vals = [
            np.nan
            if SELF.get(dim) == indicator
            else wpct(m, indicator)

            for indicator in hm_inds
        ]

        hm_rows.append(vals)

        hm_labels.append(
            f"{cat}  ({share:.0f}%)"
        )


heat1 = pd.DataFrame(
    hm_rows,
    index=hm_labels,
    columns=hm_inds
)


# ==============================================================================
# 5. Luxury Gold Color Scale
# ==============================================================================

luxury_gold_colorscale = [
    [0.00, "#14110B"],
    [0.25, "#332A15"],
    [0.50, "#8C6718"],
    [0.75, "#D4AF37"],
    [1.00, "#F7E7CE"]
]


# ==============================================================================
# 6. Create numeric Y positions
# ==============================================================================

y_positions = list(
    range(len(heat1))
)


# ==============================================================================
# 7. Create Heatmap
# ==============================================================================

fig = go.Figure()

fig.add_trace(
    go.Heatmap(

        z=heat1.values,

        x=list(
            heat1.columns
        ),

        # Numeric Y axis prevents duplicate-label problems
        y=y_positions,

        colorscale=luxury_gold_colorscale,

        xgap=2,
        ygap=2,

        hoverongaps=False,

        customdata=np.array([
            [row_label] * len(heat1.columns)
            for row_label in heat1.index
        ]),

        hovertemplate=(
            "<b>Group:</b> %{customdata}<br>"
            "<b>Indicator:</b> %{x}<br>"
            "<b>Disadvantaged Rate:</b> %{z:.1f}%"
            "<extra></extra>"
        ),

        colorbar=dict(

            title=dict(
                text="%",
                font=dict(
                    color="#F4E0B9",
                    size=12
                )
            ),

            tickfont=dict(
                color="#F4E0B9",
                size=10
            ),

            thickness=20,

            len=0.88,

            x=1.03,

            xpad=15,

            outlinewidth=0
        )
    )
)


# ==============================================================================
# 8. Add percentage labels manually
# ==============================================================================

for row_idx in range(
    len(heat1)
):

    for col_idx in range(
        len(heat1.columns)
    ):

        # Always returns exactly one scalar value
        value = heat1.iloc[
            row_idx,
            col_idx
        ]

        # Only display real values
        if pd.notna(value):

            # Dark text for very bright cells
            if value >= 75:
                number_color = "#151008"

            else:
                number_color = "#FFF4D6"

            fig.add_annotation(

                x=heat1.columns[
                    col_idx
                ],

                y=row_idx,

                text=(
                    f"<b>{value:.0f}%</b>"
                ),

                showarrow=False,

                font=dict(
                    family="Arial",
                    size=11,
                    color=number_color
                ),

                xanchor="center",

                yanchor="middle"
            )


# ==============================================================================
# 9. Layout
# ==============================================================================

fig.update_layout(

    title=dict(

        text=(
            "<b>◆ Who is disadvantaged? "
            "Household groups vs. five disadvantage indicators</b>"
            "<br>"
            "<sup>"
            "(share of households in the group in brackets; "
            "blank/dark = same variable or N/A)"
            "</sup>"
        ),

        font=dict(
            size=18,
            color="#F4E0B9",
            family="Arial"
        ),

        x=0.01,

        xanchor="left",

        y=0.99,

        yanchor="top"
    ),

    xaxis=dict(

        title="",

        tickfont=dict(
            color="#F4E0B9",
            size=11
        ),

        tickangle=-25,

        side="top",

        showgrid=False,

        zeroline=False,

        automargin=True
    ),

    yaxis=dict(

        title="",

        # Numeric positions mapped back to readable labels
        tickmode="array",

        tickvals=y_positions,

        ticktext=list(
            heat1.index
        ),

        tickfont=dict(
            color="#F4E0B9",
            size=10
        ),

        autorange="reversed",

        showgrid=False,

        zeroline=False,

        automargin=True
    ),

    paper_bgcolor="#0c0a00",

    plot_bgcolor="#0c0a00",

    width=1500,

    height=max(
        1000,
        len(heat1) * 35 + 220
    ),

    margin=dict(
        l=330,
        r=220,
        t=210,
        b=70
    ),

    hoverlabel=dict(

        bgcolor="#14110B",

        bordercolor="#D4AF37",

        font=dict(
            color="#F4E0B9",
            size=12,
            family="Arial"
        )
    )
)


# ==============================================================================
# 10. Display
# ==============================================================================

fig.show()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">8.1.3 Household Segmentation and Descriptive Summaries</h3></div>

In [ ]:
# List the numeric household features to use when creating household segments. These describe
# household size, age groups, and resources.


SEG_NUM = [
    "household_size", "num_under_14", "num_aged_65_plus",
    "num_earners"
]

SEG_CAT = [
    "head_education_level", "head_employment_status",
    "income_source", "head_gender", "urban_rural", "region",
    "asset_group", "housing_deprivation", "social_protection_group",
    "health_group"
]

cols = SEG_NUM + SEG_CAT
seg = hh[cols].copy()

seg[SEG_CAT] = seg[SEG_CAT].astype("category")

print(f"Households: {len(seg):,}")
print(f"Segmentation variables: {len(cols)}")
print(f"Numeric: {len(SEG_NUM)} | Categorical: {len(SEG_CAT)}")

display(pd.DataFrame({
    "Type": ["Numeric" if c in SEG_NUM else "Categorical" for c in cols],
    "Missing": seg[cols].isna().sum(),
    "Missing %": (seg[cols].isna().mean() * 100).round(2),
    "Unique values": seg[cols].nunique()
}, index=cols).rename_axis("Variable"))


In [ ]:
# Prepare household-level segmentation data and apply the defined segment rules. Include checks
# that required merged or household tables exist.

# These analysis cells can be run in any order after the source-data preparation
# has created df_merged. Rebuild household-level inputs when they are absent.
if "hh" not in globals():
    if "df_merged" not in globals():
        raise RuntimeError("Run the data loading/cleaning/merge cells first; df_merged is not available.")
    hh = df_merged.drop_duplicates("household_id").set_index("household_id").copy()

_required_seg_cols = [
    "household_size", "num_under_14", "num_aged_65_plus", "num_earners",
    "head_education_level", "head_employment_status", "income_source",
    "head_gender", "urban_rural", "region", "asset_group",
    "housing_deprivation", "social_protection_group", "health_group"
]
_missing_seg_cols = [c for c in _required_seg_cols if c not in hh.columns]
if _missing_seg_cols:
    raise KeyError(f"Cannot build household segmentation; missing columns in hh: {_missing_seg_cols}. Run the feature-engineering cell first.")

# Always rebuild the small segmentation table from its source instead of relying
# on whether an earlier cell happened to have been executed.
seg = hh[_required_seg_cols].copy()
seg[ [c for c in _required_seg_cols if c not in ["household_size", "num_under_14", "num_aged_65_plus", "num_earners"]] ] = seg[
    [c for c in _required_seg_cols if c not in ["household_size", "num_under_14", "num_aged_65_plus", "num_earners"]]
].astype("category")
seg["dependency_ratio"] = (
    seg["num_under_14"] + seg["num_aged_65_plus"]
) / seg["household_size"].replace(0, np.nan)
seg["earners_ratio"] = seg["num_earners"] / (
    seg["household_size"] - seg["num_under_14"] - seg["num_aged_65_plus"]
).replace(0, np.nan)
seg["household_segment"] = np.select(
    [
        (seg["household_size"] >= 6) & (seg["dependency_ratio"] >= 1),
        (seg["household_size"] >= 6) & (seg["num_earners"] <= 1),
        (seg["dependency_ratio"] >= 1) & (seg["num_earners"] <= 1),
        (seg["num_earners"] >= 2) & (seg["dependency_ratio"] < 1),
    ],
    [
        "Large high-dependency households",
        "Large households with few earners",
        "High-dependency households with few earners",
        "Multiple-earner, lower-dependency households",
    ],
    default="Other household profiles"
)

d = hh.copy()
d["household_segment"] = seg["household_segment"]

# Rule-based household segmentation
print("Household segments")
display(
    seg["household_segment"]
    .value_counts()
    .rename_axis("Segment")
    .to_frame("Households")
    .assign(**{"Share (%)": lambda frame: (frame["Households"] / len(seg) * 100).round(1)})
)


In [ ]:
def get_range(s):
    return f"{int(s.min())} - {int(s.max())}"

# Profile each household segment
profile_data = hh.copy()
profile_data["household_segment"] = seg["household_segment"]

profile_data["income_per_capita"] = (
    profile_data["income_total_disposable"] /
    profile_data["household_size"]
)

profile = profile_data.groupby("household_segment", observed=True).agg(
    Households=("household_size", "size"),
    Median_income=("income_per_capita", "median"),
    Mean_household_size=("household_size", "mean"),

    Children_range=("num_under_14", get_range),
    Earners_range=("num_earners", get_range),
    Elderly_range=("num_aged_65_plus", get_range),

    Housing_deprivation=("housing_deprivation",
                         lambda s: s.eq("Severely deprived").mean() * 100),
    Social_protection=("social_protection_group",
                       lambda s: s.eq("Covered").mean() * 100)
).round(1)

profile["Share (%)"] = (
    profile["Households"] / profile["Households"].sum() * 100
).round(1)

#(Range)
profile = profile.rename(columns={
    "Children_range": "Children (Min-Max)",
    "Earners_range": "Earners (Min-Max)",
    "Elderly_range": "Elderly (Min-Max)"
})

display(profile.reset_index().sort_values("Median_income"))


In [ ]:
# Reorder and display the main household-segment profile columns so the segments are easy to
# compare.

profile_display = profile.reset_index()

cols = [
    "household_segment", "Households", "Share (%)",
    "Median_income", "Mean_household_size", "Earners (Min-Max)",
    "Children (Min-Max)", "Elderly (Min-Max)",
    "Housing_deprivation", "Social_protection"
]

display(
    profile_display[cols].sort_values("Median_income").style
    .format({
        "Share (%)": "{:.1f}%",
        "Median_income": "{:,.0f}",
        "Mean_household_size": "{:.1f}",
        "Housing_deprivation": "{:.1f}%",
        "Social_protection": "{:.1f}%"
    })
    .hide(axis="index")
    .set_caption("Household Segment Profiles")
)


In [ ]:
# Build weighted segment summaries. The weighted-median helper accounts for household survey
# weights rather than treating every household equally.

# Weighted segment profiles
d = hh.copy()
d["household_segment"] = seg["household_segment"]
d["income_per_capita"] = d["income_total_disposable"] / d["household_size"]

def weighted_median(x, w):
    valid = x.notna() & w.notna() & (w > 0)
    x, w = x[valid], w[valid]
    order = np.argsort(x)
    x, w = x.iloc[order], w.iloc[order]
    return x.iloc[np.searchsorted(w.cumsum().to_numpy(), w.sum() / 2)]

rows = []
for name, g in d.groupby("household_segment", observed=True):
    w = g["household_weight"]
    valid_w = w.notna() & (w > 0)

    rows.append({
        "Segment": name,
        "Households": len(g),
        "Weighted share (%)": 100 * w[valid_w].sum() / d["household_weight"].where(d["household_weight"] > 0).sum(),
        "Weighted median income": weighted_median(g["income_per_capita"], w),
        "Weighted severe deprivation (%)": 100 * np.average(
            g.loc[valid_w, "housing_deprivation"].eq("Severely deprived"),
            weights=w[valid_w]
        ),
        "Weighted social protection (%)": 100 * np.average(
            g.loc[valid_w, "social_protection_group"].eq("Covered"),
            weights=w[valid_w]
        )
    })

weighted_profile = pd.DataFrame(rows).sort_values("Weighted median income")

display(weighted_profile.style.hide(axis="index").format({
    "Weighted share (%)": "{:.1f}%",
    "Weighted median income": "{:,.0f}",
    "Weighted severe deprivation (%)": "{:.1f}%",
    "Weighted social protection (%)": "{:.1f}%"
}))


In [ ]:
# For each segment, calculate its share of households in the bottom income quintile and related
# summary measures.

# Bottom-income quintile by household segment
d["bottom_quintile"] = d["income_group"].eq("Bottom 20%")

bottom_profile = d.groupby("household_segment", observed=True).apply(
    lambda g: pd.Series({
        "Households": len(g),
        "Weighted share (%)": 100 * g["household_weight"].sum() / d["household_weight"].sum(),
        "Bottom quintile (%)": 100 * np.average(
            g["bottom_quintile"],
            weights=g["household_weight"]
        )
    }),
    include_groups=False
).reset_index()

display(
    bottom_profile.sort_values("Bottom quintile (%)", ascending=False)
    .style.hide(axis="index")
    .format({
        "Weighted share (%)": "{:.1f}%",
        "Bottom quintile (%)": "{:.1f}%"
    })
)


In [ ]:
# Compare low-income status across segments and broad geographic regions. Group records before
# calculating the regional table.
# Geographic concentration of each household segment
geo = d.copy()

geo["bottom_quintile"] = geo["income_group"].eq("Bottom 20%")

geo_table = (
    geo.groupby(["household_segment", "macro_region"], observed=True)
       .agg(
           Households=("household_size", "size"),
           Weighted_households=("household_weight", "sum"),
           Bottom_quintile_rate=("bottom_quintile", "mean")
       )
       .reset_index()
)

# Share of each segment's weighted households located in each region
geo_table["Segment regional share (%)"] = (
    geo_table["Weighted_households"]
    / geo_table.groupby("household_segment")["Weighted_households"].transform("sum")
    * 100
)

display(
    geo_table.sort_values(
        ["household_segment", "Segment regional share (%)"],
        ascending=[True, False]
    ).round(1)
)


In [ ]:
# Build another segment-by-region profile using group-specific calculations rather than a simple
# overall average.
# Correct geographic profile by household segment
geo = d.copy()
geo["bottom_quintile"] = geo["income_group"].eq("Bottom 20%")

rows = []

for (segment_name, region_name), g in geo.groupby(
    ["household_segment", "macro_region"], observed=True
):
    valid = g["household_weight"].notna() & (g["household_weight"] > 0)
    w = g.loc[valid, "household_weight"]

    rows.append({
        "Segment": segment_name,
        "Region": region_name,
        "Households": len(g),
        "Weighted households": w.sum(),
        "Bottom quintile (%)": (
            100 * np.average(g.loc[valid, "bottom_quintile"], weights=w)
            if len(w) > 0 else np.nan
        )
    })

geo_corrected = pd.DataFrame(rows)

geo_corrected["Regional share within segment (%)"] = (
    geo_corrected["Weighted households"]
    / geo_corrected.groupby("Segment")["Weighted households"].transform("sum")
    * 100
)

display(
    geo_corrected.sort_values(
        ["Segment", "Regional share within segment (%)"],
        ascending=[True, False]
    ).round(1).reset_index(drop=True)
)


In [ ]:
# Compute statistics by household segment using only positive, available survey weights. This
# keeps weighted estimates valid.
# Weighted education and employment profile by segment

rows = []

for segment_name, g in d.groupby("household_segment", observed=True):
    valid = g["household_weight"].notna() & (g["household_weight"] > 0)
    gv = g.loc[valid]
    w = gv["household_weight"]

    rows.append({
        "Segment": segment_name,
        "Weighted share (%)": 100 * w.sum() / d.loc[
            d["household_weight"].notna() & (d["household_weight"] > 0),
            "household_weight"
        ].sum(),
        "Head with no/low education (%)": 100 * np.average(
            gv["head_education_level"].isin(["Illiterate", "Read and write", "Basic education"]),
            weights=w
        ),
        "Head unemployed (%)": 100 * np.average(
            gv["head_employment_status"].eq("Unemployed"),
            weights=w
        ),
        "Covered by social protection (%)": 100 * np.average(
            gv["social_protection_group"].eq("Covered"),
            weights=w
        ),
        "Has disability/chronic burden (%)": 100 * np.average(
            gv["health_group"].eq("Has disabled/chronic member"),
            weights=w
        )
    })

segment_context = pd.DataFrame(rows)

display(
    segment_context.sort_values(
        "Weighted share (%)", ascending=False
    ).round(1).reset_index(drop=True)
)


In [ ]:
# Calculate weighted shares and contextual variables for each segment. Also calculate the total
# valid survey weight for comparisons.
# Correct weighted education, employment and vulnerability profiles

rows = []

valid_all = d["household_weight"].notna() & (d["household_weight"] > 0)
total_weight = d.loc[valid_all, "household_weight"].sum()

for segment_name, g in d.groupby("household_segment", observed=True):
    valid = g["household_weight"].notna() & (g["household_weight"] > 0)
    gv = g.loc[valid]
    w = gv["household_weight"]

    rows.append({
        "Segment": segment_name,
        "Households": len(g),
        "Weighted share (%)": 100 * w.sum() / total_weight,

        "Head education: None (%)": 100 * np.average(
            gv["head_education_level"].eq("None"), weights=w
        ),
        "Head education: Primary/Lower secondary (%)": 100 * np.average(
            gv["head_education_level"].eq("Primary/Lower secondary"), weights=w
        ),
        "Head education: Secondary (%)": 100 * np.average(
            gv["head_education_level"].eq("Secondary"), weights=w
        ),

        "Employee (%)": 100 * np.average(
            gv["head_employment_status"].eq("Employee"), weights=w
        ),
        "Self-employed (%)": 100 * np.average(
            gv["head_employment_status"].eq(
                "Own-account, self-employed"
            ), weights=w
        ),
        "No current job / not applicable (%)": 100 * np.average(
            gv["head_employment_status"].eq(
                "Not applicable - no current job"
            ), weights=w
        ),

        "Social protection covered (%)": 100 * np.average(
            gv["social_protection_group"].eq("Covered"), weights=w
        ),
        "Disability/chronic burden (%)": 100 * np.average(
            gv["health_group"].eq("Has disabled/chronic member"), weights=w
        )
    })

segment_context = pd.DataFrame(rows)

display(
    segment_context.sort_values(
        "Weighted share (%)", ascending=False
    ).round(1).reset_index(drop=True)
)


In [ ]:
# Summarize income-source categories separately for each household segment to compare patterns of
# income.
# Weighted income-source composition by household segment

income_source_profile = (
    d.groupby(
        ["household_segment", "income_source"],
        observed=True
    )["household_weight"]
    .sum()
    .rename("Weighted households")
    .reset_index()
)

income_source_profile["Share within segment (%)"] = (
    income_source_profile["Weighted households"]
    / income_source_profile.groupby("household_segment")[
        "Weighted households"
    ].transform("sum")
    * 100
)

display(
    income_source_profile.sort_values(
        ["household_segment", "Share within segment (%)"],
        ascending=[True, False]
    ).round(1).reset_index(drop=True)
)


In [ ]:
# Select key weighted profile measures into a smaller priority table for interpretation.
# Rule-based vulnerability priority table

priority = weighted_profile[
    [
        "Segment",
        "Weighted share (%)",
        "Weighted median income",
        "Weighted severe deprivation (%)",
        "Weighted social protection (%)"
    ]
].copy()

bottom_lookup = bottom_profile.set_index("household_segment")["Bottom quintile (%)"]
priority["Bottom quintile (%)"] = priority["Segment"].map(bottom_lookup)

# Lower income, more bottom-quintile households, more deprivation,
# and lower social-protection coverage indicate greater potential need.
priority["Priority score"] = (
    priority["Bottom quintile (%)"].rank(pct=True)
    + priority["Weighted severe deprivation (%)"].rank(pct=True)
    + (100 - priority["Weighted social protection (%)"]).rank(pct=True)
    + (priority["Weighted median income"].max() - priority["Weighted median income"]).rank(pct=True)
)

priority = priority.sort_values("Priority score", ascending=False)

display(priority.round(1).reset_index(drop=True))


In [ ]:
# Inspect the priority table and display the most useful comparison columns.
# Verify priority score and display columns clearly

print("Priority columns:")
print(priority.columns.tolist())

display(
    priority[
        [
            "Segment",
            "Weighted share (%)",
            "Weighted median income",
            "Weighted severe deprivation (%)",
            "Weighted social protection (%)",
            "Bottom quintile (%)",
            "Priority score"
        ]
    ]
    .sort_values("Priority score", ascending=False)
    .round(2)
    .reset_index(drop=True)
)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">8.1.4 Segment Heatmap and Descriptive Comparison</h3></div>

In [ ]:
# ==============================================================================
# ==============================================================================
heat_data = d.copy()

heat_data["dependency_ratio"] = seg["dependency_ratio"]
heat_data["earners_ratio"] = seg["earners_ratio"]

heat_data["income_per_capita"] = heat_data["income_total_disposable"] / heat_data["household_size"]

heat_data["bottom_quintile"] = heat_data["income_group"].eq("Bottom 20%").astype(float)
heat_data["severe_deprivation"] = heat_data["housing_deprivation"].eq("Severely deprived").astype(float)
heat_data["social_protection"] = heat_data["social_protection_group"].eq("Covered").astype(float)
heat_data["health_burden"] = heat_data["health_group"].eq("Has disabled/chronic member").astype(float)

metrics = {
    "Median income per capita": ("income_per_capita", "median"), # Corrected median income calculation
    "Bottom income quintile (%)": ("bottom_quintile", "mean"),
    "Severe housing deprivation (%)": ("severe_deprivation", "mean"),
    "Social protection coverage (%)": ("social_protection", "mean"),
    "Disability/chronic burden (%)": ("health_burden", "mean"),
    "Mean dependency ratio": ("dependency_ratio", "mean"),
    "Mean earners ratio": ("earners_ratio", "mean"),
}

rows = []
for name, g in heat_data.groupby("household_segment", observed=True):
    row = {"Segment": name}
    w_all = g["household_weight"]

    for label, (column, method) in metrics.items():
        valid = g[column].notna() & w_all.notna() & (w_all > 0)
        if not valid.any():
            row[label] = np.nan
            continue

        x, w = g.loc[valid, column], w_all.loc[valid]

        if method == "median":
            order = np.argsort(x.to_numpy())
            x_sorted, w_sorted = x.iloc[order], w.iloc[order]
            row[label] = x_sorted.iloc[np.searchsorted(w_sorted.cumsum().to_numpy(), w_sorted.sum() / 2)]
        else:
            value = np.average(x, weights=w)
            row[label] = value * 100 if label.endswith("(%)") else value

    rows.append(row)

heat_profile = pd.DataFrame(rows).set_index("Segment")

heat_z = heat_profile.apply(
    lambda col: (col - col.mean()) / col.std(ddof=0) if col.std(ddof=0) != 0 else col * 0, axis=0
)

# ==============================================================================
# ==============================================================================
BG_COLOR = "#0b0905"
TEXT_COLOR = "#F5EFE0"

colors = ['#5C4A14', BG_COLOR, '#F3D77A']
gold_diverging = LinearSegmentedColormap.from_list('gold_div', colors)

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

sns.heatmap(
    heat_z,
    annot=heat_profile.round(1), # Display actual values
    fmt=".1f",
    cmap=gold_diverging,
    center=0,
    linewidths=1.5,
    linecolor=BG_COLOR,
    cbar_kws={"label": "Relative level across segments (z-score)"},
    ax=ax
)

ax.set_title("◆ Household Segment Profile Heatmap", color=TEXT_COLOR, weight="bold", fontsize=16, pad=20, loc="left")
ax.set_xlabel("")
ax.set_ylabel("")

ax.tick_params(colors=TEXT_COLOR, labelsize=11)
plt.xticks(rotation=25, ha="right", weight="bold")
plt.yticks(rotation=0, weight="bold")

cbar = ax.collections[0].colorbar
cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR)
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color=TEXT_COLOR)
cbar.set_label("Relative level across segments (z-score)", color=TEXT_COLOR, weight="bold")

plt.tight_layout()
plt.show()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">8.1.5 Regional Descriptive Statistics</h3></div>

In [ ]:
# Prepare household income per person by region using cleaned household records and survey
# weights.

# Temporary analysis dataframe; do not modify hh
regional_data = hh[
    [
        "income_total_disposable",
        "household_size",
        "household_weight",
        "region",
        "macro_region",
        "urban_rural",
    ]
].copy()

regional_data["income_per_capita"] = (
    regional_data["income_total_disposable"]
    / regional_data["household_size"].replace(0, np.nan)
)

regional_data = regional_data.dropna(
    subset=[
        "income_per_capita",
        "household_weight",
        "region",
    ]
)

regional_data = regional_data[
    (regional_data["household_size"] > 0)
    & (regional_data["household_weight"] > 0)
    & (regional_data["income_per_capita"] > 0)
].copy()

print("Valid households:", f"{len(regional_data):,}")
print(
    "Weighted mean income per capita:",
    f"{np.average(regional_data['income_per_capita'], weights=regional_data['household_weight']):,.2f} EGP"
)
print(
    "Weighted median is not calculated by this cell; "
    "we will calculate it separately if needed."
)

print("\nIncome per capita summary:")
display(regional_data["income_per_capita"].describe().round(2))

print("\nHouseholds by region:")
display(
    regional_data.groupby("region", observed=True)
    .size()
    .sort_values(ascending=False)
    .rename("household_count")
    .to_frame()
)


In [ ]:
# Calculate an urban-versus-rural comparison from the prepared regional household data.
# Weighted mean income per capita by Urban/Rural residence

urban_rural_results = (
    regional_data
    .groupby("urban_rural", observed=True)
    .apply(
        lambda g: np.average(
            g["income_per_capita"],
            weights=g["household_weight"]
        ),
        include_groups=False
    )
    .rename("weighted_mean_income_pc")
    .reset_index()
)

display(urban_rural_results.round(2))

urban_rural_gap = (
    urban_rural_results.set_index("urban_rural")["weighted_mean_income_pc"]
)

if {"Urban", "Rural"}.issubset(set(urban_rural_gap.index)):
    gap = urban_rural_gap["Urban"] - urban_rural_gap["Rural"]
    print(f"\nUrban − Rural weighted mean gap: {gap:,.2f} EGP")
else:
    print("\nCheck the exact Urban/Rural category labels shown in the table.")


In [ ]:
# ============================================================
# Weighted Regional Statistics
# ============================================================

regional_weighted = hh[
    [
        "income_per_capita",
        "household_weight",
        "region",
        "urban_rural",
    ]
].copy()

# Convert required columns to numeric
regional_weighted["income_per_capita"] = pd.to_numeric(
    regional_weighted["income_per_capita"],
    errors="coerce"
)

regional_weighted["household_weight"] = pd.to_numeric(
    regional_weighted["household_weight"],
    errors="coerce"
)

# Keep valid observations only
regional_weighted = regional_weighted[
    regional_weighted["income_per_capita"].notna()
    & regional_weighted["household_weight"].notna()
    & regional_weighted["household_weight"].gt(0)
    & regional_weighted["income_per_capita"].gt(0)
].copy()


# Weighted statistics by region
regional_rows = []

for region_name, group in regional_weighted.groupby(
    "region",
    observed=True
):
    regional_rows.append(
        {
            "Region": region_name,
            "Households": len(group),
            "Weighted mean income per capita": weighted_mean(
                group["income_per_capita"],
                group["household_weight"]
            ),
            "Weighted median income per capita": weighted_median(
                group["income_per_capita"],
                group["household_weight"]
            ),
            "Minimum": group["income_per_capita"].min(),
            "Maximum": group["income_per_capita"].max(),
        }
    )

regional_weighted_stats = (
    pd.DataFrame(regional_rows)
    .set_index("Region")
    .sort_values(
        "Weighted median income per capita",
        ascending=False
    )
    .round(2)
)

display(regional_weighted_stats)


# Weighted statistics by Urban/Rural residence
urban_rural_rows = []

for residence, group in regional_weighted.groupby(
    "urban_rural",
    observed=True
):
    urban_rural_rows.append(
        {
            "Residence": residence,
            "Households": len(group),
            "Weighted mean income per capita": weighted_mean(
                group["income_per_capita"],
                group["household_weight"]
            ),
            "Weighted median income per capita": weighted_median(
                group["income_per_capita"],
                group["household_weight"]
            ),
        }
    )

urban_rural_weighted_stats = (
    pd.DataFrame(urban_rural_rows)
    .set_index("Residence")
    .round(2)
)

display(urban_rural_weighted_stats)


In [ ]:
# ============================================================
# Categorical Frequency and Percentage Tables
# Weighted and Unweighted
# ============================================================

categorical_columns = {
    "Urban/Rural residence": "urban_rural",
    "Region": "region",
    "Head education level": "head_education_level",
    "Head gender": "head_gender",
    "Head employment status": "head_employment_status",
    "Income source": "income_source",
    "Housing deprivation": "housing_deprivation",
    "Asset group": "asset_group",
}

required_columns = [
    "household_weight",
    *categorical_columns.values()
]

missing_columns = [
    col for col in required_columns
    if col not in hh.columns
]

if missing_columns:
    raise KeyError(
        f"Missing columns: {missing_columns}"
    )


def categorical_frequency_table(data, column):
    temp = data[
        [column, "household_weight"]
    ].copy()

    # Convert category to text and label missing values
    temp["Category"] = (
        temp[column]
        .astype("string")
        .fillna("Missing")
    )

    # Convert weights to numeric
    temp["household_weight"] = pd.to_numeric(
        temp["household_weight"],
        errors="coerce"
    )

    # Keep valid positive weights
    temp = temp[
        temp["household_weight"].notna()
        & temp["household_weight"].gt(0)
    ].copy()

    # Unweighted counts
    result = (
        temp.groupby("Category", observed=False)
        .size()
        .rename("Count")
        .to_frame()
    )

    result["Percentage"] = (
        result["Count"]
        / result["Count"].sum()
        * 100
    )

    # Weighted counts
    weighted_counts = (
        temp.groupby("Category", observed=False)[
            "household_weight"
        ]
        .sum()
        .rename("Weighted count")
    )

    result = result.join(weighted_counts)

    result["Weighted percentage"] = (
        result["Weighted count"]
        / result["Weighted count"].sum()
        * 100
    )

    result = result.sort_values(
        "Weighted percentage",
        ascending=False
    )

    return result.round(2)


# Store all tables in one dictionary
categorical_descriptive_tables = {}

for label, column in categorical_columns.items():
    categorical_descriptive_tables[label] = (
        categorical_frequency_table(hh, column)
    )

    print(f"\n{label} — {column}")

    display(
        categorical_descriptive_tables[label]
    )


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">8.2 Inferential Statistics</h2></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">8.2.1 Chi-Square Tests and Group Comparisons</h3></div>

In [ ]:
# Run categorical statistical tests and calculations for household-group comparisons. Use the chi-
# square distribution where required by the test.

W = "household_weight"

def chi_test(label, row, col, data=None):
    d = (hh if data is None else data)[[row, col, W]].dropna(subset=[W]).copy()
    for c in (row, col):
        if not isinstance(d[c].dtype, pd.CategoricalDtype):
            d[c] = d[c].astype(object).fillna("Not stated")
    T = d.groupby([row, col], observed=True)[W].sum().unstack(fill_value=0)
    P = T / T.values.sum()                                    # weighted proportions
    r, c_ = P.sum(axis=1), P.sum(axis=0)
    E = np.outer(r, c_)                                       # expected under independence
    n_eff = d[W].sum() ** 2 / (d[W] ** 2).sum()              # effective sample size from the weights
    phi2 = ((P - E) ** 2 / E).values.sum()
    dof = (P.shape[0] - 1) * (P.shape[1] - 1)
    V = np.sqrt(phi2 / (min(P.shape) - 1))
    Z = ((P - E) / np.sqrt(E * np.outer(1 - r, 1 - c_) / n_eff)).unstack()   # adjusted residuals
    share = P.div(r, axis=0) * 100                            # weighted % inside each row group

    def cell(k):                                              # k = (column category, row group)
        return f"{k[0]} in {k[1]} ({share.loc[k[1], k[0]]:.0f}% vs {c_[k[0]] * 100:.0f}%)" if abs(Z[k]) > 1.96 else "-"

    return {"Test": label, "Cramer's V": V,
            "Strength": "negligible" if V < 0.1 else "weak" if V < 0.2 else "moderate" if V < 0.3 else "strong",
            "p-value": chi2.sf(n_eff * phi2, dof),
            "More than expected": cell(Z.idxmax()), "Less than expected": cell(Z.idxmin())}

TESTS = [
    ("Education of head", "income_group", "head_education_level"),
    ("Employment status of head", "income_group", "head_employment_status"),
    ("Source of income", "income_group", "income_source"),
    ("Urban / Rural", "income_group", "urban_rural"),
    ("Governorate", "income_group", "region"),
    ("Gender of head", "income_group", "head_gender"),
    ("Social protection", "income_group", "social_protection_group"),
    ("Housing deprivation", "income_group", "housing_deprivation"),
    ("Asset index", "income_group", "asset_group"),
    ("Household composition", "income_group", "household_composition"),
]
results = [chi_test(*t) for t in TESTS]
for place in hh["urban_rural"].dropna().unique():
    results.append(chi_test(f"Education of head | {place} only", "income_group", "head_education_level",
                            data=hh[hh["urban_rural"] == place]))

summary = pd.DataFrame(results).sort_values("Cramer's V", ascending=False)

summary_view = summary.rename(columns={
    "Cramer's V": "V",
    "Strength": "Level",
    "p-value": "p",
    "More than expected": "Above expected",
    "Less than expected": "Below expected"
})

styled_summary = (
    summary_view.style
    .hide(axis="index")
    .format({
        "V": "{:.2f}",
        "p": lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"
    })
    .bar(subset=["V"], color="#8a6a1c", vmin=0, vmax=0.5)
    .set_caption("Income group vs household characteristics")
    .set_table_styles([
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
                ("table-layout", "fixed"),
                ("font-family", "DejaVu Sans, Arial, sans-serif"),
                ("font-size", "12px")
            ]
        },
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "15px"),
                ("font-weight", "bold"),
                ("color", "#f3d77a"),
                ("padding", "10px 14px"),
                ("text-align", "left")
            ]
        },
        {
            "selector": "thead th",
            "props": [
                ("background-color", "#17130a"),
                ("color", "#f3d77a"),
                ("font-weight", "bold"),
                ("text-align", "left"),
                ("padding", "8px 10px"),
                ("border-bottom", "2px solid #8a6a1c")
            ]
        },
        {
            "selector": "tbody td",
            "props": [
                ("background-color", "#0b0905"),
                ("color", "#f5efe0"),
                ("padding", "8px 10px"),
                ("border-bottom", "1px solid #2b2410"),
                ("vertical-align", "top")
            ]
        },
        {
            "selector": "tbody tr:nth-child(even) td",
            "props": [("background-color", "#100d07")]
        },
        {
            "selector": "th.col0, td.col0",
            "props": [("width", "18%")]
        },
        {
            "selector": "th.col1, td.col1",
            "props": [("width", "8%")]
        },
        {
            "selector": "th.col2, td.col2",
            "props": [("width", "9%")]
        },
        {
            "selector": "th.col3, td.col3",
            "props": [("width", "8%")]
        },
        {
            "selector": "th.col4, td.col4",
            "props": [("width", "28%")]
        },
        {
            "selector": "th.col5, td.col5",
            "props": [("width", "29%")]
        }
    ])
    .set_properties(
        subset=["Above expected", "Below expected"],
        **{
            "white-space": "normal",
            "text-align": "left",
            "line-height": "1.35"
        }
    )
    .set_properties(
        subset=["V", "Level", "p"],
        **{"text-align": "center"}
    )
)

display(styled_summary)

display(HTML("""
<div style="background-color:#0b0905; border:1px solid #2b2410; border-left:4px solid #d4af37; border-radius:6px; padding:10px 14px; color:#a8a086; font-family:Arial,sans-serif; font-size:11.5px; line-height:1.5; margin-top:8px;">
  <b style="color:#f3d77a;">Cramér's V:</b> &lt;0.1 negligible · &lt;0.2 weak · &lt;0.3 moderate · 0.3+ strong
  <br>
  Percentages compare each category within its income group with its overall percentage.
  Results use household weights; only cells with |z| &gt; 1.96 are shown.
</div>
"""))


In [ ]:
# Calculate further statistical comparisons among household variables. Use the selected scipy
# methods and combinations of variables.

y = np.log((hh["income_total_disposable"] / hh["household_size"]).where(
    lambda s: s > 0
))

def prepare(var, min_n=10):
    d = pd.DataFrame({"g": hh[var].astype(object).fillna("Not stated"), "y": y}).dropna()
    s = d.groupby("g")["y"].agg(["count", "mean", "var"])
    s = s[s["count"] >= min_n]
    return d[d["g"].isin(s.index)], s

def welch_p(s):
    k, w = len(s), s["count"] / s["var"]
    mw = (w * s["mean"]).sum() / w.sum()
    t = (((1 - w / w.sum()) ** 2) / (s["count"] - 1)).sum()
    F = (w * (s["mean"] - mw) ** 2).sum() / ((k - 1) * (1 + 2 * (k - 2) * t / (k**2 - 1)))
    return st.f.sf(F, k - 1, (k**2 - 1) / (3 * t))

def omega_sq(s):
    k, N = len(s), s["count"].sum()
    M = (s["count"] * s["mean"]).sum() / N
    ssb = (s["count"] * (s["mean"] - M)**2).sum()
    ssw = ((s["count"] - 1) * s["var"]).sum()
    msw = ssw / (N - k)
    return max(0, (ssb - (k - 1) * msw) / (ssb + ssw + msw))

def games_howell(var):
    _, s = prepare(var)
    k, rows = len(s), []

    for a, b in combinations(s.index, 2):
        na, nb = s.loc[[a, b], "count"]
        va, vb = s.loc[[a, b], "var"]
        diff = s.loc[a, "mean"] - s.loc[b, "mean"]
        se2 = va / na + vb / nb
        dfw = se2**2 / ((va / na)**2 / (na - 1) + (vb / nb)**2 / (nb - 1))
        half = st.studentized_range.ppf(0.95, k, dfw) * np.sqrt(se2 / 2)
        p = st.studentized_range.sf(abs(diff) * np.sqrt(2 / se2), k, dfw)
        g = diff / np.sqrt(((na - 1) * va + (nb - 1) * vb) / (na + nb - 2))

        rows.append((a, b, *(np.expm1([diff, diff-half, diff+half]) * 100), p, g))

    return pd.DataFrame(rows, columns=[
        "A", "B", "A vs B %", "CI low %", "CI high %", "p", "Hedges g"
    ])

def summarize(label, var):
    d, s = prepare(var)
    w2, gh = omega_sq(s), games_howell(var)
    geo = s.assign(half=st.t.ppf(0.975, s["count"]-1) * np.sqrt(s["var"] / s["count"]))

    def show(g):
        m, h = geo.loc[g, ["mean", "half"]]
        return f"{np.exp(m):,.0f} [{np.exp(m-h):,.0f}-{np.exp(m+h):,.0f}]"

    return {
        "Variable": label,
        "Groups": len(s),
        "Welch p": welch_p(s),
        "Kruskal p": st.kruskal(*[x["y"].values for _, x in d.groupby("g")]).pvalue,
        "Omega2": w2,
        "Effect": "negligible" if w2 < 0.01 else "small" if w2 < 0.06 else "medium" if w2 < 0.14 else "large",
        "Significant pairs": f"{(gh['p'] < 0.05).sum()} / {len(gh)}",
        "Substantial pairs": f"{((gh['p'] < 0.05) & (gh['Hedges g'].abs() >= 0.5)).sum()} / {len(gh)}",
        "Highest group": show(s["mean"].idxmax()),
        "Lowest group": show(s["mean"].idxmin())
    }

VARS = [
    ("Education of head", "head_education_level"),
    ("Employment status of head", "head_employment_status"),
    ("Source of income", "income_source"),
    ("Region (7 areas)", "area"),
    ("Household composition", "household_composition"),
    ("Job sector of head", "head_job_sector"),
    ("Asset index group", "asset_group"),
    ("Number of earners", "earners_group"),
]

summary = pd.DataFrame([summarize(*v) for v in VARS]).sort_values("Omega2", ascending=False)
p_fmt = lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"

display(summary.style.hide(axis="index")
    .format({"Welch p": p_fmt, "Kruskal p": p_fmt, "Omega2": "{:.3f}"})
    .bar(subset="Omega2", color="#8a6a1c", vmin=0, vmax=0.3)
    .set_properties(**{"text-align": "left"})
    .set_caption("Log income per capita by group (ordered by effect size)"))

print("Effect size (omega squared): <0.01 negligible | <0.06 small | <0.14 medium | 0.14+ large")
print("Substantial pair = Games-Howell p < 0.05 and |Hedges g| >= 0.5. Highest/Lowest = geometric mean income per capita [95% CI].")
print("For pairwise details of one variable run: games_howell('head_education_level')")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">8.2.2 Regional Confidence Intervals</h3></div>

In [ ]:
# Use a fixed random generator to compute repeated samples for uncertainty estimation. The number
# of repetitions is set by B.

rng = np.random.default_rng(42)
B = 1000

regional_results = []

for region, group in regional_data.groupby("region", observed=True):
    income = group["income_per_capita"].to_numpy(dtype=float)
    weights = group["household_weight"].to_numpy(dtype=float)

    weighted_mean = np.average(income, weights=weights)

    # Approximate weighted bootstrap:
    # Resample households with probability proportional to their weights.
    probabilities = weights / weights.sum()
    n = len(group)

    bootstrap_means = np.empty(B)

    for b in range(B):
        sampled_idx = rng.choice(
            n,
            size=n,
            replace=True,
            p=probabilities
        )
        bootstrap_means[b] = income[sampled_idx].mean()

    ci_low, ci_high = np.percentile(bootstrap_means, [2.5, 97.5])

    regional_results.append({
        "region": region,
        "households": n,
        "weighted_mean_income_pc": weighted_mean,
        "ci_low": ci_low,
        "ci_high": ci_high
    })

regional_results = pd.DataFrame(regional_results).sort_values(
    "weighted_mean_income_pc"
).reset_index(drop=True)

display(
    regional_results.round({
        "weighted_mean_income_pc": 2,
        "ci_low": 2,
        "ci_high": 2
    })
)


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">8.2.3 Outlier Detection and Treatment — Income per Capita</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Finds unusually high or low household incomes, checks whether they are data errors or real households, and decides how to treat them before the ANOVA.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:4px solid #8a6a1c;border-radius:6px;padding:9px 16px 8px;margin:16px 0 4px;"><h4 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:15.5px;line-height:1.25;font-weight:700;color:#d4af37;">8.2.3.1 Preparing the Income-per-Capita Variable</h4><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Builds income per person, drops unusable values and checks the skewness that the ANOVA must handle.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:12px 20px 10px;margin:10px 0;font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:13.5px;line-height:1.55;color:#f5efe0;"><div style="font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:16px;font-weight:700;color:#f3d77a;margin:0 0 6px;">&#9670;&nbsp; Outliers</div><ul style="margin:0;padding-left:20px;color:#f5efe0;"><li style="margin:4px 0;">126 households (1.0%) have unusually high or low income per person (below 3,961 or above 94,451 EGP per year).</li><li style="margin:4px 0;">Raw income is very right-skewed (skewness 10.6), so we used the log scale (skewness 0.5).</li><li style="margin:4px 0;">These are real households, not data errors, so we <b style="color:#f3d77a;">kept and flagged them</b> (<code style="background-color:#17130a;color:#f3d77a;border:1px solid #2b2410;border-radius:4px;padding:1px 6px;font-family:SFMono-Regular,Consolas,'Liberation Mono',monospace;font-size:.92em;">income_outlier_flag</code>) instead of deleting them.</li></ul></div>

In [ ]:
# Calculate income per person, take its log (income is very right-skewed), find outliers with the
# IQR rule, and flag them. Outliers are real households, so they are kept and only flagged.

hh_a = hh.copy()
hh_a["income_pc"] = hh_a["income_total_disposable"].astype(float) / hh_a["household_size"].astype(float)
hh_a["log_income"] = np.log(hh_a["income_pc"])

q1, q3 = hh_a["log_income"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr

hh_a["outlier"] = (hh_a["log_income"] < low) | (hh_a["log_income"] > high)
hh["income_outlier_flag"] = np.where(hh_a["outlier"], "Outlier", "Typical")

print(f"Skewness raw income : {hh_a['income_pc'].skew():.2f}")
print(f"Skewness log income : {hh_a['log_income'].skew():.2f}")
print(f"Outlier limits (EGP per person): {np.exp(low):,.0f}  to  {np.exp(high):,.0f}")
print(f"Outliers: {hh_a['outlier'].sum()} households ({hh_a['outlier'].mean()*100:.1f}%)")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:11px 18px 10px;margin:20px 0 6px;"><h3 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:18.5px;line-height:1.25;font-weight:700;color:#f3d77a;">8.2.3 Outliers and One-Way ANOVA — Income per Capita</h3><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:12.5px;line-height:1.5;color:#a8a086;font-style:italic;margin-top:5px;">Flags outliers in income per person, then tests whether income differs between household groups.</div></div>

<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:12px 20px 10px;margin:10px 0;font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:13.5px;line-height:1.55;color:#f5efe0;"><div style="font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:16px;font-weight:700;color:#f3d77a;margin:0 0 6px;">&#9670;&nbsp; ANOVA</div><ul style="margin:0;padding-left:20px;color:#f5efe0;"><li style="margin:4px 0;">Income per person differs significantly between groups for all four factors: macro region, urban/rural, head's education and housing deprivation (p &lt; 0.05).</li><li style="margin:4px 0;">The differences are <b style="color:#f3d77a;">small</b> (eta-squared about 0.03 to 0.05). Each factor explains only 3% to 5% of the variation in income, so other factors matter too.</li><li style="margin:4px 0;">The results are almost the same with and without the outliers, so the conclusions <b style="color:#f3d77a;">do not depend on them</b>.</li></ul></div>

In [ ]:
# One-way ANOVA: does income per person (log) differ between groups? Run with all households and
# again without the outliers to see if the result changes. Eta-squared = how big the difference is.

factors = ["macro_region", "urban_rural", "head_education_level", "housing_deprivation"]
no_outliers = hh_a[~hh_a["outlier"]]

def anova(data, col):
    groups = [g["log_income"].values for _, g in data.groupby(col, observed=True)]
    F, p = st.f_oneway(*groups)
    all_values = data["log_income"]
    ss_between = sum(len(g) * (g.mean() - all_values.mean()) ** 2 for g in groups)
    eta2 = ss_between / ((all_values - all_values.mean()) ** 2).sum()
    return F, p, eta2

rows = []
for col in factors:
    F, p, e = anova(hh_a, col)
    F2, p2, e2 = anova(no_outliers, col)
    rows.append([col, F, p, e, e2])

result = pd.DataFrame(rows, columns=["Factor", "F", "p-value", "Eta² (all)", "Eta² (no outliers)"])
display(result.round(4))
print("p-value < 0.05 -> the groups differ.  Eta²: 0.01 small, 0.06 medium, 0.14 large.")


<div style="background-color:#0b0905;border:1px solid #2b2410;border-left:5px solid #d4af37;border-radius:6px;padding:12px 20px 10px;margin:10px 0;font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:13.5px;line-height:1.55;color:#f5efe0;"><div style="font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:16px;font-weight:700;color:#f3d77a;margin:0 0 6px;">&#9670;&nbsp; Regions</div><ul style="margin:0;padding-left:20px;color:#f5efe0;"><li style="margin:4px 0;">Urban Governorates have the highest typical income per person (about 23,950 EGP).</li><li style="margin:4px 0;">Lower Egypt, Frontier and Upper Egypt are about 22% to 28% lower, and Upper Egypt is the lowest.</li></ul></div>

In [ ]:
# Show the typical income per person in each region and how far each region is from the highest one.

typical = hh_a.groupby("macro_region", observed=True)["income_pc"].median().sort_values(ascending=False)
table = pd.DataFrame({
    "Typical income (EGP/person)": typical,
    "Difference from highest (%)": (typical / typical.iloc[0] - 1) * 100,
})
display(table.round(1))


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">8.3 Data Validation and Consistency Checks</h2></div>

In [ ]:
# Print merge dimensions, unique household counts, and records per household. These checks help
# spot accidental row duplication.
print("df_merged:", df_merged.shape)
print("Unique households:", df_merged["household_id"].nunique())
print("Rows per household — summary:")
print(df_merged.groupby("household_id").size().describe())

print("\nHousehold-level table:")
print("hh:", hh.shape)
print("Unique household index:", hh.index.nunique())
print("Index name:", hh.index.name)


In [ ]:
# Identify household variables in the person-household merged table and check whether values
# remain consistent within each household.
household_cols = [
    "income_total_disposable",
    "household_size",
    "num_earners",
    "num_under_14",
    "num_aged_65_plus",
    "region",
    "urban_rural",
    "household_weight"
]

available_cols = [
    col for col in household_cols
    if col in df_merged.columns
]

consistency = (
    df_merged.groupby("household_id")[available_cols]
    .nunique(dropna=False)
)

print("Household-level consistency check")
print("Number of households:", len(consistency))
print("\nNumber of households with inconsistent values per variable:")
print((consistency > 1).sum().sort_values(ascending=False))


In [ ]:
# Select important person-level variables and inspect them for missing values and expected
# formats.
check_cols = [
    "person_id",
    "age",
    "gender",
    "relation_to_head",
    "head_age",
    "head_gender",
    "head_education_level",
    "household_size"
]

check_cols = [col for col in check_cols if col in hh.columns]

print("Household table sample:")
display(hh[check_cols].head(10))

print("\nComparison of person-level vs head-level variables:")
for col in ["age", "gender", "education_level", "employment_status"]:
    head_col = f"head_{col}"

    if col in hh.columns and head_col in hh.columns:
        print(f"\n{col} vs {head_col}:")
        print(
            hh[[col, head_col]]
            .value_counts(dropna=False)
            .head(10)
        )


In [ ]:
# Compare matched individual and household-head fields to check whether corresponding details are
# consistent.
pairs = [
    ("age", "head_age"),
    ("gender", "head_gender"),
    ("education_level", "head_education_level"),
    ("employment_status", "head_employment_status"),
]

for person_col, head_col in pairs:
    if person_col not in hh.columns or head_col not in hh.columns:
        continue

    same = (
        hh[person_col].eq(hh[head_col])
        | (hh[person_col].isna() & hh[head_col].isna())
    )

    print(f"\n{person_col} vs {head_col}")
    print("Matching households:", int(same.sum()))
    print("Different households:", int((~same).sum()))
    print("Match percentage:", round(same.mean() * 100, 2), "%")

    if (~same).any():
        print("Examples of differences:")
        display(hh.loc[~same, [person_col, head_col]].head(5))


<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; SECTION 9</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">9. Engel’s Law</h1><div style="height:1px;background-color:#8a6a1c;margin:14px 0 10px;"></div><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:13.5px;line-height:1.55;color:#a8a086;font-style:italic;">Examines how the share of household expenditure allocated to food changes with income.</div></div>

In [ ]:
# ============================================================
# 9.1 Create Expenditure Quintiles
# ============================================================

data = hh.copy()

if "expenditure_per_capita" not in data.columns:
    data["expenditure_per_capita"] = safe_divide(
        data["exp_total"],
        data["household_size"],
    )

data["expenditure_per_capita"] = pd.to_numeric(
    data["expenditure_per_capita"],
    errors="coerce",
)
data["household_weight"] = pd.to_numeric(
    data["household_weight"],
    errors="coerce",
)

valid = (
    data["expenditure_per_capita"].gt(0)
    & data["household_weight"].gt(0)
)
data = data.loc[valid].copy()

data["exp_quintile"] = pd.qcut(
    data["expenditure_per_capita"].rank(method="first"),
    q=5,
    labels=["Q1 (poorest)", "Q2", "Q3", "Q4", "Q5 (richest)"],
)

print(data["exp_quintile"].value_counts(sort=False))


In [ ]:
# ============================================================
# Household Income Sources and Expenditure Shares
# ============================================================

FE = globals().get("FE", [])

income_total = hh["income_total_disposable"]
total_expenditure = hh["exp_total"]

add_feature(
    hh,
    "wage_share",
    safe_divide(
        hh["income_wages"],
        income_total
    ),
    FE
)

add_feature(
    hh,
    "self_employment_share",
    safe_divide(
        hh["income_self_employment"],
        income_total
    ),
    FE
)

add_feature(
    hh,
    "rental_share",
    safe_divide(
        hh["income_rentals"],
        income_total
    ),
    FE
)

add_feature(
    hh,
    "property_share",
    safe_divide(
        hh["income_property"],
        income_total
    ),
    FE
)

add_feature(
    hh,
    "transfer_share",
    safe_divide(
        hh["income_transfers"],
        income_total
    ),
    FE
)

income_source_columns = [
    "wage_share",
    "self_employment_share",
    "rental_share",
    "property_share",
    "transfer_share",
]

income_source_labels = {
    "wage_share": "Wages",
    "self_employment_share": "Self-employment",
    "rental_share": "Rentals",
    "property_share": "Property",
    "transfer_share": "Transfers",
}

add_feature(
    hh,
    "dominant_income_type",
    hh[income_source_columns]
    .idxmax(axis=1)
    .map(income_source_labels),
    FE
)

add_feature(
    hh,
    "food_share",
    safe_divide(
        hh["exp_total_food"],
        total_expenditure
    ),
    FE
)

add_feature(
    hh,
    "health_share",
    safe_divide(
        hh["exp_health"],
        total_expenditure
    ),
    FE
)

add_feature(
    hh,
    "education_share",
    safe_divide(
        hh["exp_education"],
        total_expenditure
    ),
    FE
)

add_feature(
    hh,
    "high_food_share",
    hh["food_share"].ge(0.5).astype(int),
    FE
)

add_feature(
    hh,
    "catastrophic_health_spending",
    hh["health_share"].ge(0.10).astype(int),
    FE
)

print("Main income source distribution:")
print(
    (
        hh["dominant_income_type"]
        .value_counts(normalize=True)
        * 100
    )
    .round(1)
    .to_string()
)

print(
    f"\nAverage food budget share: "
    f"{hh['food_share'].mean() * 100:.1f}%"
)

print(
    f"Households spending 50% or more on food: "
    f"{hh['high_food_share'].mean() * 100:.1f}%"
)

print(
    f"Households spending 10% or more on health: "
    f"{hh['catastrophic_health_spending'].mean() * 100:.1f}%"
)

In [ ]:
# ============================================================
# 9.1 Engel's Law — Food Budget Share by Expenditure Quintile
# ============================================================

# Work on a separate copy of household-level data
engel = hh.copy()

engel["expenditure_per_capita"] = pd.to_numeric(
    engel["expenditure_per_capita"], errors="coerce"
)
engel["food_share"] = pd.to_numeric(
    engel["food_share"], errors="coerce"
)
engel["household_weight"] = pd.to_numeric(
    engel["household_weight"], errors="coerce"
)

# Keep valid observations
engel = engel[
    engel["expenditure_per_capita"].gt(0)
    & engel["food_share"].notna()
    & engel["household_weight"].gt(0)
].copy()

# Create five approximately equal-sized groups
engel["exp_quintile"] = pd.qcut(
    engel["expenditure_per_capita"].rank(method="first"),
    q=5,
    labels=[
        "Q1 (poorest)", "Q2", "Q3", "Q4", "Q5 (richest)"
    ]
)

# Calculate weighted mean food expenditure share
def weighted_mean_share(group):
    return np.average(
        group["food_share"],
        weights=group["household_weight"]
    ) * 100

engel_results = (
    engel.groupby("exp_quintile", observed=True)
    .apply(weighted_mean_share)
    .rename("Weighted Food Budget Share (%)")
    .reset_index()
)

display(engel_results.round(2))


In [ ]:
# ============================================================
# 9.2 Engel Curve — Food Share vs Per-Capita Expenditure
# ============================================================


engel_plot = (
    engel.groupby("exp_quintile", observed=True)
    .apply(lambda g: pd.Series({
        "Mean Expenditure per Capita": np.average(
            g["expenditure_per_capita"],
            weights=g["household_weight"]
        ),
        "Weighted Food Share (%)": np.average(
            g["food_share"],
            weights=g["household_weight"]
        ) * 100
    }))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    engel_plot["Mean Expenditure per Capita"],
    engel_plot["Weighted Food Share (%)"],
    marker="o",
    linewidth=2
)

for _, row in engel_plot.iterrows():
    ax.annotate(
        str(row["exp_quintile"]),
        (row["Mean Expenditure per Capita"],
         row["Weighted Food Share (%)"]),
        xytext=(5, 7),
        textcoords="offset points"
    )

ax.set_xlabel("Weighted Mean Expenditure per Capita (EGP)")
ax.set_ylabel("Weighted Food Budget Share (%)")
ax.set_title("Engel Curve: Food Budget Share and Expenditure")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; STATISTICAL INSIGHT</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">Why Does the “None” Group Have Higher Income per Capita?</h1><div style="height:1px;background-color:#8a6a1c;margin:14px 0 10px;"></div><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:13.5px;line-height:1.55;color:#a8a086;font-style:italic;">Investigates the unexpected difference in income per capita between households with no recorded education and those with secondary education.</div></div>

In [ ]:
edu = hh.copy()

edu = edu.dropna(
    subset=[
        "head_education_level",
        "income_per_capita",
        "income_total_disposable",
        "household_size",
        "head_age"
    ]
)

summary = (
    edu.groupby("head_education_level", observed=True)
    .agg(
        n=("income_per_capita", "size"),
        median_income_pc=("income_per_capita", "median"),
        mean_income_pc=("income_per_capita", "mean"),
        median_household_income=("income_total_disposable", "median"),
        mean_household_income=("income_total_disposable", "mean"),
        mean_household_size=("household_size", "mean"),
        mean_head_age=("head_age", "mean"),
        mean_children_under_14=("num_under_14", "mean")
    )
    .round(2)
    .sort_values("median_income_pc", ascending=False)
)

display(summary)


In [ ]:
comparison = edu[
    edu["head_education_level"].isin(["None", "Secondary"])
].copy()

none_income = comparison.loc[
    comparison["head_education_level"] == "None",
    "income_per_capita"
].dropna()

secondary_income = comparison.loc[
    comparison["head_education_level"] == "Secondary",
    "income_per_capita"
].dropna()

result = mannwhitneyu(
    none_income,
    secondary_income,
    alternative="two-sided"
)

print("Income per Capita: None vs Secondary")
print("-" * 45)
print(f"None group n: {len(none_income):,}")
print(f"Secondary group n: {len(secondary_income):,}")
print(f"None median: {none_income.median():,.2f}")
print(f"Secondary median: {secondary_income.median():,.2f}")
print(f"Median difference (None - Secondary): "
      f"{none_income.median() - secondary_income.median():,.2f}")
print(f"Mann–Whitney U statistic: {result.statistic:,.2f}")
print(f"P-value: {result.pvalue:.4e}")


In [ ]:
none_size = comparison.loc[
    comparison["head_education_level"] == "None",
    "household_size"
].dropna()

secondary_size = comparison.loc[
    comparison["head_education_level"] == "Secondary",
    "household_size"
].dropna()

valid = edu[["household_size", "income_per_capita"]].dropna()

rho, p_value = spearmanr(
    valid["household_size"],
    valid["income_per_capita"]
)

print("Household Size Analysis")
print("-" * 45)
print(f"None median household size: {none_size.median():.2f}")
print(f"Secondary median household size: {secondary_size.median():.2f}")
print(f"Spearman correlation: {rho:.4f}")
print(f"Correlation p-value: {p_value:.4e}")


In [ ]:
none_age = comparison.loc[
    comparison["head_education_level"] == "None",
    "head_age"
].dropna()

secondary_age = comparison.loc[
    comparison["head_education_level"] == "Secondary",
    "head_age"
].dropna()

valid_age = edu[["head_age", "income_per_capita"]].dropna()

rho_age, p_age = spearmanr(
    valid_age["head_age"],
    valid_age["income_per_capita"]
)

print("Head Age Analysis")
print("-" * 45)
print(f"None group mean age: {none_age.mean():.2f}")
print(f"Secondary group mean age: {secondary_age.mean():.2f}")
print(f"None group median age: {none_age.median():.2f}")
print(f"Secondary group median age: {secondary_age.median():.2f}")
print(f"Spearman correlation: {rho_age:.4f}")
print(f"Correlation p-value: {p_age:.4e}")


In [ ]:
age_comparison = comparison.copy()

age_comparison["age_band"] = pd.cut(
    age_comparison["head_age"],
    bins=[0, 29, 39, 49, 59, 69, np.inf],
    labels=["<30", "30–39", "40–49", "50–59", "60–69", "70+"]
)

age_results = []

for age_band, group in age_comparison.groupby(
    "age_band", observed=True
):
    none_group = group.loc[
        group["head_education_level"] == "None",
        "income_per_capita"
    ].dropna()

    secondary_group = group.loc[
        group["head_education_level"] == "Secondary",
        "income_per_capita"
    ].dropna()

    if len(none_group) < 2 or len(secondary_group) < 2:
        continue

    test = mannwhitneyu(
        none_group,
        secondary_group,
        alternative="two-sided"
    )

    age_results.append({
        "Age Band": str(age_band),
        "None n": len(none_group),
        "Secondary n": len(secondary_group),
        "None Median Income PC": none_group.median(),
        "Secondary Median Income PC": secondary_group.median(),
        "Median Difference": (
            none_group.median() - secondary_group.median()
        ),
        "Mann–Whitney U": test.statistic,
        "P-value": test.pvalue
    })

age_results = pd.DataFrame(age_results)

display(
    age_results.style.format({
        "None Median Income PC": "{:,.2f}",
        "Secondary Median Income PC": "{:,.2f}",
        "Median Difference": "{:,.2f}",
        "Mann–Whitney U": "{:,.2f}",
        "P-value": "{:.4e}"
    })
)


### Statistical Insight: Why Does the “None” Group Have Higher Income per Capita?

The initial comparison revealed that households in the **“None” education category** had a higher median income per capita than those in the **“Secondary” category**. The Mann–Whitney U test indicated a statistically significant difference between the two income distributions (\(p < 0.001\)).

Further investigation examined household size and household-head age as potential factors associated with this pattern. The “None” group had a smaller median household size and a higher average age of household heads than the “Secondary” group. Spearman correlation analysis also indicated a negative association between household size and income per capita, while household-head age was positively associated with income per capita.

Age-stratified comparisons were then conducted to assess whether the income difference persisted across different age groups. The results varied across age bands, suggesting that the observed relationship should be interpreted in the context of household characteristics rather than education category alone.

**Conclusion:** The higher income per capita observed in the “None” group is an exploratory finding that may be associated with differences in household composition and the age of household heads. These results describe statistical associations and do not establish causal relationships. The meaning of the “None” category should also be verified before interpreting it as a specific educational status.

In [ ]:
reg_data = hh.copy()

# Keep the two education groups
reg_data = reg_data[
    reg_data["head_education_level"].astype("string").isin(
        ["None", "Secondary"]
    )
].copy()

# Convert required numeric variables explicitly
numeric_cols = [
    "income_per_capita",
    "head_age",
    "household_size"
]

for col in numeric_cols:
    reg_data[col] = pd.to_numeric(reg_data[col], errors="coerce")

# Log-transform the outcome
reg_data = reg_data[
    reg_data["income_per_capita"].notna()
    & (reg_data["income_per_capita"] > 0)
].copy()

reg_data["log_income_per_capita"] = np.log(
    reg_data["income_per_capita"]
)

# Set the reference group to Secondary
reg_data["head_education_level"] = pd.Categorical(
    reg_data["head_education_level"].astype(str),
    categories=["Secondary", "None"]
)

# Build predictors
numeric_predictors = ["head_age", "household_size"]
categorical_predictors = []

# Add available categorical controls
for col in ["Urban_Rural", "macro_region"]:
    if col in reg_data.columns:
        reg_data[col] = reg_data[col].astype("category")
        categorical_predictors.append(col)

# Add number of earners if available
for col in ["num_earners", "number_of_earners"]:
    if col in reg_data.columns:
        reg_data[col] = pd.to_numeric(
            reg_data[col], errors="coerce"
        )
        numeric_predictors.append(col)
        break

formula_terms = [
    'C(head_education_level, Treatment(reference="Secondary"))'
]
formula_terms += numeric_predictors
formula_terms += [f"C(Q('{col}'))" for col in categorical_predictors]

formula = "log_income_per_capita ~ " + " + ".join(formula_terms)

model_cols = [
    "log_income_per_capita",
    "head_education_level",
    *numeric_predictors,
    *categorical_predictors
]

reg_data = reg_data.dropna(subset=model_cols).copy()

# Fit statistical regression with robust standard errors
model = smf.ols(
    formula=formula,
    data=reg_data
).fit(cov_type="HC3")

print("Multiple Linear Regression — Statistical Inference")
print("=" * 60)
print(f"Observations: {int(model.nobs):,}")
print(f"R-squared: {model.rsquared:.4f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")

display(model.summary2().tables[1].round(4))


In [ ]:
# Use the same complete-case sample for both models
comparison_data = reg_data.copy()

# Model 1: Education only
model_1 = smf.ols(
    'log_income_per_capita ~ C(head_education_level, Treatment(reference="Secondary"))',
    data=comparison_data
).fit(cov_type="HC3")

# Model 2: Education + control variables
model_2 = model

# Extract the education coefficient
education_term = (
    'C(head_education_level, Treatment(reference="Secondary"))[T.None]'
)

results = pd.DataFrame({
    "Model": ["Model 1: Education Only", "Model 2: Adjusted"],
    "Education Coefficient": [
        model_1.params.get(education_term, np.nan),
        model_2.params.get(education_term, np.nan)
    ],
    "P-value": [
        model_1.pvalues.get(education_term, np.nan),
        model_2.pvalues.get(education_term, np.nan)
    ],
    "CI Lower": [
        model_1.conf_int().loc[education_term, 0],
        model_2.conf_int().loc[education_term, 0]
    ],
    "CI Upper": [
        model_1.conf_int().loc[education_term, 1],
        model_2.conf_int().loc[education_term, 1]
    ],
    "R-squared": [model_1.rsquared, model_2.rsquared],
    "Observations": [int(model_1.nobs), int(model_2.nobs)]
})

display(results.style.format({
    "Education Coefficient": "{:.4f}",
    "P-value": "{:.4e}",
    "CI Lower": "{:.4f}",
    "CI Upper": "{:.4f}",
    "R-squared": "{:.4f}",
    "Observations": "{:,.0f}"
}))


In [ ]:
# Use the same sample for all nested models
seq_data = reg_data.copy()

education_term = (
    'C(head_education_level, Treatment(reference="Secondary"))[T.None]'
)

# Define models by progressively adding controls
model_specs = {
    "M1: Education Only": [],
    "M2: + Age": ["head_age"],
    "M3: + Household Size": ["head_age", "household_size"],
    "M4: + Region": [
        "head_age", "household_size", "C(Q('macro_region'))"
    ],
    "M5: + Number of Earners": [
        "head_age", "household_size",
        "C(Q('macro_region'))", "num_earners"
    ]
}

sequential_results = []
sequential_models = {}

for name, controls in model_specs.items():

    # Skip a model if one of its required controls is unavailable
    required_controls = [
        c for c in controls
        if not c.startswith("C(")
    ]

    if any(c not in seq_data.columns for c in required_controls):
        print(f"Skipping {name}: required column unavailable.")
        continue

    formula = (
        'log_income_per_capita ~ '
        'C(head_education_level, Treatment(reference="Secondary"))'
    )

    if controls:
        formula += " + " + " + ".join(controls)

    fitted = smf.ols(
        formula=formula,
        data=seq_data
    ).fit(cov_type="HC3")

    sequential_models[name] = fitted

    ci = fitted.conf_int().loc[education_term]

    sequential_results.append({
        "Model": name,
        "Education Coefficient": fitted.params[education_term],
        "P-value": fitted.pvalues[education_term],
        "CI Lower": ci.iloc[0],
        "CI Upper": ci.iloc[1],
        "R-squared": fitted.rsquared,
        "Observations": int(fitted.nobs)
    })

sequential_table = pd.DataFrame(sequential_results)

display(
    sequential_table.style.format({
        "Education Coefficient": "{:.4f}",
        "P-value": "{:.4e}",
        "CI Lower": "{:.4f}",
        "CI Upper": "{:.4f}",
        "R-squared": "{:.4f}",
        "Observations": "{:,.0f}"
    })
)


In [ ]:
# Search for residence/location-related columns
for df_name, data in [("hh", hh), ("reg_data", reg_data)]:
    print(f"\n--- {df_name} ---")
    print([
        col for col in data.columns
        if any(word in col.lower() for word in [
            "urban", "rural", "resid", "region",
            "govern", "location", "place", "area"
        ])
    ])


In [ ]:
# Model 6: Add Urban/Rural Residence


urban_data = reg_data.copy()
urban_data["urban_rural"] = urban_data["urban_rural"].astype("category")

formula_urban = (
    'log_income_per_capita ~ '
    'C(head_education_level, Treatment(reference="Secondary"))'
    ' + head_age + household_size'
    ' + C(Q("macro_region")) + num_earners'
    ' + C(Q("urban_rural"))'
)

model_6 = smf.ols(
    formula=formula_urban,
    data=urban_data
).fit(cov_type="HC3")

term = (
    'C(head_education_level, Treatment(reference="Secondary"))[T.None]'
)

ci = model_6.conf_int().loc[term]

print("Model 6: Adding Urban/Rural Residence")
print("=" * 50)
print(f"Education coefficient: {model_6.params[term]:.4f}")
print(f"P-value: {model_6.pvalues[term]:.4e}")
print(f"95% CI: [{ci.iloc[0]:.4f}, {ci.iloc[1]:.4f}]")
print(f"R-squared: {model_6.rsquared:.4f}")
print(f"Observations: {int(model_6.nobs):,}")

display(model_6.summary2().tables[1].round(4))


### Statistical Insight: Why Does the “None” Group Have Higher Income per Capita?

The initial comparison showed that households in the **“None” education category** had a higher median income per capita than those in the **“Secondary” category**. The Mann–Whitney U test confirmed a statistically significant difference between the two income distributions (\(p < 0.001\)).

To investigate this pattern, sequential multiple regression models were estimated using log income per capita as the dependent variable. In the education-only model, the coefficient for the “None” category was positive (\(\beta = 0.1217\), \(p < 0.001\)). However, after controlling for household-head age, the coefficient became negative (\(\beta = -0.0989\), \(p < 0.001\)). Adding household size, macro-region, and number of earners further changed the coefficient, indicating that differences in household characteristics are strongly associated with the initial income pattern.

In the final model, which additionally controlled for urban/rural residence, the coefficient for the “None” category remained negative and statistically significant (\(\beta = -0.1513\), \(p < 0.001\); 95% CI: \(-0.1702\) to \(-0.1324\)). This corresponds to an adjusted income-per-capita difference of approximately **14.0% lower** for the “None” category relative to the “Secondary” category, conditional on the included variables.

Household-head age and the number of earners were positively associated with income per capita, whereas household size was negatively associated with it. Urban residence was also positively associated with income per capita relative to the reference residence category.

**Conclusion:** The higher median income per capita initially observed in the “None” group does not persist after adjusting for household characteristics. The reversal in the education coefficient suggests that differences in age and other household characteristics are important for understanding the observed pattern. These findings represent statistical associations rather than causal effects. The interpretation also depends on verifying the precise meaning of the “None” education category in the dataset documentation.


<div style="background-color:#000000;border:1px solid #4a3a10;border-top:3px solid #d4af37;border-radius:8px;padding:20px 28px 18px;margin:34px 0 10px;"><div style="font-family:Cinzel,Georgia,'Times New Roman',serif;font-size:11px;letter-spacing:.34em;color:#c9a24a;margin:0 0 8px;">&#9670;&nbsp; INCOME DISTRIBUTION INSIGHT</div><h1 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:30px;line-height:1.2;font-weight:700;color:#f3d77a;">What Are the Main Income Sources Among the Top 20% of Households?</h1><div style="height:1px;background-color:#8a6a1c;margin:14px 0 10px;"></div><div style="font-family:Manrope,'Segoe UI',Helvetica,Arial,sans-serif;font-size:13.5px;line-height:1.55;color:#a8a086;font-style:italic;">Examines the weighted distribution of primary income sources among households in the highest income group.</div></div>

In [ ]:
# ============================================================
# Main Income Sources in the Top 20% Income Group
# ============================================================

top_income = (
    hh.loc[
        hh["income_group"].astype(str).eq("Top 20%"),
        ["income_group", "income_source", "household_weight"]
    ]
    .copy()
)

top_income["household_weight"] = pd.to_numeric(
    top_income["household_weight"],
    errors="coerce"
)

top_income = top_income.dropna(
    subset=["income_source", "household_weight"]
)

top_income = top_income[
    (top_income["household_weight"] > 0)
    & (top_income["income_source"].astype(str).str.strip() != "")
].copy()

top_source_summary = (
    top_income
    .groupby("income_source", observed=True)["household_weight"]
    .sum()
    .reset_index(name="weighted_households")
    .sort_values("weighted_households", ascending=False)
)

total_weight = top_source_summary["weighted_households"].sum()

if total_weight <= 0:
    raise ValueError(
        "Total household weight is zero; shares cannot be calculated."
    )

top_source_summary["share_percent"] = (
    top_source_summary["weighted_households"]
    / total_weight
    * 100
)

top_source_summary["share_percent"] = (
    top_source_summary["share_percent"].round(1)
)

display(
    top_source_summary.style.format({
        "weighted_households": "{:,.2f}",
        "share_percent": "{:.1f}%"
    })
)

main_source = top_source_summary.iloc[0]

print(
    f"Main income source in the Top 20% group: "
    f"{main_source['income_source']} "
    f"({main_source['share_percent']:.1f}%)."
)

gold_palette = [
    "#F3D77A",
    "#D4AF37",
    "#B8902A",
    "#8A6A1C",
    "#6F5315",
    "#4E3A0D",
]

bar_colors = [
    gold_palette[i % len(gold_palette)]
    for i in range(len(top_source_summary))
]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=top_source_summary["share_percent"],
        y=top_source_summary["income_source"],
        orientation="h",
        marker=dict(
            color=bar_colors,
            line=dict(
                color="#F3D77A",
                width=1
            )
        ),
        text=[
            f"{value:.1f}%"
            for value in top_source_summary["share_percent"]
        ],
        textposition="outside",
        textfont=dict(
            family="Arial",
            size=13,
            color="#F3D77A"
        ),
        customdata=top_source_summary[
            ["weighted_households"]
        ],
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Weighted share: %{x:.1f}%<br>"
            "Weighted households: %{customdata[0]:,.0f}"
            "<extra></extra>"
        ),
        cliponaxis=False,
        showlegend=False
    )
)

fig.update_layout(
    title=dict(
        text="Main Income Sources among Top 20% Households",
        x=0.06,
        y=0.94,
        xanchor="left",
        font=dict(
            family="Georgia",
            size=25,
            color="#F3D77A"
        )
    ),
    paper_bgcolor="#0b0905",
    plot_bgcolor="#0b0905",
    font=dict(
        family="Arial",
        color="#F5EFE0",
        size=13
    ),
    width=1050,
    height=max(
        500,
        110 * len(top_source_summary) + 220
    ),
    margin=dict(
        l=250,
        r=120,
        t=150,
        b=95
    ),
    showlegend=False,
    bargap=0.35,
    xaxis=dict(
        title=dict(
            text="Weighted Share of Households (%)",
            font=dict(
                family="Arial",
                size=15,
                color="#F3D77A"
            )
        ),
        ticksuffix="%",
        range=[
            0,
            max(45, top_source_summary["share_percent"].max() * 1.25)
        ],
        showgrid=True,
        gridcolor="rgba(243, 215, 122, 0.16)",
        zeroline=False,
        showline=False,
        tickfont=dict(
            color="#A8A086",
            size=12
        )
    ),
    yaxis=dict(
        title=dict(
            text="Main Income Source",
            font=dict(
                family="Arial",
                size=15,
                color="#F3D77A"
            )
        ),
        categoryorder="total ascending",
        showgrid=False,
        tickfont=dict(
            color="#F5EFE0",
            size=13
        )
    )
)

try:
    fig.update_layout(barcornerradius=12)
except (TypeError, ValueError):
    pass

fig.show()

<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">9.1 Female Headship by Age</h2></div>

In [ ]:
# ============================================================
# Female Household Headship by Age
# Using cleaned and renamed columns
# ============================================================

# 1. Prepare the data

df_heads = df_merged[
    df_merged["relation_to_head"]
    .astype("string")
    .str.contains("Head", case=False, na=False)
].copy()

df_heads["is_female_head"] = (
    df_heads["gender"]
    .astype("string")
    .str.contains("Female", case=False, na=False)
)

df_heads["head_age"] = pd.to_numeric(
    df_heads["age"],
    errors="coerce"
)

df_heads["head_status"] = (
    df_heads["marital_status"]
    .astype("string")
)

W = "household_weight"

df_heads[W] = pd.to_numeric(
    df_heads[W],
    errors="coerce"
)

df_heads = df_heads.dropna(
    subset=["head_age", W]
)

df_heads = df_heads[
    df_heads[W].gt(0)
].copy()


# ============================================================
# 2. Helper functions
# ============================================================

def w_share(boolean_series, weights):
    boolean_series = boolean_series.astype(float)
    weights = pd.to_numeric(
        weights,
        errors="coerce"
    )

    valid = (
        boolean_series.notna()
        & weights.notna()
        & weights.gt(0)
    )

    if valid.sum() == 0:
        return np.nan

    return np.average(
        boolean_series[valid],
        weights=weights[valid]
    )


def n_eff(weights):
    weights = pd.to_numeric(
        weights,
        errors="coerce"
    )

    weights = weights[
        weights.notna()
        & weights.gt(0)
    ]

    if len(weights) == 0:
        return np.nan

    return (
        weights.sum() ** 2
    ) / (
        weights ** 2
    ).sum()


def wilson_ci(p, n, z=1.96):
    if pd.isna(p) or pd.isna(n) or n <= 0:
        return np.nan, np.nan

    denominator = 1 + z**2 / n

    center = (
        p + z**2 / (2 * n)
    ) / denominator

    spread = (
        z
        * np.sqrt(
            p * (1 - p) / n
            + z**2 / (4 * n**2)
        )
        / denominator
    )

    return (
        center - spread,
        center + spread
    )


# ============================================================
# 3. Create age groups
# ============================================================

BANDS = [
    0,
    30,
    40,
    50,
    60,
    70,
    200
]

BAND_LABELS = [
    "<30",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "70+"
]

df_heads["age_band"] = pd.cut(
    df_heads["head_age"],
    bins=BANDS,
    right=False,
    labels=BAND_LABELS
)


# ============================================================
# 4. Calculate summary tables
# ============================================================

rows = []
route_rows = []

for band, group in df_heads.groupby(
    "age_band",
    observed=True
):
    if group.empty:
        continue

    female_share = w_share(
        group["is_female_head"],
        group[W]
    )

    lower_ci, upper_ci = wilson_ci(
        female_share,
        n_eff(group[W])
    )

    rows.append(
        {
            "Head age": band,
            "Households (n)": len(group),
            "Female heads (n)": int(
                group["is_female_head"].sum()
            ),
            "% female-headed": female_share * 100,
            "CI low": lower_ci * 100,
            "CI high": upper_ci * 100,
        }
    )

    female_group = group[
        group["is_female_head"]
    ].copy()

    if not female_group.empty:
        status = (
            female_group["head_status"]
            .astype("string")
            .str.lower()
        )

        route_masks = {
            "Widowed": status.str.contains(
                "widowed",
                na=False
            ),
            "Divorced": status.str.contains(
                "divorc",
                na=False
            ),
            "Married": (
                status.str.contains(
                    "married",
                    na=False
                )
                & ~status.str.contains(
                    "never married",
                    na=False
                )
            ),
            "Never married": status.str.contains(
                "never married",
                na=False
            ),
        }

        route_row = {
            "Head age": band
        }

        for route_name, route_mask in route_masks.items():
            route_row[route_name] = (
                w_share(
                    route_mask,
                    female_group[W]
                )
                * 100
            )

        route_rows.append(route_row)


age_table = (
    pd.DataFrame(rows)
    .set_index("Head age")
)

route_by_age = (
    pd.DataFrame(route_rows)
    .set_index("Head age")
)


# ============================================================
# 5. Display summary tables
# ============================================================

display(
    age_table.round(2)
)

display(
    route_by_age.round(2)
)


# ============================================================
# 6. Plot styling
# ============================================================

BG_COLOR = "#0c0a00"
TEXT_COLOR = "#F4E0B9"
ACCENT_GOLD = "#E5B84B"

COLORS = {
    "Widowed": "#D4AF37",
    "Divorced": "#8C6718",
    "Married": "#F7E7CE",
    "Never married": "#594313",
}


# ============================================================
# 7. Create interactive subplots
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "<b>◆ Female headship profile by age</b>",
        "<b>◆ Route to female headship changes with age</b>",
    ),
    horizontal_spacing=0.15,
)


# Chart 1: Female headship by age

fig.add_trace(
    go.Scatter(
        x=age_table.index,
        y=age_table["% female-headed"],
        error_y={
            "type": "data",
            "symmetric": False,
            "array": (
                age_table["CI high"]
                - age_table["% female-headed"]
            ),
            "arrayminus": (
                age_table["% female-headed"]
                - age_table["CI low"]
            ),
            "color": ACCENT_GOLD,
            "thickness": 1.5,
            "width": 4,
        },
        mode="lines+markers+text",
        text=(
            age_table["% female-headed"]
            .round(0)
            .astype(str)
            + "%"
        ),
        textposition="top center",
        textfont={
            "color": TEXT_COLOR,
            "size": 11,
            "family": "Arial",
        },
        marker={
            "size": 8,
            "color": ACCENT_GOLD,
            "line": {
                "color": BG_COLOR,
                "width": 1.5,
            },
        },
        line={
            "color": ACCENT_GOLD,
            "width": 2.5,
        },
        name="Female Head %",
        hovertemplate=(
            "<b>Age Group:</b> %{x}"
            "<br><b>Female Headed:</b> %{y:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)


# Chart 2: Marital-status route

if not route_by_age.empty:
    for route_name in [
        "Widowed",
        "Divorced",
        "Married",
        "Never married",
    ]:
        if route_name in route_by_age.columns:
            fig.add_trace(
                go.Bar(
                    x=route_by_age.index,
                    y=route_by_age[route_name],
                    name=route_name,
                    marker={
                        "color": COLORS[route_name],
                        "line": {
                            "color": BG_COLOR,
                            "width": 1,
                        },
                    },
                    hovertemplate=(
                        "<b>Age Group:</b> %{x}"
                        f"<br><b>{route_name}:</b> "
                        "%{y:.1f}%"
                        "<extra></extra>"
                    ),
                ),
                row=1,
                col=2,
            )


# ============================================================
# 8. Layout
# ============================================================

fig.update_layout(
    barmode="stack",
    paper_bgcolor=BG_COLOR,
    plot_bgcolor=BG_COLOR,
    height=550,
    margin={
        "l": 60,
        "r": 40,
        "t": 80,
        "b": 80,
    },
    title={
        "text": (
            "<b>Analysis of Female Headship by Age</b>"
        ),
        "font": {
            "size": 16,
            "color": TEXT_COLOR,
        },
        "x": 0.05,
        "y": 0.95,
    },
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": -0.22,
        "xanchor": "center",
        "x": 0.5,
        "font": {
            "color": TEXT_COLOR,
            "size": 11,
        },
        "bgcolor": "rgba(0,0,0,0)",
    },
    hoverlabel={
        "bgcolor": "#1C1C1C",
        "font_color": TEXT_COLOR,
        "font_size": 12,
    },
)

# Subplot titles
for annotation in fig["layout"]["annotations"]:
    annotation["font"] = {
        "color": TEXT_COLOR,
        "size": 13,
        "family": "Arial",
    }

# X-axis
fig.update_xaxes(
    title_text="<b>Age of household head</b>",
    title_font={
        "color": TEXT_COLOR,
        "size": 12,
    },
    tickfont={
        "color": TEXT_COLOR,
    },
    gridcolor="rgba(244, 224, 185, 0.1)",
    showgrid=False,
)

# Y-axis
fig.update_yaxes(
    title_text="<b>Percentage of households (%)</b>",
    title_font={
        "color": TEXT_COLOR,
        "size": 12,
    },
    tickfont={
        "color": TEXT_COLOR,
    },
    gridcolor="rgba(244, 224, 185, 0.15)",
    showgrid=True,
    zeroline=False,
)

fig.show()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">9.2 Regional and Segment Descriptive Visualizations</h2></div>

In [ ]:
# ==============================================================================
# 1. Data Preparation
# ==============================================================================
region_col = 'region' if 'region' in hh.columns else 'governorate'
region_deprivation = pd.crosstab(hh[region_col], hh['housing_deprivation'], normalize='index') * 100

# Sort the governorates based on the worst condition category (the last column)
sort_category = region_deprivation.columns[-1]
region_deprivation = region_deprivation.sort_values(by=sort_category, ascending=True)

# ==============================================================================
# 2. Luxury Gold Gradient Color Palette for Dark Theme
# ==============================================================================
GOLD_COLORS = [
    '#F5EFE0', # Champagne Gold (Lightest)
    '#E8C35A', # Bright Gold
    '#D4AF37', # Classic Metallic Gold
    '#8f6f1f', # Deep Bronze Gold (Darkest)
]

# ==============================================================================
# 3. Create Interactive Plotly Horizontal Stacked Bar Chart
# ==============================================================================
fig = go.Figure()

for i, col in enumerate(region_deprivation.columns):
    color = GOLD_COLORS[i % len(GOLD_COLORS)]
    fig.add_trace(go.Bar(
        y=region_deprivation.index,
        x=region_deprivation[col],
        name=str(col),
        orientation='h',
        marker=dict(
            color=color,
            line=dict(color='#0b0905', width=1)
        ),
        text=region_deprivation[col].apply(lambda val: f"{val:.0f}%" if val >= 5 else ""),
        textposition='auto',
        textfont=dict(color='#0b0905' if i < 2 else '#F5EFE0', size=11, family='Arial Black'),
        hovertemplate=(
            f"<b>Region / Governorate:</b> %{{y}}<br>"
            f"<b>Category ({col}):</b> %{{x:.1f}}%<br>"
            "<extra></extra>"
        )
    ))

# ==============================================================================
# 4. Layout Customization & Dark Theme Setup (Increased height & spacing)
# ==============================================================================
fig.update_layout(
    barmode='stack',
    height=850,  # Increased graph height for better readability
    title=dict(
        text="<b>◆ Housing Deprivation Breakdown by Governorate (Interactive)</b>",
        font=dict(size=18, color='#F5EFE0', family='Arial'),
        x=0.0,
        y=0.97
    ),
    xaxis=dict(
        title=dict(
            text="<b>Percentage of Households (%)</b>",
            font=dict(color='#F5EFE0', size=13)
        ),
        tickfont=dict(color='#F5EFE0'),
        gridcolor='rgba(245, 239, 224, 0.15)',
        showgrid=True,
        zeroline=False
    ),
    yaxis=dict(
        title="",
        tickfont=dict(color='#F5EFE0', size=11),
        showgrid=False
    ),
    paper_bgcolor='#0b0905',
    plot_bgcolor='#0b0905',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.12,
        xanchor="center",
        x=0.5,
        font=dict(color='#F5EFE0', size=12),
        bgcolor='rgba(0,0,0,0)'
    ),
    margin=dict(l=120, r=40, t=80, b=100),
    hoverlabel=dict(
        bgcolor="#14100A",
        font_color="#F5EFE0",
        font_size=12
    )
)

fig.show()

# ==============================================================================
# 5. Display Summary Table
# ==============================================================================
print("\nRegional Housing Deprivation Details (% of Households):")
display(region_deprivation.round(1).sort_values(by=sort_category, ascending=False))


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">9.3 Household Segment Flow and Summary</h2></div>

In [ ]:
# Prepare an additional household analysis using existing merged and household data. Check inputs
# before calculating results.

# These analysis cells can be run in any order after the source-data preparation
# has created df_merged. Rebuild household-level inputs when they are absent.
if "hh" not in globals():
    if "df_merged" not in globals():
        raise RuntimeError("Run the data loading/cleaning/merge cells first; df_merged is not available.")
    hh = df_merged.drop_duplicates("household_id").set_index("household_id").copy()

_required_seg_cols = [
    "household_size", "num_under_14", "num_aged_65_plus", "num_earners",
    "head_education_level", "head_employment_status", "income_source",
    "head_gender", "urban_rural", "region", "asset_group",
    "housing_deprivation", "social_protection_group", "health_group"
]
_missing_seg_cols = [c for c in _required_seg_cols if c not in hh.columns]
if _missing_seg_cols:
    raise KeyError(f"Cannot build household segmentation; missing columns in hh: {_missing_seg_cols}. Run the feature-engineering cell first.")

# Always rebuild the small segmentation table from its source instead of relying
# on whether an earlier cell happened to have been executed.
seg = hh[_required_seg_cols].copy()
seg[ [c for c in _required_seg_cols if c not in ["household_size", "num_under_14", "num_aged_65_plus", "num_earners"]] ] = seg[
    [c for c in _required_seg_cols if c not in ["household_size", "num_under_14", "num_aged_65_plus", "num_earners"]]
].astype("category")
seg["dependency_ratio"] = (
    seg["num_under_14"] + seg["num_aged_65_plus"]
) / seg["household_size"].replace(0, np.nan)
seg["earners_ratio"] = seg["num_earners"] / (
    seg["household_size"] - seg["num_under_14"] - seg["num_aged_65_plus"]
).replace(0, np.nan)
seg["household_segment"] = np.select(
    [
        (seg["household_size"] >= 6) & (seg["dependency_ratio"] >= 1),
        (seg["household_size"] >= 6) & (seg["num_earners"] <= 1),
        (seg["dependency_ratio"] >= 1) & (seg["num_earners"] <= 1),
        (seg["num_earners"] >= 2) & (seg["dependency_ratio"] < 1),
    ],
    [
        "Large high-dependency households",
        "Large households with few earners",
        "High-dependency households with few earners",
        "Multiple-earner, lower-dependency households",
    ],
    default="Other household profiles"
)

d = hh.copy()
d["household_segment"] = seg["household_segment"]

# Chart 2 — Prepare weighted data for the Income Quintile Alluvial Plot
required_cols = [
    "head_education_level", "head_employment_status", "household_segment",
    "income_group", "household_weight"
]
alluvial_data = d.copy()
missing_cols = [c for c in required_cols if c not in alluvial_data.columns]
if missing_cols:
    raise KeyError(f"Alluvial plot cannot be built; missing columns: {missing_cols}")
alluvial_data = alluvial_data[required_cols].copy()
alluvial_data = alluvial_data[
    alluvial_data["household_weight"].notna()
    & (alluvial_data["household_weight"] > 0)
    & alluvial_data[required_cols[:-1]].notna().all(axis=1)
].copy()
print("Households available:", len(alluvial_data))
print("Total household weight:", round(alluvial_data["household_weight"].sum(), 2))
print("\nEducation categories:")
print(alluvial_data["head_education_level"].value_counts().to_string())
print("\nEmployment categories:")
print(alluvial_data["head_employment_status"].value_counts().to_string())
print("\nIncome quintiles:")
print(alluvial_data["income_group"].value_counts().to_string())


In [ ]:
# ==============================================================================
# Education and Household Income Analysis
# Descriptive Statistics — HIECS 2021
# ==============================================================================


edu = hh.copy()

# Use actual column names from the dataset
required = [
    "head_education_level",
    "income_per_capita",
    "income_total_disposable",
    "household_size",
    "household_weight",
    "head_age",
    "num_under_14"
]

missing = [c for c in required if c not in edu.columns]

if missing:
    print("Missing columns:", missing)
else:
    numeric_cols = [
        "income_per_capita",
        "income_total_disposable",
        "household_size",
        "household_weight",
        "head_age",
        "num_under_14"
    ]

    for col in numeric_cols:
        edu[col] = pd.to_numeric(edu[col], errors="coerce")

    # Keep valid household observations
    edu = edu[
        edu["head_education_level"].notna()
        & edu["income_per_capita"].notna()
        & edu["income_total_disposable"].notna()
        & edu["household_size"].gt(0)
        & edu["household_weight"].gt(0)
    ].copy()

    # Descriptive statistics by education level
    summary = (
        edu.groupby("head_education_level", observed=True)
        .agg(
            Households=("household_size", "size"),
            Median_Income_PC=("income_per_capita", "median"),
            Mean_Income_PC=("income_per_capita", "mean"),
            Median_Household_Income=(
                "income_total_disposable", "median"
            ),
            Mean_Household_Income=(
                "income_total_disposable", "mean"
            ),
            Mean_Household_Size=("household_size", "mean"),
            Mean_Head_Age=("head_age", "mean"),
            Mean_Children_Under_14=("num_under_14", "mean")
        )
        .round(2)
        .reset_index()
    )

    display(summary)


In [ ]:
comparison = edu[
    edu["head_education_level"].isin(["None", "Secondary"])
    & edu["income_per_capita"].notna()
].copy()

none_income = comparison.loc[
    comparison["head_education_level"] == "None",
    "income_per_capita"
]

secondary_income = comparison.loc[
    comparison["head_education_level"] == "Secondary",
    "income_per_capita"
]

result = mannwhitneyu(
    none_income,
    secondary_income,
    alternative="two-sided"
)

print("None group n:", len(none_income))
print("Secondary group n:", len(secondary_income))
print(f"Mann–Whitney U statistic: {result.statistic:.2f}")
print(f"P-value: {result.pvalue:.6g}")


In [ ]:
comparison = edu[
    edu["head_education_level"].isin(["None", "Secondary"])
].copy()

# Compare household size between education groups
none_size = comparison.loc[
    comparison["head_education_level"] == "None",
    "household_size"
].dropna()

secondary_size = comparison.loc[
    comparison["head_education_level"] == "Secondary",
    "household_size"
].dropna()

size_test = mannwhitneyu(
    none_size,
    secondary_size,
    alternative="two-sided"
)

print("=== Household Size Comparison ===")
print(f"None median household size: {none_size.median():.2f}")
print(f"Secondary median household size: {secondary_size.median():.2f}")
print(f"Mann–Whitney p-value: {size_test.pvalue:.6g}")

# Association between household size and income per capita
valid = edu[
    edu["household_size"].notna()
    & edu["income_per_capita"].notna()
]

rho, p_value = spearmanr(
    valid["household_size"],
    valid["income_per_capita"]
)

print("\n=== Spearman Correlation ===")
print(f"Spearman rho: {rho:.4f}")
print(f"P-value: {p_value:.6g}")


In [ ]:
comparison = edu[
    edu["head_education_level"].isin(["None", "Secondary"])
].copy()

# Compare head age between education groups
none_age = comparison.loc[
    comparison["head_education_level"] == "None",
    "head_age"
].dropna()

secondary_age = comparison.loc[
    comparison["head_education_level"] == "Secondary",
    "head_age"
].dropna()

age_test = mannwhitneyu(
    none_age,
    secondary_age,
    alternative="two-sided"
)

print("=== Head Age Comparison ===")
print(f"None mean age: {none_age.mean():.2f}")
print(f"Secondary mean age: {secondary_age.mean():.2f}")
print(f"Mann–Whitney p-value: {age_test.pvalue:.6g}")

# Association between age and income per capita
valid = edu[
    edu["head_age"].notna()
    & edu["income_per_capita"].notna()
]

rho_age, p_age = spearmanr(
    valid["head_age"],
    valid["income_per_capita"]
)

print("\n=== Spearman Correlation ===")
print(f"Age vs. income per capita: {rho_age:.4f}")
print(f"P-value: {p_age:.6g}")


In [ ]:
# Keep only the two education groups being investigated
comparison = edu.loc[
    edu["head_education_level"].isin(["None", "Secondary"])
    & edu["head_age"].notna()
    & edu["income_per_capita"].notna()
].copy()

# Create age groups
comparison["age_band"] = pd.cut(
    comparison["head_age"],
    bins=[0, 30, 40, 50, 60, 70, float("inf")],
    labels=["<30", "30–39", "40–49", "50–59", "60–69", "70+"],
    right=False
)

results = []

for age_band, group in comparison.groupby("age_band", observed=True):

    none_income = group.loc[
        group["head_education_level"] == "None",
        "income_per_capita"
    ]

    secondary_income = group.loc[
        group["head_education_level"] == "Secondary",
        "income_per_capita"
    ]

    # Skip groups with insufficient observations
    if len(none_income) < 2 or len(secondary_income) < 2:
        continue

    test = mannwhitneyu(
        none_income,
        secondary_income,
        alternative="two-sided"
    )

    results.append({
        "Age Band": str(age_band),
        "None (n)": len(none_income),
        "Secondary (n)": len(secondary_income),
        "None Median Income PC": none_income.median(),
        "Secondary Median Income PC": secondary_income.median(),
        "Median Difference": (
            none_income.median() - secondary_income.median()
        ),
        "Mann-Whitney p-value": test.pvalue
    })

age_education_comparison = pd.DataFrame(results)

display(age_education_comparison.round(4))


In [ ]:
# Keep only the two education groups
comparison = edu.loc[
    edu["head_education_level"].isin(["None", "Secondary"])
    & edu["head_age"].notna()
    & edu["income_per_capita"].notna()
].copy()

# Age bands
comparison["age_band"] = pd.cut(
    comparison["head_age"],
    bins=[0, 30, 40, 50, 60, 70, float("inf")],
    labels=["<30", "30–39", "40–49", "50–59", "60–69", "70+"],
    right=False
)

# Overall comparison
overall = comparison.groupby(
    "head_education_level"
)["income_per_capita"].agg(
    n="count",
    median="median",
    mean="mean"
)

print("OVERALL COMPARISON")
display(overall.round(2))

# Comparison within each age band
results = []

for age_band, group in comparison.groupby("age_band", observed=True):
    none = group.loc[
        group["head_education_level"] == "None",
        "income_per_capita"
    ]

    secondary = group.loc[
        group["head_education_level"] == "Secondary",
        "income_per_capita"
    ]

    if len(none) == 0 or len(secondary) == 0:
        continue

    p_value = (
        mannwhitneyu(none, secondary, alternative="two-sided").pvalue
        if len(none) >= 2 and len(secondary) >= 2
        else float("nan")
    )

    results.append({
        "Age Band": str(age_band),
        "None n": len(none),
        "Secondary n": len(secondary),
        "None Median": none.median(),
        "Secondary Median": secondary.median(),
        "Median Difference (None - Secondary)": (
            none.median() - secondary.median()
        ),
        "P-value": p_value
    })

age_results = pd.DataFrame(results)

print("\nAGE-STRATIFIED COMPARISON")
display(age_results.round(4))


### Education and Household Income: An Unexpected Pattern

**Research Question:** Why does the group classified as having no education (`None`) have a higher median income per capita than the secondary education group (`Secondary`) in the HIECS 2021 dataset?

**Key Findings:** The median income per capita was EGP 19,965.83 for the `None` group, compared with EGP 16,720.50 for the `Secondary` group. The Mann–Whitney U test indicated a statistically significant difference between their income distributions (p < 0.001). The `None` group also had a smaller median household size (3 versus 4 persons) and an older household head on average (56.12 versus 43.91 years). Household size was negatively associated with income per capita (Spearman’s ρ = −0.6008), while household-head age was positively associated with it (ρ = 0.4157).

**Interpretation:** The aggregate difference may be related to differences in household composition and demographic characteristics. However, the age-stratified comparisons showed that the income pattern varied across age groups. The available results do not establish causality or conclusively demonstrate Simpson’s paradox. Further interpretation should also account for the survey design and household weights.

**Statistical Note:** The reported descriptive statistics, correlations, and hypothesis tests are unweighted; therefore, they describe the analyzed sample and should not automatically be interpreted as nationally representative estimates.


In [ ]:
# ==============================================================================
# 1. Data Pipeline & Segmentation Processing
# ==============================================================================
if "hh" not in globals():
    if "df_merged" not in globals():
        raise RuntimeError("Run the data loading/cleaning/merge cells first; df_merged is not available.")
    hh = df_merged.drop_duplicates("household_id").set_index("household_id").copy()

_required_seg_cols = [
    "household_size", "num_under_14", "num_aged_65_plus", "num_earners",
    "head_education_level", "head_employment_status", "income_source",
    "head_gender", "urban_rural", "region", "asset_group",
    "housing_deprivation", "social_protection_group", "health_group"
]

seg = hh[_required_seg_cols].copy()
seg["dependency_ratio"] = (seg["num_under_14"] + seg["num_aged_65_plus"]) / seg["household_size"].replace(0, np.nan)
seg["household_segment"] = np.select(
    [
        (seg["household_size"] >= 6) & (seg["dependency_ratio"] >= 1),
        (seg["household_size"] >= 6) & (seg["num_earners"] <= 1),
        (seg["dependency_ratio"] >= 1) & (seg["num_earners"] <= 1),
        (seg["num_earners"] >= 2) & (seg["dependency_ratio"] < 1),
    ],
    [
        "Large high-dependency households",
        "Large households with few earners",
        "High-dependency households with few earners",
        "Multiple-earner, lower-dependency households",
    ],
    default="Other household profiles"
)

d = hh.copy()
d["household_segment"] = seg["household_segment"]

# Build Aggregated Household Flows
required_cols = ["head_education_level", "head_employment_status", "household_segment", "income_group", "household_weight"]
alluvial_data = d[required_cols].dropna().copy()
alluvial_data = alluvial_data[alluvial_data["household_weight"] > 0]

flows = (
    alluvial_data.groupby(required_cols[:-1], observed=True)["household_weight"]
    .sum()
    .reset_index(name="Weighted households")
)

# Calculate Share (%)
total_weighted_hh = flows["Weighted households"].sum()
flows["Share (%)"] = (flows["Weighted households"] / total_weighted_hh * 100).round(2)

# Top 10 Flows Table
top_flows = flows.sort_values("Weighted households", ascending=False).head(10).reset_index(drop=True)

# ==============================================================================
# 2. Key Insights & Policy Recommendations Extraction
# ==============================================================================
bottom_20_flows = flows[flows["income_group"] == "Bottom 20%"]
top_vulnerable_segment = bottom_20_flows.groupby("household_segment", observed=True)["Weighted households"].sum().idxmax()
top_vulnerable_share = (bottom_20_flows.groupby("household_segment", observed=True)["Weighted households"].sum().max() / bottom_20_flows["Weighted households"].sum() * 100).round(1)

top_earner_segment = flows[flows["income_group"] == "Top 20%"].groupby("household_segment", observed=True)["Weighted households"].sum().idxmax()

# Display Top Flows Dataframe
print("◆ Top 10 Dominant Household Pathway Flows in Dataset:")
display(top_flows)

# ==============================================================================
# 3. HTML Executive Summary Box (Dashboard Style)
# ==============================================================================
summary_html = f"""
<div style="background-color: #0b0905; border: 1.5px solid #D4AF37; padding: 20px; border-radius: 8px; color: #F5EFE0; font-family: Arial, sans-serif;">
    <h3 style="color: #F3D77A; margin-top: 0; font-size: 18px;"><b>Executive Analytical Insights & Actionable Recommendations</b></h3>

    <p style="font-size: 14px; margin-bottom: 8px;"><b>◆ Key Observations (Insights):</b></p>
    <ul style="font-size: 13px; line-height: 1.6; padding-left: 20px; color: #D4AF37;">
        <li><b>Poverty Vulnerability Cluster:</b> Households belonging to <i>"{top_vulnerable_segment}"</i> represent <b>{top_vulnerable_share}%</b> of all households residing in the <b>Bottom 20% income quintile</b>. High dependency combined with low earner capacity is the primary driver of income deprivation.</li>
        <li><b>Economic Mobility Driver:</b> The <i>"{top_earner_segment}"</i> segment exhibits the highest representation in the upper income brackets (Top 20%), proving that multiple earners per household significantly mitigate dependency risks.</li>
        <li><b>Education-Employment Link:</b> Head of household education levels combined with employment stability serve as the foundational bottleneck dictating progression into higher income groups.</li>
    </ul>

    <p style="font-size: 14px; margin-bottom: 8px; margin-top: 15px;"><b>◆ Strategic Recommendations:</b></p>
    <ol style="font-size: 13px; line-height: 1.6; padding-left: 20px; color: #F3D77A;">
        <li><b>Targeted Social Protection:</b> Prioritize direct cash transfers and welfare programs towards <i>High-dependency households with few earners</i>, specifically focusing on child allowances and elderly support to reduce the dependency burden ratio.</li>
        <li><b>Labor Force Participation:</b> Implement localized female and youth vocational training programs to transition single-earner households into <i>Multiple-earner households</i>.</li>
        <li><b>Housing & Infrastructure Intervention:</b> Align housing infrastructure investments with regions showing heavy concentration of vulnerable segments to address multidimensional poverty simultaneously.</li>
    </ol>
</div>
"""

display(HTML(summary_html))


In [ ]:
# ==============================================================================
# ==============================================================================
simple_flows = (
    d.groupby(["household_segment", "income_group"], observed=True)["household_weight"]
    .sum()
    .reset_index(name="value")
)

segments = sorted(simple_flows["household_segment"].unique().tolist())
incomes = [q for q in ["Bottom 20%", "Lower-middle 20%", "Middle 20%", "Upper-middle 20%", "Top 20%"]
           if q in simple_flows["income_group"].unique()]

labels = segments + incomes

node_map = {label: idx for idx, label in enumerate(labels)}

source = [node_map[src] for src in simple_flows["household_segment"]]
target = [node_map[tgt] for tgt in simple_flows["income_group"]]
value = simple_flows["value"].tolist()

# ==============================================================================
# ==============================================================================
BG_COLOR = "#0b0905"
TEXT_COLOR = "#F5EFE0"

node_colors = ['#F3D77A', '#B8902A', '#8A6A1C', '#D4AF37', '#A07F22'] * 2

link_color = "rgba(184, 144, 42, 0.25)"

# ==============================================================================
# ==============================================================================
fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        label=labels,
        pad=20,
        thickness=20,
        color=node_colors[:len(labels)],
        line=dict(color=BG_COLOR, width=1)
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color=link_color,
        hovertemplate="%{source.label} → %{target.label}<br>Weighted households: %{value:,.0f}<extra></extra>"
    )
))

fig.update_layout(
    title=dict(
        text="<b>◆ Household Segment Flow to Income Quintile</b>",
        font=dict(size=16, color=TEXT_COLOR, family="Arial"),
        x=0.0,
        y=0.96
    ),
    paper_bgcolor=BG_COLOR,
    plot_bgcolor=BG_COLOR,
    font=dict(color=TEXT_COLOR, size=12, family="Arial"),
    height=550,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.show()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">9.4 Labor Force and Household Segment Comparisons</h2></div>

In [ ]:
# ==============================================================================
# ==============================================================================
reason_col = None
possible_cols = ['reason_not_working', 'unemployment_reason', 'not_working_reason', 'head_employment_status']

for col in possible_cols:
    if col in df_merged.columns:
        reason_col = col
        break

if reason_col is None:
    print("No specific non-working reason column was found. Please verify the column name in df_merged.")
else:
    non_working_df = df_merged[df_merged[reason_col].notna()].copy()

    if 'person_weight' in non_working_df.columns:
        reason_counts = non_working_df.groupby(reason_col, observed=True)['person_weight'].sum().sort_values(ascending=True)
    else:
        reason_counts = non_working_df[reason_col].value_counts(ascending=True)

    reason_percentages = (reason_counts / reason_counts.sum()) * 100

    # ==============================================================================
    # ==============================================================================
    BG_COLOR = "#0b0905"
    TEXT_COLOR = "#F5EFE0"
    ACCENT_GOLD = "#D4AF37"
    BAR_COLORS = ['#4A3A10', '#6B5416', '#8A6A1C', '#B8902A', '#F3D77A']

    fig, ax = plt.subplots(figsize=(10, 5.5))
    fig.patch.set_facecolor(BG_COLOR)
    ax.set_facecolor(BG_COLOR)

    colors_to_use = BAR_COLORS[-len(reason_counts):]
    bars = ax.barh(reason_counts.index.astype(str), reason_percentages.values,
                   color=colors_to_use, edgecolor=ACCENT_GOLD, height=0.55, linewidth=1.2)

    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width + 0.8, bar.get_y() + bar.get_height()/2, f"{width:.1f}%",
                ha='left', va='center', color=TEXT_COLOR, fontsize=11, fontweight='bold')

    ax.set_title("◆ Reasons for Not Working / Unemployment Breakdown",
                 loc="left", color=TEXT_COLOR, weight="bold", fontsize=14, pad=20)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_color(ACCENT_GOLD)
    ax.spines['left'].set_linewidth(1.5)

    ax.set_xticks([])
    ax.tick_params(axis='y', colors=TEXT_COLOR, labelsize=11, length=0)

    plt.tight_layout()
    plt.show()

    summary_reasons = pd.DataFrame({
        'Weighted Total': reason_counts[::-1].round(0),
        'Share (%)': reason_percentages[::-1].round(1)
    })
    display(summary_reasons)


In [ ]:
# ==============================================================================
# ==============================================================================
plot_df = pd.crosstab(
    d['household_segment'],
    d['income_group'],
    values=d['household_weight'],
    aggfunc='sum',
    normalize='index'
) * 100

quintile_order = ["Bottom 20%", "Lower-middle 20%", "Middle 20%", "Upper-middle 20%", "Top 20%"]
plot_df = plot_df[quintile_order]

# ==============================================================================
# ==============================================================================
BG_COLOR = "#0b0905"
TEXT_COLOR = "#F5EFE0"
ACCENT_GOLD = "#D4AF37"

INCOME_COLORS = ['#4A3A10', '#6B5416', '#8A6A1C', '#B8902A', '#F3D77A']

# ==============================================================================
# ==============================================================================
fig = go.Figure()

for i, col in enumerate(plot_df.columns):
    values = plot_df[col].values

    text_labels = [f"{val:.0f}%" if val >= 6.0 else "" for val in values]
    text_colors = "#000000" if i >= 3 else "#F5EFE0"

    fig.add_trace(go.Bar(
        y=plot_df.index,
        x=values,
        name=col,
        orientation='h',
        marker=dict(
            color=INCOME_COLORS[i],
            line=dict(color=BG_COLOR, width=1.5)
        ),
        text=text_labels,
        textposition='inside',
        insidetextanchor='middle',
        textfont=dict(color=text_colors, size=11, family="Arial"),
        hovertemplate=(
            "<b>Segment:</b> %{y}<br>"
            "<b>Income Group:</b> " + col + "<br>"
            "<b>Weighted Share:</b> %{x:.1f}%"
            "<extra></extra>"
        )
    ))

# ==============================================================================
# ==============================================================================
fig.update_layout(
    title=dict(
        text="<b>◆ Impact of Dependency & Earners Ratio on Household Income Quintiles</b>",
        font=dict(size=16, color=TEXT_COLOR, family="Arial"),
        x=0.0,
        y=0.96
    ),
    barmode='stack',
    paper_bgcolor=BG_COLOR,
    plot_bgcolor=BG_COLOR,
    height=550,
    margin=dict(l=20, r=20, t=70, b=80),
    xaxis=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[0, 100]
    ),
    yaxis=dict(
        tickfont=dict(color=TEXT_COLOR, size=11, family="Arial"),
        showgrid=False,
        linecolor=ACCENT_GOLD,
        linewidth=1.5
    ),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.12,
        xanchor="center",
        x=0.5,
        font=dict(color=TEXT_COLOR, size=11, family="Arial")
    ),
    hoverlabel=dict(
        bgcolor="#14100A",
        font_color=TEXT_COLOR,
        font_size=12,
        font_family="Arial"
    )
)

fig.show()

# ==============================================================================
# ==============================================================================
recommendations_html = (
    '<div style="background-color: #0b0905; border: 1.5px solid #D4AF37; padding: 20px; border-radius: 8px; color: #F5EFE0; font-family: Arial, sans-serif; margin-top: 15px;">'
    '<h4 style="color: #F3D77A; margin-top: 0; font-size: 16px;"><b>◆ Key Takeaways & Strategic Insights:</b></h4>'
    '<p style="font-size: 14px; margin-bottom: 6px; color: #D4AF37;"><b>1. Identifying Vulnerable Households (Poverty Drivers):</b></p>'
    '<ul style="font-size: 13px; line-height: 1.6; padding-left: 20px; color: #F5EFE0; margin-top: 0; margin-bottom: 12px;">'
    '<li>Shows which segments mostly fall into the <b>Bottom 20%</b> and <b>Lower-middle 20%</b> (the poorest 40% of the population).</li>'
    '<li>Households labeled as <i>"High-dependency households with few earners"</i> have a huge share in the Bottom 20%. This proves that high dependency (many children/elderly) combined with few working members is the main driver of poverty.</li>'
    '</ul>'
    '<p style="font-size: 14px; margin-bottom: 6px; color: #D4AF37;"><b>2. Drivers of Economic Mobility (Pathways out of Poverty):</b></p>'
    '<ul style="font-size: 13px; line-height: 1.6; padding-left: 20px; color: #F5EFE0; margin-top: 0; margin-bottom: 0;">'
    '<li>Highlights which segments successfully reach the <b>Top 20%</b> income group.</li>'
    '<li>Households labeled as <i>"Multiple-earner, lower-dependency households"</i> hold the largest share in top income brackets. This proves that having multiple earners per household is the most powerful tool to escape poverty.</li>'
    '</ul>'
    '</div>'
)

display(HTML(recommendations_html))


In [ ]:
# ============================================================
# Access to Services by Household Segment
# Data-driven chart with readable labels
# ============================================================

# ------------------------------------------------------------
# 1. Prepare household-level data
# ------------------------------------------------------------

services_data = hh.copy()

if "household_segment" in seg.columns:
    services_data["household_segment"] = (
        seg["household_segment"]
    )
elif "household_segment" in hh.columns:
    services_data["household_segment"] = (
        hh["household_segment"]
    )
else:
    raise KeyError(
        "household_segment was not found. "
        "Run the segmentation cells first."
    )

required_columns = [
    "household_weight",
    "has_electricity",
    "water_facility",
    "has_internet",
    "household_segment",
]

missing_columns = [
    col for col in required_columns
    if col not in services_data.columns
]

if missing_columns:
    raise KeyError(
        f"Missing columns: {missing_columns}"
    )


# ------------------------------------------------------------
# 2. Standardize service indicators
# ------------------------------------------------------------

def service_flag(series, positive_values):
    normalized_values = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    positive_values = [
        str(value).lower()
        for value in positive_values
    ]

    return normalized_values.isin(
        positive_values
    )


services_data["electricity_flag"] = service_flag(
    services_data["has_electricity"],
    [
        "Electrified",
        "Yes",
        "True",
        "1",
    ]
)

services_data["piped_water_flag"] = service_flag(
    services_data["water_facility"],
    [
        "Piped supply",
    ]
)

services_data["internet_flag"] = service_flag(
    services_data["has_internet"],
    [
        "Own one or many",
        "Yes",
        "True",
        "1",
    ]
)

services_data["household_weight"] = pd.to_numeric(
    services_data["household_weight"],
    errors="coerce"
)

services_data = services_data[
    services_data["household_weight"].notna()
    & services_data["household_weight"].gt(0)
].copy()


# ------------------------------------------------------------
# 3. Weighted percentage function
# ------------------------------------------------------------

def weighted_percentage(flag, weights):
    flag = flag.astype(float)
    weights = pd.to_numeric(
        weights,
        errors="coerce"
    )

    valid = (
        flag.notna()
        & weights.notna()
        & weights.gt(0)
    )

    if valid.sum() == 0:
        return np.nan

    return np.average(
        flag[valid],
        weights=weights[valid]
    ) * 100


# ------------------------------------------------------------
# 4. Calculate service percentages by segment
# ------------------------------------------------------------

segment_order = [
    "Multiple-earner, lower-dependency households",
    "Large households with few earners",
    "High-dependency households with few earners",
    "Other household profiles",
]

service_rows = []

for segment_name in segment_order:
    group = services_data[
        services_data["household_segment"]
        == segment_name
    ]

    if group.empty:
        continue

    service_rows.append(
        {
            "Segment": segment_name,
            "Electricity": weighted_percentage(
                group["electricity_flag"],
                group["household_weight"]
            ),
            "Piped water": weighted_percentage(
                group["piped_water_flag"],
                group["household_weight"]
            ),
            "Internet access": weighted_percentage(
                group["internet_flag"],
                group["household_weight"]
            ),
        }
    )

services_summary = (
    pd.DataFrame(service_rows)
    .set_index("Segment")
    .round(2)
)

display(services_summary)


# ------------------------------------------------------------
# 5. Short labels for the chart
# ------------------------------------------------------------

short_labels = {
    "Multiple-earner, lower-dependency households":
        "Multiple earners\nLow dependency",

    "Large households with few earners":
        "Large household\nFew earners",

    "High-dependency households with few earners":
        "High dependency\nFew earners",

    "Other household profiles":
        "Other profiles",
}

chart_x_labels = [
    short_labels.get(
        str(segment),
        str(segment)
    )
    for segment in services_summary.index
]

x_positions = list(
    range(len(services_summary.index))
)


# ------------------------------------------------------------
# 6. Create the chart
# ------------------------------------------------------------

fig_services = go.Figure()

fig_services.add_trace(
    go.Bar(
        name="% with Electricity",
        x=x_positions,
        y=services_summary["Electricity"],
        marker_color="#E5C158",
        text=[
            f"{value:.1f}%"
            for value in services_summary["Electricity"]
        ],
        textposition="auto",
        hovertemplate=(
            "<b>Segment:</b> %{customdata}"
            "<br><b>Electricity:</b> %{y:.1f}%"
            "<extra></extra>"
        ),
        customdata=chart_x_labels,
    )
)

fig_services.add_trace(
    go.Bar(
        name="% with Piped Water",
        x=x_positions,
        y=services_summary["Piped water"],
        marker_color="#C59B27",
        text=[
            f"{value:.1f}%"
            for value in services_summary["Piped water"]
        ],
        textposition="auto",
        hovertemplate=(
            "<b>Segment:</b> %{customdata}"
            "<br><b>Piped Water:</b> %{y:.1f}%"
            "<extra></extra>"
        ),
        customdata=chart_x_labels,
    )
)

fig_services.add_trace(
    go.Bar(
        name="% with Internet Access",
        x=x_positions,
        y=services_summary["Internet access"],
        marker_color="#D4AF37",
        text=[
            f"{value:.1f}%"
            for value in services_summary["Internet access"]
        ],
        textposition="auto",
        hovertemplate=(
            "<b>Segment:</b> %{customdata}"
            "<br><b>Internet Access:</b> %{y:.1f}%"
            "<extra></extra>"
        ),
        customdata=chart_x_labels,
    )
)


# ------------------------------------------------------------
# 7. Improve chart layout
# ------------------------------------------------------------

fig_services.update_layout(
    barmode="group",

    width=1500,
    height=700,

    title=dict(
        text=(
            "<b>◆ Access to Services "
            "by Household Segment</b>"
        ),
        font=dict(
            color="#F4E0B9",
            size=18,
            family="Arial"
        ),
        x=0.02,
        y=0.96,
        xanchor="left",
        yanchor="top",
    ),

    paper_bgcolor="#0c0a00",
    plot_bgcolor="#0c0a00",

    font=dict(
        color="#F4E0B9",
        family="Arial"
    ),

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(
            color="#F4E0B9",
            size=12
        ),
        bgcolor="rgba(0,0,0,0)"
    ),

    xaxis=dict(
        tickmode="array",
        tickvals=x_positions,
        ticktext=chart_x_labels,
        tickangle=0,
        tickfont=dict(
            color="#F4E0B9",
            size=12
        ),
        automargin=True,
        showgrid=False,
        zeroline=False,
    ),

    yaxis=dict(
        title=dict(
            text="<b>Percentage (%)</b>",
            font=dict(
                color="#F4E0B9",
                size=13
            )
        ),
        tickfont=dict(
            color="#F4E0B9",
            size=11
        ),
        gridcolor="rgba(244, 224, 185, 0.15)",
        showgrid=True,
        zeroline=False,
        range=[0, 105],
    ),

    margin=dict(
        l=80,
        r=50,
        t=140,
        b=100,
    ),

    hoverlabel=dict(
        bgcolor="#1C1C1C",
        font_color="#F4E0B9",
        font_size=12,
    ),
    barcornerradius=12,
)

fig_services.show()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">9.5 Household Profile Distributions</h2></div>

In [ ]:
# ==============================================================================
# ==============================================================================
urban_rural = hh["urban_rural"].value_counts(dropna=True)

BG_COLOR = "#0b0905"       # Dark panel background
TEXT_COLOR = "#F5EFE0"     # Light text color
ACCENT_GOLD = "#D4AF37"    # Accent gold for borders

PIE_COLORS = ['#F3D77A', '#8A6A1C']

fig, ax = plt.subplots(figsize=(7, 7))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

# ==============================================================================
# ==============================================================================
wedges, texts, autotexts = ax.pie(
    urban_rural.values,
    labels=urban_rural.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=PIE_COLORS,
    wedgeprops=dict(width=0.6, edgecolor=BG_COLOR, linewidth=2) # Modern donut styling
)

for text in texts:
    text.set_color(TEXT_COLOR)
    text.set_fontsize(12)
    text.set_weight("bold")

for autotext in autotexts:
    autotext.set_color("#000000") # Black labels on gold sectors
    autotext.set_fontsize(11)
    autotext.set_weight("bold")

# ==============================================================================
# ==============================================================================
ax.set_title("◆ Urban vs Rural Distribution", color=TEXT_COLOR, weight="bold", fontsize=15, pad=20)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# ==============================================================================
gender = hh["head_gender"].value_counts(dropna=True)

BG_COLOR = "#0b0905"       # Dark panel background
TEXT_COLOR = "#F5EFE0"     # Light text color

PIE_COLORS = ['#F3D77A', '#8A6A1C']

fig, ax = plt.subplots(figsize=(7, 7))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

# ==============================================================================
# ==============================================================================
wedges, texts, autotexts = ax.pie(
    gender.values,
    labels=gender.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=PIE_COLORS,
    wedgeprops=dict(width=0.6, edgecolor=BG_COLOR, linewidth=2) # Modern donut styling
)

for text in texts:
    text.set_color(TEXT_COLOR)
    text.set_fontsize(12)
    text.set_weight("bold")

for autotext in autotexts:
    autotext.set_color("#000000") # Black labels on gold sectors
    autotext.set_fontsize(11)
    autotext.set_weight("bold")

# ==============================================================================
# ==============================================================================
ax.set_title("◆ Gender Distribution of Household Heads", color=TEXT_COLOR, weight="bold", fontsize=15, pad=20)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# ==============================================================================
internet = hh["has_internet"].copy()

internet = internet.map({
    "Own one or many": "Yes",
    1: "Yes",
    0: "No"
})

internet = internet.fillna("No")
internet_counts = internet.value_counts()

BG_COLOR = "#0b0905"       # Dark panel background
TEXT_COLOR = "#F5EFE0"     # Light text color

PIE_COLORS = ['#F3D77A', '#8A6A1C']

fig, ax = plt.subplots(figsize=(7, 7))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

# ==============================================================================
# ==============================================================================
wedges, texts, autotexts = ax.pie(
    internet_counts.values,
    labels=internet_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=PIE_COLORS,
    wedgeprops=dict(width=0.6, edgecolor=BG_COLOR, linewidth=2) # Modern donut styling
)

for text in texts:
    text.set_color(TEXT_COLOR)
    text.set_fontsize(12)
    text.set_weight("bold")

for autotext in autotexts:
    autotext.set_color("#000000") # Black labels on gold sectors
    autotext.set_fontsize(11)
    autotext.set_weight("bold")

# ==============================================================================
# ==============================================================================
ax.set_title("◆ Household Internet Access Distribution", color=TEXT_COLOR, weight="bold", fontsize=15, pad=20)

plt.tight_layout()
plt.show()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">9.6 Budget Composition by Income Quintile</h2></div>

In [ ]:
# ==============================================================================
# ==============================================================================
expense_categories = {
    "exp_food_beverages": "Food & Beverages",
    "exp_alcohol_tobacco": "Alcohol & Tobacco",
    "exp_clothing_footwear": "Clothing & Footwear",
    "exp_housing_utilities": "Housing & Utilities",
    "exp_furniture_equipment": "Furniture & Equipment",
    "exp_health": "Health",
    "exp_transport": "Transport",
    "exp_communication": "Communication",
    "exp_recreation_culture": "Recreation & Culture",
    "exp_education": "Education",
    "exp_restaurants_hotels": "Restaurants & Hotels",
    "exp_misc_goods_services": "Miscellaneous Goods & Services",
    "exp_durables": "Durables",
    "exp_housing_rent": "Housing Rent",
    "exp_lumpy": "Lumpy Expenditures",
    "exp_taxes_levies": "Taxes & Levies"
}

grouped = hh.groupby("income_group", observed=True)[
    list(expense_categories)
].sum()

shares = grouped.div(grouped.sum(axis=1), axis=0) * 100
shares = shares.rename(columns=expense_categories)

quintile_order = [
    "Bottom 20%",
    "Lower-middle 20%",
    "Middle 20%",
    "Upper-middle 20%",
    "Top 20%"
]

shares = shares.reindex(quintile_order)

plot_data = (
    shares.rename_axis("Income Quintile")
    .reset_index()
    .melt(
        id_vars="Income Quintile",
        var_name="Expenditure Category",
        value_name="Budget Share (%)"
    )
)

# ==============================================================================
# ==============================================================================
BG_COLOR = "#0b0905"       # Dark panel background
TEXT_COLOR = "#F5EFE0"     # Light gold text color
ACCENT_GOLD = "#D4AF37"    # Accent gold

gold_palette = [
    '#F3D77A', '#D4AF37', '#B8902A', '#A07F22', '#8A6A1C',
    '#7a5f1a', '#5C4A14', '#3A2D0C', '#4A3A10', '#D97736',
    '#E8903F', '#C99A2E', '#E8C35A', '#C7AE82', '#FFF7C9', '#5F5A47'
]

fig = px.bar(
    plot_data,
    x="Budget Share (%)",
    y="Income Quintile",
    color="Expenditure Category",
    orientation="h",
    barmode="stack",
    title="<b>◆ Budget Composition by Income Quintile</b>",
    category_orders={"Income Quintile": quintile_order},
    color_discrete_sequence=gold_palette,
    hover_data={"Budget Share (%)": ":.2f"}
)

# ==============================================================================
# ==============================================================================
fig.update_layout(
    paper_bgcolor=BG_COLOR,
    plot_bgcolor=BG_COLOR,
    font=dict(color=TEXT_COLOR, family="Arial"),
    width=1300,
    height=650,
    margin=dict(l=20, r=20, t=70, b=50),
    title=dict(
        font=dict(size=16, color=TEXT_COLOR, family="Arial"),
        x=0.0,
        y=0.96
    ),
    xaxis=dict(
        title="Share of Total Expenditure (%)",
        title_font=dict(color=TEXT_COLOR, size=12),
        tickfont=dict(color=TEXT_COLOR, size=11),
        range=[0, 100],
        ticksuffix="%",
        showgrid=False,
        zeroline=False
    ),
    yaxis=dict(
        title="",
        tickfont=dict(color=TEXT_COLOR, size=12, family="Arial"),
        showgrid=False,
        linecolor=ACCENT_GOLD,
        linewidth=1.5
    ),
    legend=dict(
        title=dict(text="Expenditure Category", font=dict(color=TEXT_COLOR, size=11)),
        font=dict(color=TEXT_COLOR, size=10),
        bgcolor=BG_COLOR,
        bordercolor=BG_COLOR
    ),
    hoverlabel=dict(
        bgcolor="#14100A",
        font_color=TEXT_COLOR,
        font_size=12,
        font_family="Arial"
    )
)

fig.show()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">9.7 Governorate-Level Income Analysis</h2></div>

In [ ]:
# Governorate Treemap: Household Count vs Median Income per Capita

# --- Luxury Gold & Dark Theme Color Palette ---
BG_COLOR = "#0b0905"
TEXT_COLOR = "#F5EFE0"
ACCENT_GOLD = "#D4AF37"
HEATMAP_COLORS = [
    '#5C4A14', # Dark Bronze (Below Average)
    '#0b0905', # Deep Black (Average)
    '#F3D77A'  # Bright Gold (Above Average)
]

gov = (
    geo.groupby(["macro_region", "region"], observed=True)
    .agg(
        households=("household_weight", "size"),
        median_ipc=("income_per_capita", "median")
    )
    .reset_index()
    .dropna(subset=["median_ipc"])
)

fig = px.treemap(
    gov,
    path=["macro_region", "region"],
    values="households",
    color="median_ipc",
    color_continuous_scale=HEATMAP_COLORS,
    hover_data={
        "households": True,
        "median_ipc": ":,.0f"
    }
)

fig.update_traces(
    texttemplate="<b>%{label}</b><br>%{value:,.0f} households",
    marker=dict(line=dict(color=ACCENT_GOLD, width=1.5)),
    hoverlabel=dict(
        bgcolor=BG_COLOR,
        bordercolor=ACCENT_GOLD,
        font=dict(color=TEXT_COLOR)
    )
)

fig.update_layout(
    paper_bgcolor=BG_COLOR,
    plot_bgcolor=BG_COLOR,
    font=dict(color=TEXT_COLOR, family="Arial, sans-serif"),
    height=750,
    title=dict(
        text="Egypt Governorates: Household Count and Median Income per Capita",
        x=0.5,
        font=dict(color=ACCENT_GOLD, size=20)
    ),
    coloraxis_colorbar=dict(
        title=dict(text="Median Income<br>per Capita (EGP)", font=dict(color=ACCENT_GOLD)),
        tickfont=dict(color=TEXT_COLOR),
        outlinewidth=1,
        outlinecolor=ACCENT_GOLD
    ),
    margin=dict(t=70, l=10, r=10, b=10)
)

fig.show()


In [ ]:
# ==============================================================================
# 1. Luxury Gold & Dark Theme Color Palette
# ==============================================================================
BG_COLOR = "#0b0905"       # Deep Royal Dark Background
TEXT_COLOR = "#F5EFE0"     # Champagne Gold for Text & Titles
ACCENT_GOLD = "#D4AF37"    # Classic Gold for Lines and Accents

# Luxury Gold Gradient (From Dark Bronze to Bright Champagne)
GOLD_SCALE = [
    [0.0, "#1d180a"],  # Deep Black-Gold (Lowest value)
    [0.3, "#7A5F1A"],  # Antique Bronze
    [0.6, "#C99A2E"],  # Classic Metallic Gold
    [1.0, "#FFF3B0"]   # Bright Champagne Gold (Highest value)
]

# ==============================================================================
# 2. Dynamic DataFrame & Column Detection
# ==============================================================================
def norm_col(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

df_income = None
inc_col = None
wgt_col = None
gov_col = None

valid_inc = {"incomepercapita", "incomeperperson", "income", "totalincome", "medianipc"}
valid_wgt = {"householdweight", "weight", "hhweight"}
valid_gov = {"governorate", "region", "governoratekey", "regionkey", "govname"}

for name, df in list(globals().items()):
    if isinstance(df, pd.DataFrame) and not df.empty and name not in ["chart_data", "df_div", "views", "view_results"]:
        norm_cols = {c: norm_col(c) for c in df.columns}

        i_col = next((c for c, nc in norm_cols.items() if nc in valid_inc), None)
        w_col = next((c for c, nc in norm_cols.items() if nc in valid_wgt), None)
        g_col = next((c for c, nc in norm_cols.items() if nc in valid_gov), None)

        if i_col and w_col and g_col:
            df_income = df.copy()
            inc_col = i_col
            wgt_col = w_col
            gov_col = g_col
            print(f"Data automatically sourced from DataFrame: '{name}'")
            break

if df_income is None:
    raise ValueError("Could not find any DataFrame containing income, weight, and governorate columns in memory.")

# ==============================================================================
# 3. Prepare Diverging Data (Deviation from National Median)
# ==============================================================================
# Calculate the overall National Median
national_median = weighted_quantile(df_income[inc_col], df_income[wgt_col], 0.50)

# Calculate each governorate's median
rows = []
for gov, group in df_income.groupby(gov_col):
    gov_median = weighted_quantile(group[inc_col], group[wgt_col], 0.50)
    clean_gov = str(gov).replace("_", " ").title()
    rows.append({"governorate": clean_gov, "median": gov_median})

df_div = pd.DataFrame(rows).dropna().sort_values("median").reset_index(drop=True)

# Calculate percentage difference from national average
df_div["pct_diff"] = ((df_div["median"] - national_median) / national_median) * 100

# ==============================================================================
# 4. Draw Diverging Bar Chart (Rounded Corners & Gold Gradient)
# ==============================================================================
fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_div["pct_diff"],
    y=df_div["governorate"],
    orientation="h",
    marker=dict(
        color=df_div["pct_diff"],
        colorscale=GOLD_SCALE,
        line=dict(color=BG_COLOR, width=1.5),
        cornerradius="50%"
    ),
    customdata=df_div["median"],
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Median Income: %{customdata:,.0f} EGP<br>"
        "Difference from National: %{x:+.1f}%<extra></extra>"
    )
))

# ==============================================================================
# 5. Layout & Luxury Styling
# ==============================================================================
title_text = f"Governorate Income vs. National Median ({national_median:,.0f} EGP)"
xaxis_title = "Percentage Above / Below National Median (%)"

fig.update_layout(
    title=dict(
        text=title_text,
        font=dict(family="Georgia, serif", size=22, color=ACCENT_GOLD)
    ),
    paper_bgcolor=BG_COLOR,
    plot_bgcolor=BG_COLOR,
    font=dict(family="Arial, sans-serif", color=TEXT_COLOR, size=13),
    width=1000,
    height=max(700, 32 * len(df_div) + 150),
    margin=dict(l=160, r=50, t=100, b=80),
    xaxis=dict(
        title=dict(text=xaxis_title, font=dict(color=ACCENT_GOLD)),
        showgrid=True,
        gridcolor="rgba(212, 175, 55, 0.12)",
        zeroline=True,
        zerolinecolor=ACCENT_GOLD,
        zerolinewidth=1.5,
        ticksuffix="%"
    ),
    yaxis=dict(
        categoryorder="array",
        categoryarray=df_div["governorate"].tolist(),
        showgrid=False,
        tickfont=dict(size=12)
    ),
    showlegend=False,
    hoverlabel=dict(
        bgcolor=BG_COLOR,
        bordercolor=ACCENT_GOLD,
        font=dict(color=TEXT_COLOR)
    )
)

# Baseline annotation
fig.add_annotation(
    x=0,
    y=len(df_div) - 0.5,
    text="National Median",
    showarrow=True,
    arrowhead=2,
    arrowcolor=TEXT_COLOR,
    font=dict(color=TEXT_COLOR, size=12),
    ax=50,
    ay=-30
)

fig.show()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">9.8 Household Head Education and Income</h2></div>

In [ ]:
# Sankey Diagram: Head's Education → Income Quintile

# ==========================================
# Luxury Theme Colors
# ==========================================
BG_COLOR = "#0b0905"
TEXT_COLOR = "#F5EFE0"
ACCENT_GOLD = "#D4AF37"
LINK_COLOR = "rgba(212, 175, 55, 0.3)" # Transparent gold for links

EDU_ORDER = ["None", "Primary/Lower secondary", "Secondary", "Post secondary or equivalent", "University", "Postgraduate"]
QUINTILES = ["Bottom 20%", "Lower-middle 20%", "Middle 20%", "Upper-middle 20%", "Top 20%"]

# 1. Data Preparation
flow = (
    hh.dropna(subset=["head_education_level", "income_group", "household_weight"])
    .groupby(["head_education_level", "income_group"], observed=True)["household_weight"]
    .sum().reset_index(name="weight")
)

# 2. Nodes Definition
edu_nodes = [x for x in EDU_ORDER if x in flow["head_education_level"].unique()]
income_nodes = [x for x in QUINTILES if x in flow["income_group"].unique()]
labels = edu_nodes + income_nodes

# 3. Source & Target Mapping (Simplified using .apply)
source = flow["head_education_level"].apply(lambda x: edu_nodes.index(x))
target = flow["income_group"].apply(lambda x: len(edu_nodes) + income_nodes.index(x))

# 4. Sankey Chart Construction
fig = go.Figure(go.Sankey(
    node=dict(
        pad=15,
        thickness=15,
        line=dict(color=TEXT_COLOR, width=0.5), # Subtle highlighted node border
        label=labels,
        color=ACCENT_GOLD
    ),
    link=dict(
        source=source,
        target=target,
        value=flow["weight"],
        color=LINK_COLOR,
        hovercolor="rgba(212, 175, 55, 0.7)" # Hover highlight
    )
))

# 5. Layout & Theme Application
fig.update_layout(
    title=dict(
        text="Head's Education → Household Income Quintile",
        font=dict(color=ACCENT_GOLD, size=18)
    ),
    paper_bgcolor=BG_COLOR,
    plot_bgcolor=BG_COLOR,
    font=dict(color=TEXT_COLOR, size=12),
    height=650,
    margin=dict(t=60, b=40, l=40, r=40)
)

fig.show()


In [ ]:
# ==============================================================================
# Income per Capita Distribution by Head's Education (Fixed & Luxury Styled)
# ==============================================================================


# ==============================================================================
# 1. Luxury Gold & Dark Theme Color Palette
# ==============================================================================
BG_COLOR = "#0b0905"       # Deep Royal Dark Background
TEXT_COLOR = "#F5EFE0"     # Champagne Gold for Text & Labels
ACCENT_GOLD = "#D4AF37"    # Bright Gold for Outlines & Titles
VIOLIN_COLOR = "#D4AF37"   # Classic Metallic Gold for Violins

EDU_ORDER = [
    "None",
    "Primary/Lower secondary",
    "Secondary",
    "Post secondary or equivalent",
    "University",
    "Postgraduate"
]

# ==============================================================================
# 2. Data Preparation & Outlier Clipping for Clear Visualization
# ==============================================================================
violin_data = geo.dropna(
    subset=["head_education_level", "income_per_capita"]
).copy()

violin_data = violin_data[violin_data["income_per_capita"] > 0]

upper_limit = violin_data["income_per_capita"].quantile(0.98)
violin_data_filtered = violin_data[violin_data["income_per_capita"] <= upper_limit]

# ==============================================================================
# 3. Draw Clean Violin Plot
# ==============================================================================
fig = px.violin(
    violin_data_filtered,
    x="head_education_level",
    y="income_per_capita",
    category_orders={"head_education_level": EDU_ORDER},
    box=True,
    points=False,
    color_discrete_sequence=[VIOLIN_COLOR],
    labels={
        "head_education_level": "Head's Education",
        "income_per_capita": "Income per Capita (EGP)"
    }
)

fig.update_traces(
    fillcolor="rgba(212, 175, 55, 0.25)",
    line=dict(color=ACCENT_GOLD, width=1.5),
    box=dict(
        fillcolor="#1a1405",
        line=dict(color=ACCENT_GOLD, width=1)
    ),
    meanline=dict(visible=True, color="#FFF3B0")
)

# ==============================================================================
# 4. Layout & Luxury Styling
# ==============================================================================
fig.update_layout(
    paper_bgcolor=BG_COLOR,
    plot_bgcolor=BG_COLOR,
    height=650,
    width=1200,
    showlegend=False,
    title=dict(
        text="Income per Capita Distribution by Head's Education",
        font=dict(family="Georgia, serif", size=22, color=ACCENT_GOLD),
        x=0.5,
        xanchor="center"
    ),
    font=dict(family="Arial, sans-serif", color=TEXT_COLOR, size=13),
    xaxis=dict(
        title=dict(text="Head's Education", font=dict(color=ACCENT_GOLD)),
        tickangle=-15,
        showgrid=False,
        tickfont=dict(color=TEXT_COLOR)
    ),
    yaxis=dict(
        title=dict(text="Income per Capita (EGP)", font=dict(color=ACCENT_GOLD)),
        showgrid=True,
        gridcolor="rgba(212, 175, 55, 0.12)",
        zeroline=False,
        tickfont=dict(color=TEXT_COLOR)
    ),
    hoverlabel=dict(
        bgcolor=BG_COLOR,
        bordercolor=ACCENT_GOLD,
        font=dict(color=TEXT_COLOR, size=13)
    ),
    margin=dict(l=80, r=50, t=100, b=100)
)

fig.show()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">9.9 Income Source Distribution</h2></div>

In [ ]:
# Find a household-level DataFrame already loaded in the notebook
data = None
for name in ["households", "hh_clean", "hh", "df_merged", "df"]:
    candidate = globals().get(name)
    if isinstance(candidate, pd.DataFrame) and not candidate.empty:
        data = candidate.copy()
        break

if data is None:
    raise NameError(
        "No household DataFrame found. Expected one of: "
        "households, hh_clean, hh, df_merged, or df."
    )

# Match column names despite spaces, capitalization, or underscores
def normalize_column(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

column_lookup = {normalize_column(c): c for c in data.columns}

def find_column(options, label, required=True):
    for option in options:
        match = column_lookup.get(normalize_column(option))
        if match is not None:
            return match

    if required:
        raise KeyError(
            f"Could not find the {label} column. "
            f"Available columns: {list(data.columns)}"
        )

    return None

source_col = find_column(
    ["main_income_source", "source_of_income", "income_source"],
    "main income source"
)
weight_col = find_column(
    ["household_weight", "weight", "hh_weight"],
    "household weight"
)
residence_col = find_column(
    ["urban_rural", "urban_rural_residence", "residence_type", "residence"],
    "residence",
    required=False
)

# Clean the required fields
chart_data = data[[source_col, weight_col]].copy()
chart_data.columns = ["income_source", "weight"]
chart_data["weight"] = pd.to_numeric(chart_data["weight"], errors="coerce")
chart_data = chart_data.dropna(subset=["income_source", "weight"])
chart_data = chart_data[chart_data["weight"] > 0]
chart_data["income_source"] = chart_data["income_source"].astype(str).str.strip()
chart_data = chart_data[chart_data["income_source"] != ""]

if chart_data.empty:
    raise ValueError("There are no valid income-source records to display.")

# Create the overall view and, if available, separate residence views
views = {"All Egypt": chart_data}

if residence_col is not None:
    residence_data = data[[source_col, weight_col, residence_col]].copy()
    residence_data.columns = ["income_source", "weight", "residence"]
    residence_data["weight"] = pd.to_numeric(
        residence_data["weight"], errors="coerce"
    )
    residence_data = residence_data.dropna(
        subset=["income_source", "weight", "residence"]
    )
    residence_data = residence_data[residence_data["weight"] > 0]
    residence_data["income_source"] = (
        residence_data["income_source"].astype(str).str.strip()
    )
    residence_data["residence"] = (
        residence_data["residence"].astype(str).str.strip()
    )

    for value in residence_data["residence"].unique():
        label = value.title()
        subset = residence_data.loc[
            residence_data["residence"] == value,
            ["income_source", "weight"]
        ]
        if not subset.empty:
            views[label] = subset

# Gold shades: brightest bar highlights the most common source
gold_shades = ["#F3D77A", "#D4AF37", "#B8902A", "#8A6A1C", "#E8C35A", "#A07F22"]

fig = go.Figure()
trace_by_view = {}
view_details = {}

for view_name, frame in views.items():
    totals = (
        frame.groupby("income_source", as_index=False)["weight"]
        .sum()
        .sort_values("weight", ascending=False)
    )

    total_weight = totals["weight"].sum()
    if total_weight <= 0:
        continue

    totals["share"] = totals["weight"] / total_weight * 100

    labels = totals["income_source"].tolist()
    shares = totals["share"].tolist()
    weights = totals["weight"].tolist()

    # Keep the largest category bright and vary the remaining gold tones
    colors = [
        gold_shades[i % len(gold_shades)]
        for i in range(len(labels))
    ]

    trace_index = len(fig.data)
    trace_by_view[view_name] = trace_index
    view_details[view_name] = {
        "labels": labels,
        "max_x": max(shares) * 1.28
    }

    fig.add_trace(go.Bar(
        x=shares,
        y=labels,
        orientation="h",
        marker=dict(
            color=colors,
            line=dict(color="#F3D77A", width=1)
        ),
        text=[f"{value:.1f}%" for value in shares],
        textposition="outside",
        textfont=dict(
            family="Arial",
            size=14,
            color="#F3D77A"
        ),
        customdata=weights,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Weighted household share: %{x:.1f}%<br>"
            "Weighted households: %{customdata:,.0f}"
            "<extra></extra>"
        ),
        visible=(view_name == "All Egypt"),
        cliponaxis=False
    ))

# Dropdown switches the chart between the available views
buttons = []

for view_name, trace_index in trace_by_view.items():
    visibility = [False] * len(fig.data)
    visibility[trace_index] = True

    details = view_details[view_name]

    buttons.append(dict(
        label=view_name,
        method="update",
        args=[
            {"visible": visibility},
            {
                "title": f"Where Egyptian Households Get Their Income — {view_name}",
                "xaxis.range": [0, details["max_x"]],
                "yaxis.categoryorder": "array",
                "yaxis.categoryarray": details["labels"][::-1]
            }
        ]
    ))

initial_view = "All Egypt"
initial_details = view_details[initial_view]

fig.update_layout(
    title=dict(
        text="Where Egyptian Households Get Their Income — All Egypt",
        x=0.06,
        y=0.94,
        xanchor="left",
        font=dict(family="Georgia", size=26, color="#F3D77A")
    ),
    paper_bgcolor="#0b0905",
    plot_bgcolor="#0b0905",
    font=dict(family="Arial", color="#F5EFE0", size=14),
    width=1100,
    height=max(560, 110 * len(initial_details["labels"]) + 210),
    margin=dict(l=270, r=100, t=160, b=95),
    bargap=0.35,
    showlegend=False,
    xaxis=dict(
        title=dict(
            text="Weighted share of households (%)",
            font=dict(size=15, color="#F3D77A")
        ),
        range=[0, initial_details["max_x"]],
        ticksuffix="%",
        showgrid=True,
        gridcolor="rgba(243, 215, 122, 0.14)",
        zeroline=False,
        showline=False,
        tickfont=dict(color="#A8A086")
    ),
    yaxis=dict(
        title=None,
        categoryorder="array",
        categoryarray=initial_details["labels"][::-1],
        autorange="reversed",
        showgrid=False,
        tickfont=dict(size=14, color="#F5EFE0")
    ),
    updatemenus=[dict(
        type="dropdown",
        direction="down",
        x=0.94,
        y=1.13,
        xanchor="right",
        yanchor="top",
        buttons=buttons,
        bgcolor="#17130A",
        bordercolor="#D4AF37",
        borderwidth=1,
        font=dict(color="#F3D77A", size=13),
        showactive=True
    )],
    annotations=[
        dict(
            text="Survey-weighted household shares · Hover over a bar for details",
            x=0.06,
            y=1.04,
            xref="paper",
            yref="paper",
            xanchor="left",
            showarrow=False,
            font=dict(size=14, color="#A8A086")
        )
    ]
)

# Rounded bar corners on Plotly versions that support them
try:
    fig.update_layout(barcornerradius=12)
except (TypeError, ValueError):
    pass

fig.show()


<div style="background-color:#0b0905;border:1px solid #2b2410;border-top:2px solid #d4af37;border-radius:6px;padding:14px 22px 12px;margin:26px 0 8px;"><h2 style="margin:0;padding:0;border:0;font-family:Fraunces,Georgia,'Times New Roman',serif;font-size:23px;line-height:1.25;font-weight:700;color:#f3d77a;">9.10 Household Asset Ownership</h2></div>

In [ ]:
# ------------------------------------------------------------
# Helpers: normalize and locate columns
# ------------------------------------------------------------

def normalize_column(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


weight_names = {
    normalize_column(name)
    for name in ["household_weight", "weight", "hh_weight"]
}

residence_names = {
    normalize_column(name)
    for name in [
        "residence_type",
        "urban_rural",
        "urban_rural_residence",
        "residence",
    ]
}


def get_matching_column(frame, valid_names):
    for column in frame.columns:
        if normalize_column(column) in valid_names:
            return column
    return None


# ------------------------------------------------------------
# Detect asset columns in standardized or original survey data
# ------------------------------------------------------------

asset_keywords = {
    "Vehicle": ["vehicle", "car", "taxi"],
    "Bicycle / motorcycle": ["bicycle", "scooter", "motorcycle"],
    "Television": ["television", "tv"],
    "Satellite receiver": ["satellite"],
    "Media player": ["dvd", "vcr", "audio player"],
    "Camera": ["camera", "photo", "video camera"],
    "Mobile phone": ["phone", "telephone", "cellphone"],
    "Computer": ["computer", "laptop"],
    "Internet access": ["internet"],
    "Refrigerator": ["refrigerator", "freezer"],
    "Cooker / stove": ["cooker", "stove", "oven"],
    "Microwave": ["microwave"],
    "Food processor": ["food processor", "blender", "grinder"],
    "Washing machine": ["washing machine"],
    "Dishwasher": ["dishwasher"],
    "Air conditioner": ["air conditioner"],
    "Electric fan": ["electric fan"],
    "Water heater": ["water heater"],
    "Heater": ["heater"],
    "Sewing machine": ["sewing machine"],
    "Vacuum cleaner": ["vacuum cleaner"],
    "Iron": ["iron"],
    "Water motor": ["water motor"],
    "Other durables": ["other durables"],
}


def detect_assets(frame):
    found = {}

    for column in frame.columns:
        key = normalize_column(column)

        # Survey ownership fields usually begin with "owns" or "has".
        if not (key.startswith("owns") or key.startswith("has")):
            continue

        for label, terms in asset_keywords.items():
            if any(normalize_column(term) in key for term in terms):
                found.setdefault(label, column)
                break

    return found


# ------------------------------------------------------------
# Find the notebook DataFrame that contains the most assets
# ------------------------------------------------------------

candidates = []

for name, candidate in list(globals().items()):
    if not isinstance(candidate, pd.DataFrame) or candidate.empty:
        continue

    weight = get_matching_column(candidate, weight_names)
    assets = detect_assets(candidate)

    if weight is not None and assets:
        is_household_named = int(
            "household" in name.lower() or name.lower().startswith("hh")
        )

        candidates.append(
            (
                len(assets),
                is_household_named,
                name,
                candidate,
                weight,
                assets,
            )
        )

if not candidates:
    raise NameError(
        "No DataFrame containing household weights and asset columns was found."
    )

# Prefer the DataFrame with the most detected assets.
# If tied, prefer a variable named households or hh_*.
_, _, selected_name, data, weight_col, asset_columns = max(candidates)

residence_col = get_matching_column(data, residence_names)

print(f"Using '{selected_name}' with {len(asset_columns)} detected assets:")
print(", ".join(asset_columns.keys()))

# ------------------------------------------------------------
# Convert ownership values to 1 = owns, 0 = does not own
# ------------------------------------------------------------

def to_owned(value):
    if pd.isna(value):
        return pd.NA

    if isinstance(value, bool):
        return int(value)

    if isinstance(value, (int, float)) and value in (0, 1):
        return int(value)

    text = str(value).strip().lower()

    if text in {"1", "yes", "y", "true", "owned", "available"}:
        return 1

    if text in {"0", "no", "n", "false", "not owned", "unavailable"}:
        return 0

    return pd.NA


# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

columns_to_use = [weight_col] + list(asset_columns.values())

if residence_col is not None:
    columns_to_use.append(residence_col)

chart_data = data[columns_to_use].copy()
chart_data = chart_data.rename(columns={weight_col: "weight"})
chart_data["weight"] = pd.to_numeric(chart_data["weight"], errors="coerce")
chart_data = chart_data.dropna(subset=["weight"])
chart_data = chart_data[chart_data["weight"] > 0]

for asset_label, column in asset_columns.items():
    chart_data[asset_label] = chart_data[column].map(to_owned)

# Overall view and optional residence views
views = {"All Egypt": chart_data}

if residence_col is not None:
    for value in chart_data[residence_col].dropna().astype(str).unique():
        label = value.strip().title()

        subset = chart_data[
            chart_data[residence_col].astype(str).str.strip() == value.strip()
        ]

        if not subset.empty:
            views[label] = subset


# ------------------------------------------------------------
# Calculate survey-weighted ownership rates
# ------------------------------------------------------------

view_results = {}

for view_name, frame in views.items():
    rows = []

    for asset_label in asset_columns:
        valid = frame[["weight", asset_label]].dropna()

        if valid.empty or valid["weight"].sum() == 0:
            continue

        weighted_total = valid["weight"].sum()
        weighted_owned = (valid[asset_label] * valid["weight"]).sum()

        rows.append({
            "asset": asset_label,
            "rate": 100 * weighted_owned / weighted_total,
            "weighted_owners": valid.loc[
                valid[asset_label] == 1, "weight"
            ].sum(),
            "valid_responses": len(valid),
        })

    if rows:
        view_results[view_name] = (
            pd.DataFrame(rows)
            .sort_values("rate", ascending=True)
            .reset_index(drop=True)
        )

if "All Egypt" not in view_results:
    raise ValueError(
        "No usable asset ownership data was found. "
        "Check how ownership responses are coded in the data."
    )


# ------------------------------------------------------------
# Draw the lollipop chart
# ------------------------------------------------------------

fig = go.Figure()
trace_indices = {}

for view_name, result in view_results.items():
    line_trace_index = len(fig.data)
    dot_trace_index = line_trace_index + 1
    trace_indices[view_name] = (line_trace_index, dot_trace_index)

    # Horizontal stems
    stem_x = []
    stem_y = []

    for row in result.itertuples():
        stem_x.extend([0, row.rate, None])
        stem_y.extend([row.asset, row.asset, None])

    fig.add_trace(go.Scatter(
        x=stem_x,
        y=stem_y,
        mode="lines",
        line=dict(color="rgba(212, 175, 55, 0.42)", width=3),
        hoverinfo="skip",
        showlegend=False,
        visible=(view_name == "All Egypt"),
    ))

    # Ownership rate dots
    marker_colors = [
        "#F3D77A" if rate >= 50 else
        "#B8902A" if rate >= 20 else
        "#7A5F1A"
        for rate in result["rate"]
    ]

    customdata = result[
        ["weighted_owners", "valid_responses"]
    ].to_numpy()

    fig.add_trace(go.Scatter(
        x=result["rate"],
        y=result["asset"],
        mode="markers+text",
        text=[f"{rate:.1f}%" for rate in result["rate"]],
        textposition="middle right",
        textfont=dict(
            family="Arial",
            size=13,
            color="#F3D77A",
        ),
        marker=dict(
            size=15,
            color=marker_colors,
            line=dict(color="#F3D77A", width=1.5),
        ),
        customdata=customdata,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Weighted ownership rate: %{x:.1f}%<br>"
            "Weighted households owning it: %{customdata[0]:,.0f}<br>"
            "Valid survey responses: %{customdata[1]:,.0f}"
            "<extra></extra>"
        ),
        showlegend=False,
        visible=(view_name == "All Egypt"),
        cliponaxis=False,
    ))


# ------------------------------------------------------------
# Residence selector
# ------------------------------------------------------------

buttons = []

for view_name, indices in trace_indices.items():
    visibility = [False] * len(fig.data)
    visibility[indices[0]] = True
    visibility[indices[1]] = True

    max_rate = max(
        100,
        view_results[view_name]["rate"].max() * 1.18
    )

    buttons.append(dict(
        label=view_name,
        method="update",
        args=[
            {"visible": visibility},
            {
                "title": (
                    "Household Asset Ownership in Egypt — "
                    f"{view_name}"
                ),
                "xaxis.range": [0, max_rate],
            },
        ],
    ))


# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------

initial = view_results["All Egypt"]
initial_max = max(100, initial["rate"].max() * 1.18)

fig.update_layout(
    title=dict(
        text="Household Asset Ownership in Egypt — All Egypt",
        x=0.06,
        y=0.94,
        xanchor="left",
        font=dict(
            family="Georgia",
            size=26,
            color="#F3D77A",
        ),
    ),
    paper_bgcolor="#0b0905",
    plot_bgcolor="#0b0905",
    font=dict(
        family="Arial",
        color="#F5EFE0",
        size=14,
    ),
    width=1100,
    height=max(650, 48 * len(initial) + 230),
    margin=dict(l=230, r=100, t=155, b=90),
    xaxis=dict(
        title=dict(
            text="Survey-weighted households owning the asset (%)",
            font=dict(size=15, color="#F3D77A"),
        ),
        range=[0, initial_max],
        ticksuffix="%",
        dtick=20,
        showgrid=True,
        gridcolor="rgba(243, 215, 122, 0.13)",
        zeroline=False,
        tickfont=dict(color="#A8A086"),
    ),
    yaxis=dict(
        title=None,
        categoryorder="array",
        categoryarray=initial["asset"].tolist(),
        showgrid=False,
        tickfont=dict(size=14, color="#F5EFE0"),
    ),
    updatemenus=[dict(
        type="dropdown",
        direction="down",
        x=0.94,
        y=1.13,
        xanchor="right",
        yanchor="top",
        buttons=buttons,
        bgcolor="#17130A",
        bordercolor="#D4AF37",
        borderwidth=1,
        font=dict(color="#F3D77A", size=13),
        showactive=True,
    )],
    annotations=[
        dict(
            text="Survey-weighted ownership rates · Hover over a dot for details",
            x=0.06,
            y=1.04,
            xref="paper",
            yref="paper",
            xanchor="left",
            showarrow=False,
            font=dict(size=14, color="#A8A086"),
        )
    ],
)

fig.show()
